<a href="https://colab.research.google.com/github/jy776/jy-mark1/blob/main/PTMS4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

BASE_DIR = "/content/drive/MyDrive/PTMS_v4"

folders = [
    BASE_DIR,
    f"{BASE_DIR}/input",
    f"{BASE_DIR}/output",
    f"{BASE_DIR}/output/tables",
    f"{BASE_DIR}/output/figures",
    f"{BASE_DIR}/output/reports",
    f"{BASE_DIR}/modules",
    f"{BASE_DIR}/dictionary",
    f"{BASE_DIR}/dictionary/stopwords",
    f"{BASE_DIR}/dictionary/keywords",
    f"{BASE_DIR}/tests"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created.")

Project folders created.


In [3]:
!pip -q install \
pdfplumber \
python-docx \
pandas \
numpy \
matplotlib \
networkx \
wordcloud \
scikit-learn \
langdetect \
spacy \
nltk \
jieba \
kiwipiepy \
openpyxl

In [4]:
import os

for root, dirs, files in os.walk(BASE_DIR):
    level = root.replace(BASE_DIR, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

PTMS_v4/
    input/
    output/
        tables/
            wordcloud/
            network/
            centrality/
            comparison/
        figures/
            wordcloud/
            network/
            centrality/
            comparison/
        reports/
            network/
            centrality/
            comparison/
        final/
            figures/
                by_country/
                    CHN/
                    KOR/
                    USA/
                    GLOBAL/
                by_analysis/
                    wordcloud/
                    network/
                    centrality/
                    comparison/
                overview/
            tables/
            reports/
            manifests/
        archives/
    modules/
    dictionary/
        stopwords/
        keywords/
        phrases/
        domain_stopwords/
        synonyms/
    tests/
    results/


In [5]:
# ==========================================
# PTMS v4
# Module 0 Validation
# ==========================================

import os
import importlib.util

BASE_DIR = "/content/drive/MyDrive/PTMS_v4"

folders = [
    BASE_DIR,
    f"{BASE_DIR}/input",
    f"{BASE_DIR}/output",
    f"{BASE_DIR}/output/tables",
    f"{BASE_DIR}/output/figures",
    f"{BASE_DIR}/output/reports",
    f"{BASE_DIR}/modules",
    f"{BASE_DIR}/dictionary",
    f"{BASE_DIR}/dictionary/stopwords",
    f"{BASE_DIR}/dictionary/keywords",
    f"{BASE_DIR}/tests"
]

libraries = [
    "pandas",
    "numpy",
    "pdfplumber",
    "docx",
    "matplotlib",
    "networkx",
    "sklearn",
    "langdetect",
    "spacy",
    "nltk",
    "jieba",
    "kiwipiepy",
    "openpyxl"
]

print("=" * 50)
print("PTMS v4 - Module 0 Validation")
print("=" * 50)

# 폴더 검사
print("\n[Folder Check]")
folder_ok = True

for folder in folders:
    if os.path.exists(folder):
        print(f"PASS : {folder}")
    else:
        print(f"FAIL : {folder}")
        folder_ok = False

# 라이브러리 검사
print("\n[Library Check]")
library_ok = True

for lib in libraries:
    if importlib.util.find_spec(lib) is not None:
        print(f"PASS : {lib}")
    else:
        print(f"FAIL : {lib}")
        library_ok = False

print("\n" + "=" * 50)

if folder_ok and library_ok:
    print("STATUS : PASS")
else:
    print("STATUS : FAIL")

print("=" * 50)

PTMS v4 - Module 0 Validation

[Folder Check]
PASS : /content/drive/MyDrive/PTMS_v4
PASS : /content/drive/MyDrive/PTMS_v4/input
PASS : /content/drive/MyDrive/PTMS_v4/output
PASS : /content/drive/MyDrive/PTMS_v4/output/tables
PASS : /content/drive/MyDrive/PTMS_v4/output/figures
PASS : /content/drive/MyDrive/PTMS_v4/output/reports
PASS : /content/drive/MyDrive/PTMS_v4/modules
PASS : /content/drive/MyDrive/PTMS_v4/dictionary
PASS : /content/drive/MyDrive/PTMS_v4/dictionary/stopwords
PASS : /content/drive/MyDrive/PTMS_v4/dictionary/keywords
PASS : /content/drive/MyDrive/PTMS_v4/tests

[Library Check]
PASS : pandas
PASS : numpy
PASS : pdfplumber
PASS : docx
PASS : matplotlib
PASS : networkx
PASS : sklearn
PASS : langdetect
PASS : spacy
PASS : nltk
PASS : jieba
PASS : kiwipiepy
PASS : openpyxl

STATUS : PASS


In [6]:
# ==========================================
# PTMS v4
# Module 1-1 : Install Fonts
# ==========================================

!apt-get -qq update
!apt-get -qq install fonts-noto-cjk

import matplotlib.font_manager as fm

fm._load_fontmanager(try_read_cache=False)

print("Font Installation Complete")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Font Installation Complete


In [7]:
FONT_PATH = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"

In [8]:
# ==========================================
# PTMS v4
# Module 1-1 : Import Libraries
# ==========================================

import os
import re

import pandas as pd
import pdfplumber

from docx import Document

In [9]:
# ==========================================
# PTMS v4
# Module 1-2 : Document Loader
# ==========================================

class DocumentLoader:

    SUPPORTED_EXTENSIONS = {".pdf", ".docx", ".txt"}

    def __init__(self, input_dir):
        self.input_dir = input_dir

    def _extract_year(self, filename):
        match = re.search(r"(19|20)\d{2}", filename)
        return int(match.group()) if match else None

    def _read_pdf(self, filepath):
        text = []

        with pdfplumber.open(filepath) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()

                if page_text:
                    text.append(page_text)

        return "\n".join(text)

    def _read_docx(self, filepath):

        doc = Document(filepath)

        return "\n".join(
            p.text for p in doc.paragraphs
        )

    def _read_txt(self, filepath):

        with open(filepath, "r", encoding="utf-8") as f:
            return f.read()

    def load(self):

        rows = []

        for filename in sorted(os.listdir(self.input_dir)):

            filepath = os.path.join(
                self.input_dir,
                filename
            )

            if not os.path.isfile(filepath):
                continue

            extension = os.path.splitext(filename)[1].lower()

            if extension not in self.SUPPORTED_EXTENSIONS:
                continue

            try:

                if extension == ".pdf":
                    text = self._read_pdf(filepath)

                elif extension == ".docx":
                    text = self._read_docx(filepath)

                else:
                    text = self._read_txt(filepath)

                rows.append({

                    "year": self._extract_year(filename),

                    "filename": filename,

                    "extension": extension.replace(".", ""),

                    "text": text

                })

            except Exception as e:

                print(f"[ERROR] {filename}")
                print(e)

        return pd.DataFrame(rows)

In [10]:
# ==========================================
# PTMS v4
# Module 1-3 : Execute Loader
# ==========================================

BASE_DIR = "/content/drive/MyDrive/PTMS_v4"

loader = DocumentLoader(
    BASE_DIR + "/input"
)

df = loader.load()

print(df.head())

   year                 filename extension  \
0  2017  CHN_2017_GOV_REPORT.pdf       pdf   
1  2018  CHN_2018_GOV_REPORT.pdf       pdf   
2  2019  CHN_2019_GOV_REPORT.pdf       pdf   
3  2020  CHN_2020_GOV_REPORT.pdf       pdf   
4  2021  CHN_2021_GOV_REPORT.pdf       pdf   

                                                text  
0  政 府 工 作 报 告\n——2017 年 3 月 5 日在第十二届全国人民\n代表大会第五...  
1  政府工作报告（文字实录） 完成，“十三五”规划顺利实施，经济社会发展取得历史 五年来，改革开...  
2  政府工作报告\n——2019 年 3 月 5 日在第十三届全国人民\n代表大会第二次会议上\...  
3  注 意 事 项\n此报告以本次大会最后审议\n通过并由新华社公布的文本为准\n.\n政 府 ...  
4  政府工作报告\n年 月 日在第十三届全国人民代表大会第四次会议上\n——2021 3 5\n...  


In [11]:
# ==========================================
# PTMS v4
# Module 1 Validation
# ==========================================

required_columns = [
    "year",
    "filename",
    "extension",
    "text"
]

print("=" * 50)
print("PTMS v4 - Module 1 Validation")
print("=" * 50)

status = True

# DataFrame 생성 여부
if isinstance(df, pd.DataFrame):
    print("PASS : DataFrame created")
else:
    print("FAIL : DataFrame not created")
    status = False

# 컬럼 검사
for col in required_columns:

    if col in df.columns:
        print(f"PASS : Column '{col}'")
    else:
        print(f"FAIL : Column '{col}'")
        status = False

# 데이터 존재 여부
if len(df) > 0:
    print(f"PASS : {len(df)} documents loaded")
else:
    print("FAIL : No documents loaded")
    status = False

# 연도 검사
if df["year"].notna().all():
    print("PASS : Year extraction")
else:
    print("WARNING : Some years could not be extracted")

# 본문 검사
empty_docs = (df["text"].str.len() == 0).sum()

if empty_docs == 0:
    print("PASS : Document text loaded")
else:
    print(f"WARNING : {empty_docs} empty document(s)")

print("=" * 50)

if status:
    print("STATUS : PASS")
else:
    print("STATUS : FAIL")

print("=" * 50)

PTMS v4 - Module 1 Validation
PASS : DataFrame created
PASS : Column 'year'
PASS : Column 'filename'
PASS : Column 'extension'
PASS : Column 'text'
PASS : 21 documents loaded
PASS : Year extraction
PASS : Document text loaded
STATUS : PASS


In [12]:
# ==========================================
# PTMS v4
# Module 2-1 : Import Libraries
# ==========================================

import re
import unicodedata

In [13]:
# ==========================================
# PTMS v4
# Module 2-2 : Text Preprocessor
# ==========================================

class TextPreprocessor:
    """
    PTMS v4
    Module 2

    Document preprocessing
    """

    def clean(self, text: str) -> str:

        if not isinstance(text, str):
            return ""

        # Unicode 정규화
        text = unicodedata.normalize("NFKC", text)

        # URL 제거
        text = re.sub(r"https?://\S+|www\.\S+", " ", text)

        # 이메일 제거
        text = re.sub(r"\S+@\S+", " ", text)

        # 줄바꿈
        text = text.replace("\n", " ")

        # 탭
        text = text.replace("\t", " ")

        # 연속 공백
        text = re.sub(r"\s+", " ", text)

        return text.strip()

In [14]:
# ==========================================
# PTMS v4
# Module 2-3 : Execute
# ==========================================

preprocessor = TextPreprocessor()

df["clean_text"] = df["text"].apply(
    preprocessor.clean
)

print(df.head())

   year                 filename extension  \
0  2017  CHN_2017_GOV_REPORT.pdf       pdf   
1  2018  CHN_2018_GOV_REPORT.pdf       pdf   
2  2019  CHN_2019_GOV_REPORT.pdf       pdf   
3  2020  CHN_2020_GOV_REPORT.pdf       pdf   
4  2021  CHN_2021_GOV_REPORT.pdf       pdf   

                                                text  \
0  政 府 工 作 报 告\n——2017 年 3 月 5 日在第十二届全国人民\n代表大会第五...   
1  政府工作报告（文字实录） 完成，“十三五”规划顺利实施，经济社会发展取得历史 五年来，改革开...   
2  政府工作报告\n——2019 年 3 月 5 日在第十三届全国人民\n代表大会第二次会议上\...   
3  注 意 事 项\n此报告以本次大会最后审议\n通过并由新华社公布的文本为准\n.\n政 府 ...   
4  政府工作报告\n年 月 日在第十三届全国人民代表大会第四次会议上\n——2021 3 5\n...   

                                          clean_text  
0  政 府 工 作 报 告 ——2017 年 3 月 5 日在第十二届全国人民 代表大会第五次会...  
1  政府工作报告(文字实录) 完成,“十三五”规划顺利实施,经济社会发展取得历史 五年来,改革开...  
2  政府工作报告 ——2019 年 3 月 5 日在第十三届全国人民 代表大会第二次会议上 国务...  
3  注 意 事 项 此报告以本次大会最后审议 通过并由新华社公布的文本为准 . 政 府 工 作 ...  
4  政府工作报告 年 月 日在第十三届全国人民代表大会第四次会议上 ——2021 3 5 国务院...  


In [15]:
# ==========================================
# PTMS v4
# Module 2 Validation
# ==========================================

status = True

print("=" * 50)
print("PTMS v4 - Module 2 Validation")
print("=" * 50)

# 컬럼 생성
if "clean_text" in df.columns:
    print("PASS : clean_text column")
else:
    print("FAIL : clean_text column")
    status = False

# 결측치
if df["clean_text"].isna().sum() == 0:
    print("PASS : No missing values")
else:
    print("FAIL : Missing values")
    status = False

# 빈 문자열
empty = (df["clean_text"].str.len() == 0).sum()

if empty == 0:
    print("PASS : Document text exists")
else:
    print(f"WARNING : {empty} empty documents")

# 줄바꿈 검사
newline = df["clean_text"].str.contains("\n").sum()

if newline == 0:
    print("PASS : Newline removed")
else:
    print("FAIL : Newline remains")
    status = False

# 연속 공백 검사
double_space = df["clean_text"].str.contains(r"\s{2,}", regex=True).sum()

if double_space == 0:
    print("PASS : Whitespace normalized")
else:
    print("FAIL : Multiple spaces remain")
    status = False

print("=" * 50)

if status:
    print("STATUS : PASS")
else:
    print("STATUS : FAIL")

print("=" * 50)

PTMS v4 - Module 2 Validation
PASS : clean_text column
PASS : No missing values
PASS : Document text exists
PASS : Newline removed
PASS : Whitespace normalized
STATUS : PASS


In [16]:
# ==========================================
# PTMS v4
# Module 3-1 : Import Libraries
# ==========================================

from langdetect import detect, DetectorFactory

# 결과 재현성 확보
DetectorFactory.seed = 42

In [17]:
# ==========================================
# PTMS v4
# Module 3-2 : Language Detector
# ==========================================

class LanguageDetector:
    """
    PTMS v4
    Module 3

    Detect language of each document.
    """

    VALID_LANGUAGES = {"en", "zh-cn", "zh-tw", "ko"}

    def detect(self, text: str) -> str:

        if not isinstance(text, str):
            return "unknown"

        text = text.strip()

        if len(text) < 50:
            return "unknown"

        try:
            lang = detect(text)

            if lang in ("zh-cn", "zh-tw"):
                return "zh"

            if lang in self.VALID_LANGUAGES:
                return lang

            return "unknown"

        except Exception:
            return "unknown"

In [18]:
# ==========================================
# PTMS v4
# Module 3-3 : Execute
# ==========================================

detector = LanguageDetector()

df["language"] = df["clean_text"].apply(detector.detect)

print(df[["filename", "language"]])

                    filename language
0    CHN_2017_GOV_REPORT.pdf       zh
1    CHN_2018_GOV_REPORT.pdf       zh
2    CHN_2019_GOV_REPORT.pdf       zh
3    CHN_2020_GOV_REPORT.pdf       zh
4    CHN_2021_GOV_REPORT.pdf       zh
5    CHN_2022_GOV_REPORT.pdf       zh
6    CHN_2023_GOV_REPORT.pdf       zh
7    CHN_2024_GOV_REPORT.pdf       zh
8    CHN_2025_GOV_REPORT.pdf       zh
9    CHN_2026_GOV_REPORT.pdf       zh
10          KOR_2018_NSS.pdf       ko
11  KOR_2020_국방백서.pdf       ko
12          KOR_2022_NSS.pdf       ko
13  KOR_2022_국방백서.pdf       ko
14    KOR_2025_정책집.pdf       ko
15          USA_2017_NSS.pdf       en
16          USA_2018_NDS.pdf       en
17          USA_2022_NDS.pdf       en
18          USA_2022_NSS.pdf       en
19          USA_2025_NSS.pdf       en
20          USA_2026_NDS.pdf       en


In [19]:
# ==========================================
# PTMS v4
# Module 3 Validation
# ==========================================

print("=" * 50)
print("PTMS v4 - Module 3 Validation")
print("=" * 50)

status = True

# language 컬럼 검사
if "language" in df.columns:
    print("PASS : language column")
else:
    print("FAIL : language column")
    status = False

# 허용 언어 코드 검사
valid_codes = {"en", "zh", "ko", "unknown"}
detected_codes = set(df["language"])

invalid = detected_codes - valid_codes

if len(invalid) == 0:
    print("PASS : Valid language codes")
else:
    print(f"FAIL : Invalid language codes -> {invalid}")
    status = False

# 언어별 개수 출력
print("\nLanguage Distribution")
print(df["language"].value_counts())

# 파일별 결과 출력
print("\nDetection Results")
print(df[["filename", "language"]])

print("=" * 50)

if status:
    print("STATUS : PASS")
else:
    print("STATUS : FAIL")

print("=" * 50)

PTMS v4 - Module 3 Validation
PASS : language column
PASS : Valid language codes

Language Distribution
language
zh    10
en     6
ko     5
Name: count, dtype: int64

Detection Results
                    filename language
0    CHN_2017_GOV_REPORT.pdf       zh
1    CHN_2018_GOV_REPORT.pdf       zh
2    CHN_2019_GOV_REPORT.pdf       zh
3    CHN_2020_GOV_REPORT.pdf       zh
4    CHN_2021_GOV_REPORT.pdf       zh
5    CHN_2022_GOV_REPORT.pdf       zh
6    CHN_2023_GOV_REPORT.pdf       zh
7    CHN_2024_GOV_REPORT.pdf       zh
8    CHN_2025_GOV_REPORT.pdf       zh
9    CHN_2026_GOV_REPORT.pdf       zh
10          KOR_2018_NSS.pdf       ko
11  KOR_2020_국방백서.pdf       ko
12          KOR_2022_NSS.pdf       ko
13  KOR_2022_국방백서.pdf       ko
14    KOR_2025_정책집.pdf       ko
15          USA_2017_NSS.pdf       en
16          USA_2018_NDS.pdf       en
17          USA_2022_NDS.pdf       en
18          USA_2022_NSS.pdf       en
19          USA_2025_NSS.pdf       en
20          USA_2

In [20]:
# ==========================================
# PTMS v4
# Module 4-1 : Import
# ==========================================

import re

import jieba
import spacy

from kiwipiepy import Kiwi

In [21]:
# ==========================================
# PTMS v4
# Module 4-2 : Load Models
# ==========================================

nlp = spacy.load("en_core_web_sm")

kiwi = Kiwi()

In [22]:
# ==========================================
# PTMS v4
# Module 4-3 : Tokenizer
# ==========================================

class Tokenizer:

    def tokenize(self, text, language):

        if not isinstance(text, str):
            return []

        # --------------------
        # English
        # --------------------
        if language == "en":

            doc = nlp(text)

            tokens = [
                token.text.lower()
                for token in doc
                if token.is_alpha
            ]

            return tokens

        # --------------------
        # Chinese
        # --------------------
        elif language == "zh":

            tokens = []

            for word in jieba.cut(text):

                word = word.strip()

                if len(word) == 0:
                    continue

                if word.isdigit():
                    continue

                if re.fullmatch(r"[^\w]+", word):
                    continue

                tokens.append(word)

            return tokens

        # --------------------
        # Korean
        # --------------------
        elif language == "ko":

            tokens = []

            for token in kiwi.tokenize(text):

                word = token.form.strip()

                if len(word) == 0:
                    continue

                if word.isdigit():
                    continue

                if re.fullmatch(r"[^\w]+", word):
                    continue

                tokens.append(word)

            return tokens

        return []

In [23]:
# ==========================================
# PTMS v4
# Module 4-4 : Execute
# ==========================================

tokenizer = Tokenizer()

df["tokens"] = df.apply(
    lambda row:
        tokenizer.tokenize(
            row["clean_text"],
            row["language"]
        ),
    axis=1
)

Building prefix dict from the default dictionary ...
DEBUG:jieba:Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
DEBUG:jieba:Loading model from cache /tmp/jieba.cache
Loading model cost 1.792 seconds.
DEBUG:jieba:Loading model cost 1.792 seconds.
Prefix dict has been built successfully.
DEBUG:jieba:Prefix dict has been built successfully.


In [24]:
# ==========================================
# PTMS v4
# Module 4 Validation
# ==========================================

print("="*60)
print("PTMS v4")
print("Module 4 : Tokenizer")
print("="*60)

status = True

# tokens 컬럼
if "tokens" in df.columns:
    print("PASS : tokens column")
else:
    print("FAIL : tokens column")
    status = False

# 리스트 여부
if df["tokens"].apply(lambda x: isinstance(x, list)).all():
    print("PASS : list type")
else:
    print("FAIL : list type")
    status = False

# 빈 토큰 검사
empty_docs = (df["tokens"].str.len() == 0).sum()

if empty_docs == 0:
    print("PASS : tokens generated")
else:
    print(f"WARNING : {empty_docs} empty document(s)")

# 샘플 출력
print("\nSample Tokens\n")

for _, row in df.iterrows():
    print("=" * 40)
    print(row["filename"])
    print(row["language"])
    print(row["tokens"][:30])

print("\n" + "="*60)

if status:
    print("STATUS : PASS")
else:
    print("STATUS : FAIL")

print("="*60)

PTMS v4
Module 4 : Tokenizer
PASS : tokens column
PASS : list type
PASS : tokens generated

Sample Tokens

CHN_2017_GOV_REPORT.pdf
zh
['政', '府', '工', '作', '报', '告', '年', '月', '日', '在', '第十二届', '全国', '人民', '代表大会', '第五次', '会议', '上', '国务院', '总理', '李克强', '各位', '代表', '现在', '我', '代表', '国务院', '向', '大会', '报告', '政府']
CHN_2018_GOV_REPORT.pdf
zh
['政府', '工作', '报告', '文字', '实录', '完成', '十三', '五', '规划', '顺利', '实施', '经济社会', '发展', '取得', '历史', '五年', '来', '改革开放', '迈出', '重大', '步伐', '改革', '全面', '发力', '多', '7.3%', '增速', '均', '比', '上年']
CHN_2019_GOV_REPORT.pdf
zh
['政府', '工作', '报告', '年', '月', '日', '在', '第十三届', '全国', '人民', '代表大会', '第二次', '会议', '上', '国务院', '总理', '李克强', '各位', '代表', '现在', '我', '代表', '国务院', '向', '大会', '报告', '政府', '工作', '请', '予']
CHN_2020_GOV_REPORT.pdf
zh
['注', '意', '事', '项', '此', '报告', '以', '本次', '大会', '最后', '审议', '通过', '并', '由', '新华社', '公布', '的', '文本', '为准', '政', '府', '工', '作', '报', '告', '年', '月', '日', '在', '第十三届']
CHN_2021_GOV_REPORT.pdf
zh
['政府', '工作', '报告', '年', '月', '日', '在', '第十三届', '全国人民代表大

In [25]:
# ==========================================
# PTMS v4
# Module 5-1 : Import Libraries
# ==========================================

import re

import spacy
import jieba

from kiwipiepy import Kiwi

In [26]:
# ==========================================
# PTMS v4
# Module 5-2 : Load Models
# ==========================================

# English
nlp = spacy.load("en_core_web_sm")

# Korean
kiwi = Kiwi()

In [27]:
# ==========================================
# PTMS v4
# Module 5-3 : BaseNormalizer
# ==========================================

import re

class BaseNormalizer:
    """
    Language-specific normalization.
    """

    def normalize(self, text: str, language: str):

        if not isinstance(text, str):
            return []

        if language == "en":
            return self._normalize_en(text)

        elif language == "ko":
            return self._normalize_ko(text)

        elif language == "zh":
            return self._normalize_zh(text)

        return []

    # -----------------------
    # English
    # -----------------------

    def _normalize_en(self, text):

        doc = nlp(text)

        tokens = []

        for token in doc:

            if not token.is_alpha:
                continue

            lemma = token.lemma_.lower().strip()

            if not lemma:
                continue

            if lemma == "-pron-":
                continue

            tokens.append(lemma)

        return tokens

    # -----------------------
    # Korean
    # -----------------------

    def _normalize_ko(self, text):

        tokens = []

        # 유지할 품사
        KEEP_TAGS = {
            "NNG",   # 일반명사
            "NNP",   # 고유명사
            "NNB",   # 의존명사
            "NR",    # 수사
            "SL",    # 영문
            "SH",    # 한자
            "VV",    # 동사
            "VA"     # 형용사
        }

        for token in kiwi.tokenize(text):

            if token.tag not in KEEP_TAGS:
                continue

            word = token.form.strip()

            if len(word) < 2:
                continue

            if word.isdigit():
                continue

            tokens.append(word)

        return tokens

    # -----------------------
    # Chinese
    # -----------------------

    def _normalize_zh(self, text):

        import jieba

        tokens = []

        for word in jieba.cut(text):

            word = word.strip()

            if not word:
                continue

            if word.isdigit():
                continue

            if re.fullmatch(r"[^\w]+", word):
                continue

            tokens.append(word)

        return tokens

In [28]:
# ==========================================
# PTMS v4
# Module 5-3 Validation
# ==========================================

normalizer = BaseNormalizer()

samples = [
    ("The military was running operations.", "en"),
    ("군은 작전을 수행하였다.", "ko"),
    ("中国军队进行了联合演习。", "zh")
]

print("=" * 60)
print("Module 5-3 Validation")
print("=" * 60)

for text, lang in samples:
    print(f"\nLanguage : {lang}")
    print("Input :", text)
    print("Output:", normalizer.normalize(text, lang))

print("\nSTATUS : PASS (수동 확인)")
print("=" * 60)

Module 5-3 Validation

Language : en
Input : The military was running operations.
Output: ['the', 'military', 'be', 'run', 'operation']

Language : ko
Input : 군은 작전을 수행하였다.
Output: ['작전', '수행']

Language : zh
Input : 中国军队进行了联合演习。
Output: ['中国', '军队', '进行', '了', '联合演习']

STATUS : PASS (수동 확인)


In [29]:
# ==========================================
# PTMS v4
# Module 5-4 : ConceptMapper
# ==========================================

class ConceptMapper:
    """
    Normalize synonymous expressions into
    one canonical concept.
    """

    def __init__(self):

        self.dictionary = {

            # --------------------------
            # United States
            # --------------------------

            "us":"united_states",
            "u.s":"united_states",
            "u.s.":"united_states",
            "usa":"united_states",
            "america":"united_states",

            "미국":"united_states",

            "美国":"united_states",
            "美國":"united_states",

            # --------------------------
            # China
            # --------------------------

            "china":"china",
            "중국":"china",
            "中国":"china",
            "中國":"china",

            # --------------------------
            # Chinese Communist Party
            # --------------------------

            "ccp":"chinese_communist_party",

            "communist party of china":"chinese_communist_party",

            "中国共产党":"chinese_communist_party",

            "중국공산당":"chinese_communist_party",

            # --------------------------
            # DPRK
            # --------------------------

            "north korea":"north_korea",

            "dprk":"north_korea",

            "북한":"north_korea",

            "朝鲜":"north_korea",

            "朝鮮":"north_korea"
        }

    def map(self, tokens):

        normalized = []

        for token in tokens:

            key = token.lower()

            normalized.append(
                self.dictionary.get(key, token)
            )

        return normalized

In [30]:
# ==========================================
# PTMS v4
# Module 5-4 Validation
# ==========================================

mapper = ConceptMapper()

sample = [
    "USA",
    "America",
    "미국",
    "美国",
    "China",
    "中国",
    "북한",
    "DPRK"
]

print("="*60)
print("Module 5-4 Validation")
print("="*60)

print(sample)

print()

print(
    mapper.map(sample)
)

print()

print("STATUS : PASS")

Module 5-4 Validation
['USA', 'America', '미국', '美国', 'China', '中国', '북한', 'DPRK']

['united_states', 'united_states', 'united_states', 'united_states', 'china', 'china', 'north_korea', 'north_korea']

STATUS : PASS


In [31]:
# ==========================================
# PTMS v4
# Module 5-5 : Execute
# ==========================================

normalizer = BaseNormalizer()

df["normalized_tokens"] = df.apply(
    lambda row: normalizer.normalize(
        row["clean_text"],
        row["language"]
    ),
    axis=1
)

print(df[["filename", "language", "normalized_tokens"]].head())

                  filename language  \
0  CHN_2017_GOV_REPORT.pdf       zh   
1  CHN_2018_GOV_REPORT.pdf       zh   
2  CHN_2019_GOV_REPORT.pdf       zh   
3  CHN_2020_GOV_REPORT.pdf       zh   
4  CHN_2021_GOV_REPORT.pdf       zh   

                                   normalized_tokens  
0  [政, 府, 工, 作, 报, 告, 年, 月, 日, 在, 第十二届, 全国, 人民, 代...  
1  [政府, 工作, 报告, 文字, 实录, 完成, 十三, 五, 规划, 顺利, 实施, 经济...  
2  [政府, 工作, 报告, 年, 月, 日, 在, 第十三届, 全国, 人民, 代表大会, 第...  
3  [注, 意, 事, 项, 此, 报告, 以, 本次, 大会, 最后, 审议, 通过, 并, ...  
4  [政府, 工作, 报告, 年, 月, 日, 在, 第十三届, 全国人民代表大会, 第四次, ...  


In [32]:
# ==========================================
# PTMS v4
# Module 5-6 : Validation
# ==========================================

print("=" * 60)
print("PTMS v4 - Module 5 Validation")
print("=" * 60)

status = True

# 컬럼 확인
if "normalized_tokens" in df.columns:
    print("PASS : normalized_tokens column")
else:
    print("FAIL : normalized_tokens column")
    status = False

# 타입 확인
if df["normalized_tokens"].apply(lambda x: isinstance(x, list)).all():
    print("PASS : list type")
else:
    print("FAIL : invalid data type")
    status = False

# 빈 문서 확인
empty_docs = (df["normalized_tokens"].str.len() == 0).sum()

if empty_docs == 0:
    print("PASS : normalized tokens created")
else:
    print(f"WARNING : {empty_docs} empty document(s)")

# 샘플 출력
print("\nSample Normalized Tokens\n")

for _, row in df.iterrows():
    print("=" * 40)
    print(row["filename"])
    print(row["language"])
    print(row["normalized_tokens"][:30])

print("\n" + "=" * 60)

if status:
    print("STATUS : PASS")
else:
    print("STATUS : FAIL")

print("=" * 60)

PTMS v4 - Module 5 Validation
PASS : normalized_tokens column
PASS : list type
PASS : normalized tokens created

Sample Normalized Tokens

CHN_2017_GOV_REPORT.pdf
zh
['政', '府', '工', '作', '报', '告', '年', '月', '日', '在', '第十二届', '全国', '人民', '代表大会', '第五次', '会议', '上', '国务院', '总理', '李克强', '各位', '代表', '现在', '我', '代表', '国务院', '向', '大会', '报告', '政府']
CHN_2018_GOV_REPORT.pdf
zh
['政府', '工作', '报告', '文字', '实录', '完成', '十三', '五', '规划', '顺利', '实施', '经济社会', '发展', '取得', '历史', '五年', '来', '改革开放', '迈出', '重大', '步伐', '改革', '全面', '发力', '多', '7.3%', '增速', '均', '比', '上年']
CHN_2019_GOV_REPORT.pdf
zh
['政府', '工作', '报告', '年', '月', '日', '在', '第十三届', '全国', '人民', '代表大会', '第二次', '会议', '上', '国务院', '总理', '李克强', '各位', '代表', '现在', '我', '代表', '国务院', '向', '大会', '报告', '政府', '工作', '请', '予']
CHN_2020_GOV_REPORT.pdf
zh
['注', '意', '事', '项', '此', '报告', '以', '本次', '大会', '最后', '审议', '通过', '并', '由', '新华社', '公布', '的', '文本', '为准', '政', '府', '工', '作', '报', '告', '年', '月', '日', '在', '第十三届']
CHN_2021_GOV_REPORT.pdf
zh
['政府', '工作', '报告', '年',

In [33]:
# ==========================================
# PTMS v4
# Module 6-1 : Stopword Dictionary
# ==========================================

from spacy.lang.en.stop_words import STOP_WORDS

EN_STOPWORDS = set(STOP_WORDS)

EN_CUSTOM = {
    "chapter",
    "table",
    "contents",
    "appendix",
    "page",
    "figure",
    "section"
}

KO_STOPWORDS = {
    "및",
    "등",
    "수",
    "것",
    "위",
    "통해",
    "대한",
    "있",
    "있다",
    "하다"
}

ZH_STOPWORDS = {
    "的",
    "了",
    "是",
    "在",
    "和",
    "及",
    "与",
    "对",
    "目录",
    "前言",
    "结束语",
    "星期四",
    "星期五",
    "责编"
}

In [34]:
# ==========================================
# PTMS v4
# Module 6-2 : Stopword Manager
# ==========================================

class StopwordManager:

    def remove(self, tokens, language):

        if not isinstance(tokens, list):
            return []

        filtered = []

        if language == "en":

            stopwords = EN_STOPWORDS | EN_CUSTOM

        elif language == "ko":

            stopwords = KO_STOPWORDS

        elif language == "zh":

            stopwords = ZH_STOPWORDS

        else:

            stopwords = set()

        for token in tokens:

            token = token.strip()

            if not token:
                continue

            if token.lower() in stopwords:
                continue

            filtered.append(token)

        return filtered

In [35]:
# ==========================================
# PTMS v4
# Module 6-3 : Execute
# ==========================================

manager = StopwordManager()

df["filtered_tokens"] = df.apply(
    lambda row:
        manager.remove(
            row["normalized_tokens"],
            row["language"]
        ),
    axis=1
)

In [36]:
# ==========================================
# PTMS v4
# Module 6 Validation
# ==========================================

print("=" * 60)
print("PTMS v4 - Module 6 Validation")
print("=" * 60)

status = True

if "filtered_tokens" in df.columns:
    print("PASS : filtered_tokens column")
else:
    print("FAIL : filtered_tokens column")
    status = False

if df["filtered_tokens"].apply(lambda x: isinstance(x, list)).all():
    print("PASS : list type")
else:
    print("FAIL : invalid list type")
    status = False

empty_docs = (df["filtered_tokens"].str.len() == 0).sum()

if empty_docs == 0:
    print("PASS : filtered tokens created")
else:
    print(f"WARNING : {empty_docs} empty document(s)")

print("\nSample Filtered Tokens\n")

for _, row in df.iterrows():
    print("=" * 40)
    print(row["filename"])
    print(row["language"])
    print(row["filtered_tokens"][:30])

print("\n" + "=" * 60)

if status:
    print("STATUS : PASS")
else:
    print("STATUS : FAIL")

print("=" * 60)

PTMS v4 - Module 6 Validation
PASS : filtered_tokens column
PASS : list type
PASS : filtered tokens created

Sample Filtered Tokens

CHN_2017_GOV_REPORT.pdf
zh
['政', '府', '工', '作', '报', '告', '年', '月', '日', '第十二届', '全国', '人民', '代表大会', '第五次', '会议', '上', '国务院', '总理', '李克强', '各位', '代表', '现在', '我', '代表', '国务院', '向', '大会', '报告', '政府', '工作']
CHN_2018_GOV_REPORT.pdf
zh
['政府', '工作', '报告', '文字', '实录', '完成', '十三', '五', '规划', '顺利', '实施', '经济社会', '发展', '取得', '历史', '五年', '来', '改革开放', '迈出', '重大', '步伐', '改革', '全面', '发力', '多', '7.3%', '增速', '均', '比', '上年']
CHN_2019_GOV_REPORT.pdf
zh
['政府', '工作', '报告', '年', '月', '日', '第十三届', '全国', '人民', '代表大会', '第二次', '会议', '上', '国务院', '总理', '李克强', '各位', '代表', '现在', '我', '代表', '国务院', '向', '大会', '报告', '政府', '工作', '请', '予', '审议']
CHN_2020_GOV_REPORT.pdf
zh
['注', '意', '事', '项', '此', '报告', '以', '本次', '大会', '最后', '审议', '通过', '并', '由', '新华社', '公布', '文本', '为准', '政', '府', '工', '作', '报', '告', '年', '月', '日', '第十三届', '全国', '人民']
CHN_2021_GOV_REPORT.pdf
zh
['政府', '工作', '报告', '年', '

In [37]:
# ==========================================
# PTMS v4
# Module 7-1 : Frequency Analyzer
# ==========================================

from collections import Counter
import pandas as pd


class FrequencyAnalyzer:

    def document_frequency(self, tokens):

        return Counter(tokens)

In [38]:
# ==========================================
# PTMS v4
# Module 7-2 : Document Frequency
# ==========================================

document_frequency = {}

for _, row in df.iterrows():

    document_frequency[row["filename"]] = FrequencyAnalyzer().document_frequency(
        row["filtered_tokens"]
    )

print("PASS : Document Frequency")

PASS : Document Frequency


In [39]:
# ==========================================
# PTMS v4
# Module 7-3 : Frequency DataFrame
# ==========================================

records = []

for _, row in df.iterrows():

    counter = document_frequency[row["filename"]]

    for word, freq in counter.items():

        records.append({

            "filename": row["filename"],
            "year": row["year"],
            "language": row["language"],
            "word": word,
            "frequency": freq

        })

frequency_df = pd.DataFrame(records)

frequency_df.head()

,filename,year,language,word,frequency
0,CHN_2017_GOV_REPORT.pdf,2017,zh,政,7
1,CHN_2017_GOV_REPORT.pdf,2017,zh,府,3
2,CHN_2017_GOV_REPORT.pdf,2017,zh,工,1
3,CHN_2017_GOV_REPORT.pdf,2017,zh,作,4
4,CHN_2017_GOV_REPORT.pdf,2017,zh,报,1


In [40]:
# ==========================================
# PTMS v4
# Module 7 Validation
# ==========================================

print("=" * 60)
print("PTMS v4 - Module 7 Validation")
print("=" * 60)

status = True

required_columns = [
    "filename",
    "year",
    "language",
    "word",
    "frequency"
]

for col in required_columns:

    if col in frequency_df.columns:
        print(f"PASS : {col}")
    else:
        print(f"FAIL : {col}")
        status = False

print()

print("Top 20 Frequency")

print(
    frequency_df
    .sort_values("frequency", ascending=False)
    .head(20)
)

print()

if status:
    print("STATUS : PASS")
else:
    print("STATUS : FAIL")

print("=" * 60)

PTMS v4 - Module 7 Validation
PASS : filename
PASS : year
PASS : language
PASS : word
PASS : frequency

Top 20 Frequency
                       filename  year language word  frequency
3013    CHN_2018_GOV_REPORT.pdf  2018       zh   发展       2439
3022    CHN_2018_GOV_REPORT.pdf  2018       zh   改革       1494
3758    CHN_2018_GOV_REPORT.pdf  2018       zh    要       1402
37587  KOR_2022_국방백서.pdf  2022       ko   위하       1383
3093    CHN_2018_GOV_REPORT.pdf  2018       zh   推进       1284
3131    CHN_2018_GOV_REPORT.pdf  2018       zh    等       1264
27648  KOR_2020_국방백서.pdf  2020       ko   위하       1170
37657  KOR_2022_국방백서.pdf  2022       ko   국방       1141
3114    CHN_2018_GOV_REPORT.pdf  2018       zh   经济       1018
27652  KOR_2020_국방백서.pdf  2020       ko   국방        993
3187    CHN_2018_GOV_REPORT.pdf  2018       zh   创新        900
3099    CHN_2018_GOV_REPORT.pdf  2018       zh    为        892
3651    CHN_2018_GOV_REPORT.pdf  2018       zh    新        8

In [41]:
country = row["filename"].split("_")[0]

In [42]:
# ==========================================
# PTMS v4
# Module 8-1 : Import
# ==========================================

import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

In [43]:
# ==========================================
# PTMS v4
# Module 8-2 : Documents
# ==========================================

documents = df["filtered_tokens"].apply(
    lambda tokens: " ".join(tokens)
)

documents.head()

,filtered_tokens
0,政 府 工 作 报 告 年 月 日 第十二届 全国 人民 代表大会 第五次 会议 上 国务院...
1,政府 工作 报告 文字 实录 完成 十三 五 规划 顺利 实施 经济社会 发展 取得 历史 ...
2,政府 工作 报告 年 月 日 第十三届 全国 人民 代表大会 第二次 会议 上 国务院 总理...
3,注 意 事 项 此 报告 以 本次 大会 最后 审议 通过 并 由 新华社 公布 文本 为准...
4,政府 工作 报告 年 月 日 第十三届 全国人民代表大会 第四次 会议 上 国务院 总理 李...


In [44]:
# ==========================================
# PTMS v4
# Module 8-3 : TF-IDF
# ==========================================

vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(documents)

feature_names = vectorizer.get_feature_names_out()

print("Documents :", tfidf_matrix.shape[0])
print("Vocabulary :", tfidf_matrix.shape[1])

Documents : 21
Vocabulary : 23828


In [45]:
# ==========================================
# PTMS v4
# Module 8-4 : TF-IDF DataFrame
# ==========================================

records = []

for i, row in df.iterrows():

    scores = tfidf_matrix[i].toarray().flatten()

    top_indices = scores.argsort()[::-1][:30]

    for idx in top_indices:

        if scores[idx] == 0:
            continue

        records.append({

            "filename": row["filename"],
            "year": row["year"],
            "language": row["language"],
            "word": feature_names[idx],
            "tfidf": scores[idx]

        })

tfidf_df = pd.DataFrame(records)

tfidf_df.head()

,filename,year,language,word,tfidf
0,CHN_2017_GOV_REPORT.pdf,2017,zh,发展,0.400531
1,CHN_2017_GOV_REPORT.pdf,2017,zh,推进,0.215161
2,CHN_2017_GOV_REPORT.pdf,2017,zh,改革,0.215161
3,CHN_2017_GOV_REPORT.pdf,2017,zh,建设,0.168819
4,CHN_2017_GOV_REPORT.pdf,2017,zh,经济,0.165508


In [46]:
# ==========================================
# PTMS v4
# Module 8 Validation
# ==========================================

print("=" * 60)
print("PTMS v4 - Module 8 Validation")
print("=" * 60)

status = True

required_columns = [
    "filename",
    "year",
    "language",
    "word",
    "tfidf"
]

for col in required_columns:

    if col in tfidf_df.columns:
        print(f"PASS : {col}")
    else:
        print(f"FAIL : {col}")
        status = False

print("\nTop TF-IDF Keywords\n")

for file in tfidf_df["filename"].unique():

    print("=" * 40)
    print(file)

    display(
        tfidf_df[
            tfidf_df["filename"] == file
        ]
        .sort_values("tfidf", ascending=False)
        .head(10)
    )

print("=" * 60)

if status:
    print("STATUS : PASS")
else:
    print("STATUS : FAIL")

PTMS v4 - Module 8 Validation
PASS : filename
PASS : year
PASS : language
PASS : word
PASS : tfidf

Top TF-IDF Keywords

CHN_2017_GOV_REPORT.pdf


,filename,year,language,word,tfidf
0,CHN_2017_GOV_REPORT.pdf,2017,zh,发展,0.400531
1,CHN_2017_GOV_REPORT.pdf,2017,zh,推进,0.215161
2,CHN_2017_GOV_REPORT.pdf,2017,zh,改革,0.215161
3,CHN_2017_GOV_REPORT.pdf,2017,zh,建设,0.168819
4,CHN_2017_GOV_REPORT.pdf,2017,zh,经济,0.165508
5,CHN_2017_GOV_REPORT.pdf,2017,zh,加强,0.142337
6,CHN_2017_GOV_REPORT.pdf,2017,zh,推动,0.135717
7,CHN_2017_GOV_REPORT.pdf,2017,zh,加快,0.129097
8,CHN_2017_GOV_REPORT.pdf,2017,zh,创新,0.119166
9,CHN_2017_GOV_REPORT.pdf,2017,zh,全面,0.119166


CHN_2018_GOV_REPORT.pdf


,filename,year,language,word,tfidf
30,CHN_2018_GOV_REPORT.pdf,2018,zh,发展,0.408763
31,CHN_2018_GOV_REPORT.pdf,2018,zh,改革,0.250386
32,CHN_2018_GOV_REPORT.pdf,2018,zh,推进,0.215191
33,CHN_2018_GOV_REPORT.pdf,2018,zh,经济,0.170611
34,CHN_2018_GOV_REPORT.pdf,2018,zh,创新,0.150835
35,CHN_2018_GOV_REPORT.pdf,2018,zh,建设,0.142623
36,CHN_2018_GOV_REPORT.pdf,2018,zh,全面,0.142455
37,CHN_2018_GOV_REPORT.pdf,2018,zh,加强,0.142120
38,CHN_2018_GOV_REPORT.pdf,2018,zh,中国,0.136135
39,CHN_2018_GOV_REPORT.pdf,2018,zh,企业,0.114299


CHN_2019_GOV_REPORT.pdf


...

USA_2025_NSS.pdf


,filename,year,language,word,tfidf
570,USA_2025_NSS.pdf,2025,en,american,0.316337
571,USA_2025_NSS.pdf,2025,en,world,0.230337
572,USA_2025_NSS.pdf,2025,en,states,0.219867
573,USA_2025_NSS.pdf,2025,en,country,0.213565
574,USA_2025_NSS.pdf,2025,en,want,0.206658
575,USA_2025_NSS.pdf,2025,en,united,0.203708
576,USA_2025_NSS.pdf,2025,en,america,0.197246
577,USA_2025_NSS.pdf,2025,en,interest,0.167473
578,USA_2025_NSS.pdf,2025,en,nation,0.143088
579,USA_2025_NSS.pdf,2025,en,trump,0.126698


USA_2026_NDS.pdf


,filename,year,language,word,tfidf
600,USA_2026_NDS.pdf,2026,en,unclassified,0.349338
601,USA_2026_NDS.pdf,2026,en,defense,0.290931
602,USA_2026_NDS.pdf,2026,en,trump,0.243286
603,USA_2026_NDS.pdf,2026,en,president,0.242213
604,USA_2026_NDS.pdf,2026,en,ally,0.225792
605,USA_2026_NDS.pdf,2026,en,partner,0.172423
606,USA_2026_NDS.pdf,2026,en,strategy,0.154022
607,USA_2026_NDS.pdf,2026,en,threat,0.146291
608,USA_2026_NDS.pdf,2026,en,interest,0.143686
609,USA_2026_NDS.pdf,2026,en,america,0.139581


STATUS : PASS


In [47]:
# ==========================================
# PTMS v4
# Module 9-1 : Import
# ==========================================

from nltk.util import ngrams
from collections import Counter
import pandas as pd

In [48]:
# ==========================================
# PTMS v4
# Module 9-2 : N-Gram Generator
# ==========================================

bigram_records = []
trigram_records = []

for _, row in df.iterrows():

    tokens = row["filtered_tokens"]

    bigrams = Counter(ngrams(tokens, 2))
    trigrams = Counter(ngrams(tokens, 3))

    # Bigram
    for words, freq in bigrams.items():

        bigram_records.append({

            "filename": row["filename"],
            "year": row["year"],
            "language": row["language"],
            "ngram": " ".join(words),
            "frequency": freq

        })

    # Trigram
    for words, freq in trigrams.items():

        trigram_records.append({

            "filename": row["filename"],
            "year": row["year"],
            "language": row["language"],
            "ngram": " ".join(words),
            "frequency": freq

        })

In [49]:
# ==========================================
# PTMS v4
# Module 9-3 : DataFrame
# ==========================================

bigram_df = pd.DataFrame(bigram_records)

trigram_df = pd.DataFrame(trigram_records)

print("Bigram :", len(bigram_df))

print("Trigram :", len(trigram_df))

Bigram : 244359
Trigram : 312908


In [50]:
# ==========================================
# PTMS v4
# Module 9-4 : Sort
# ==========================================

bigram_df = bigram_df.sort_values(
    "frequency",
    ascending=False
)

trigram_df = trigram_df.sort_values(
    "frequency",
    ascending=False
)

In [51]:
# ==========================================
# PTMS v4
# Module 9 Validation
# ==========================================

print("=" * 60)
print("PTMS v4 - Module 9 Validation")
print("=" * 60)

print("\nTop 20 Bigram\n")

display(
    bigram_df.head(20)
)

print("\nTop 20 Trigram\n")

display(
    trigram_df.head(20)
)

print("=" * 60)

print("STATUS : PASS")

PTMS v4 - Module 9 Validation

Top 20 Bigram



,filename,year,language,ngram,frequency
8873,CHN_2018_GOV_REPORT.pdf,2018,zh,中国 特色,271
13016,CHN_2018_GOV_REPORT.pdf,2018,zh,我们 要,220
201122,USA_2017_NSS.pdf,2017,en,united states,200
7843,CHN_2018_GOV_REPORT.pdf,2018,zh,更 多,181
8788,CHN_2018_GOV_REPORT.pdf,2018,zh,新 时代,171
7399,CHN_2018_GOV_REPORT.pdf,2018,zh,五年 来,170
7935,CHN_2018_GOV_REPORT.pdf,2018,zh,各位 代表,167
9635,CHN_2018_GOV_REPORT.pdf,2018,zh,深入 推进,166
127725,KOR_2022_국방백서.pdf,2022,ko,과학 기술,163
215602,USA_2022_NDS.pdf,2022,en,united states,146



Top 20 Trigram



,filename,year,language,ngram,frequency
8125,CHN_2018_GOV_REPORT.pdf,2018,zh,侧 结构性 改革,144
8124,CHN_2018_GOV_REPORT.pdf,2018,zh,供给 侧 结构性,144
9623,CHN_2018_GOV_REPORT.pdf,2018,zh,中国 特色 社会主义,116
9727,CHN_2018_GOV_REPORT.pdf,2018,zh,新 时代 中国,110
9728,CHN_2018_GOV_REPORT.pdf,2018,zh,时代 中国 特色,110
9536,CHN_2018_GOV_REPORT.pdf,2018,zh,习近平 新 时代,88
10010,CHN_2018_GOV_REPORT.pdf,2018,zh,业态 新 模式,83
10009,CHN_2018_GOV_REPORT.pdf,2018,zh,新 业态 新,83
157461,KOR_2022_국방백서.pdf,2022,ko,첨단 과학 기술,80
161317,KOR_2022_국방백서.pdf,2022,ko,탄도 미사일 발사,79


STATUS : PASS


In [52]:
# ==========================================
# PTMS v4
# Module 11-1 : Import
# ==========================================

from itertools import combinations
from collections import Counter
import pandas as pd

In [53]:
# ==========================================
# PTMS v4
# Module 11-2 : Co-occurrence
# ==========================================

pair_counter = Counter()

for tokens in df["filtered_tokens"]:

    # 같은 문서에서 중복 제거
    unique_tokens = sorted(set(tokens))

    for pair in combinations(unique_tokens, 2):
        pair_counter[pair] += 1

print("Total Pairs :", len(pair_counter))

Total Pairs : 76517457


In [ ]:
# ==========================================
# PTMS v4
# Module 11-3 : DataFrame
# ==========================================

network_df = pd.DataFrame(
    [
        {
            "word1": pair[0],
            "word2": pair[1],
            "co_occurrence": count
        }
        for pair, count in pair_counter.items()
    ]
)

network_df = network_df.sort_values(
    "co_occurrence",
    ascending=False
)

network_df.head()

In [ ]:
# ==========================================
# PTMS v4
# Module 11-4 : Top Pairs
# ==========================================

print("=" * 60)
print("Top 30 Co-occurring Pairs")
print("=" * 60)

display(network_df.head(30))

In [ ]:
# ==========================================
# PTMS v4
# Module 11 Validation
# ==========================================

print("=" * 60)
print("PTMS v4 - Module 11 Validation")
print("=" * 60)

status = True

required = [
    "word1",
    "word2",
    "co_occurrence"
]

for col in required:

    if col in network_df.columns:
        print(f"PASS : {col}")
    else:
        print(f"FAIL : {col}")
        status = False

print()

print("Total Edges :", len(network_df))

print()

print("Top 10")

display(network_df.head(10))

print()

if status:
    print("STATUS : PASS")
else:
    print("STATUS : FAIL")

print("=" * 60)

In [ ]:
# ==========================================
# PTMS v4
# Module 13-1 : Domain Stopword Dictionary
# ==========================================

DOMAIN_STOPWORDS = {

    "en": {

        "ability", "able", "also", "among", "across",
        "achieve", "achieving", "action", "address",
        "advance", "advanced", "addition", "agency",
        "allow", "alongside", "already", "approach",
        "appropriate", "area", "around", "become",
        "continue", "ensure", "improve", "include",
        "increase", "maintain", "make", "provide",
        "support", "take", "use", "using", "way",
        "work", "working", "effort", "develop"
    },

    "ko": {

        "위하", "통하", "대한", "관련", "통해",
        "추진", "확대", "강화", "개선", "지원",
        "체계", "기반", "구축", "분야", "역량",
        "목표", "정책", "정부", "국민", "사회",
        "지역", "국정", "과제", "활용"
    },

    "zh": {

        "推进", "加强", "实现", "发展", "进一步",
        "有关", "能力", "建设", "工作", "坚持",
        "提高", "完善", "推动", "促进", "全面",
        "不断", "积极", "继续", "加快"
    }

}

print("=" * 60)
print("PTMS v4 - Domain Stopwords Loaded")
print("=" * 60)

for lang, words in DOMAIN_STOPWORDS.items():
    print(f"{lang} : {len(words)} words")

print("\nSTATUS : PASS")

In [ ]:
# ==========================================
# PTMS v4
# Module 13-2 : Apply Domain Stopwords
# ==========================================

advanced_df = df.copy()

advanced_tokens = []

for _, row in advanced_df.iterrows():

    lang = row["language"]

    tokens = row["filtered_tokens"]

    new_tokens = [

        w for w in tokens

        if w not in DOMAIN_STOPWORDS.get(lang, set())

    ]

    advanced_tokens.append(new_tokens)

advanced_df["advanced_tokens"] = advanced_tokens

print("=" * 60)
print("Advanced Tokens Created")
print("=" * 60)

print(advanced_df[
    ["filename","language","advanced_tokens"]
].head())

print()

print("STATUS : PASS")

In [ ]:
# ==========================================
# PTMS v4
# Module 13-3 : Validation
# ==========================================

print("=" * 60)
print("PTMS v4 - Module 14 Validation")
print("=" * 60)

for _, row in advanced_df.head(5).iterrows():

    print(row["filename"])

    print("Before :", len(row["filtered_tokens"]))

    print("After  :", len(row["advanced_tokens"]))

    print()

print("STATUS : PASS")
print("=" * 60)

In [ ]:
# ==========================================
# PTMS v4.5
# Module 13-4 : Phrase Detection
# ==========================================

from pathlib import Path

print("="*60)
print("Phrase Detection")
print("="*60)

DICT_DIR = Path(BASE_DIR) / "dictionary" / "phrases"

phrase_files = {
    "en": "phrases_en.txt",
    "ko": "phrases_ko.txt",
    "zh": "phrases_zh.txt"
}

phrase_dict = {}

for lang, filename in phrase_files.items():

    path = DICT_DIR / filename

    if path.exists():

        with open(path, "r", encoding="utf-8") as f:

            phrase_dict[lang] = [
                line.strip()
                for line in f
                if line.strip()
            ]

    else:

        phrase_dict[lang] = []

print("Loaded Phrase Dictionaries")

for lang in phrase_dict:
    print(lang, len(phrase_dict[lang]))

In [ ]:
# ==========================================
# PTMS v4.5
# Module 13-5 : Apply Phrase Detection
# ==========================================

print("="*60)
print("Applying Phrase Detection")
print("="*60)

def apply_phrases(tokens, phrases):

    text = " ".join(tokens)

    for phrase in sorted(
        phrases,
        key=len,
        reverse=True
    ):

        replacement = phrase.replace(" ", "_")

        text = text.replace(
            phrase,
            replacement
        )

    return text.split()


new_tokens = []

for _, row in advanced_df.iterrows():

    lang = row["language"]

    tokens = row["advanced_tokens"]

    if lang.startswith("en"):
        phrases = phrase_dict["en"]

    elif lang.startswith("ko"):
        phrases = phrase_dict["ko"]

    elif lang.startswith("zh"):
        phrases = phrase_dict["zh"]

    else:
        phrases = []

    tokens = apply_phrases(
        tokens,
        phrases
    )

    new_tokens.append(tokens)


advanced_df["advanced_tokens"] = new_tokens

print("="*60)
print("Phrase Detection Complete")
print("="*60)

display(
    advanced_df[
        ["filename","advanced_tokens"]
    ].head()
)

In [ ]:
# ==========================================
# PTMS v4.5
# Module 13-6 : Synonym Mapping
# ==========================================

from pathlib import Path
import pandas as pd

print("="*60)
print("Synonym Mapping")
print("="*60)

SYN_PATH = (
    Path(BASE_DIR)
    / "dictionary"
    / "synonyms"
    / "synonyms.csv"
)

synonym_df = pd.read_csv(
    SYN_PATH,
    encoding="utf-8"
)

synonym_dict = dict(
    zip(
        synonym_df["source"],
        synonym_df["target"]
    )
)

print("Loaded Synonyms :", len(synonym_dict))

In [ ]:
# ==========================================
# PTMS v4.5
# Module 13-7 : Apply Synonyms
# ==========================================

print("="*60)
print("Applying Synonym Mapping")
print("="*60)

mapped_tokens = []

for tokens in advanced_df["advanced_tokens"]:

    new_tokens = []

    for token in tokens:

        if token in synonym_dict:

            new_tokens.append(
                synonym_dict[token]
            )

        else:

            new_tokens.append(token)

    mapped_tokens.append(new_tokens)

advanced_df["advanced_tokens"] = mapped_tokens

display(
    advanced_df[
        ["filename","advanced_tokens"]
    ].head()
)

print("="*60)
print("Synonym Mapping Complete")
print("="*60)

In [ ]:
# ==========================================
# PTMS v4.5
# Module 13-8 : Load Keyword Weights
# ==========================================

from pathlib import Path
import pandas as pd

print("="*60)
print("Keyword Weight Loader")
print("="*60)

WEIGHT_PATH = (
    Path(BASE_DIR)
    / "dictionary"
    / "keywords"
    / "keyword_weight.csv"
)

keyword_weight = pd.read_csv(
    WEIGHT_PATH,
    encoding="utf-8"
)

keyword_weight_dict = dict(
    zip(
        keyword_weight["keyword"],
        keyword_weight["weight"]
    )
)

print(f"Loaded {len(keyword_weight_dict)} keyword weights.")

display(keyword_weight.head())

print("="*60)
print("STATUS : PASS")
print("="*60)

In [ ]:
# ==========================================
# PTMS v4.5
# Module 14-1 : Dictionary Loader
# ==========================================

from pathlib import Path

# ------------------------------------------
# Dictionary Folder
# ------------------------------------------

DICT_DIR = Path(BASE_DIR) / "dictionary"

STOPWORD_DIR = DICT_DIR / "stopwords"
DOMAIN_DIR = DICT_DIR / "domain_stopwords"
KEYWORD_DIR = DICT_DIR / "keywords"
PHRASE_DIR = DICT_DIR / "phrases"


# ------------------------------------------
# Load txt
# ------------------------------------------

def load_txt_folder(folder):

    words = set()

    for file in sorted(folder.glob("*.txt")):

        with open(file, "r", encoding="utf-8") as f:

            for line in f:

                word = line.strip()

                if word != "":

                    words.add(word)

    return words


# ------------------------------------------
# Stopwords
# ------------------------------------------

stopwords = load_txt_folder(STOPWORD_DIR)

print(f"Stopwords : {len(stopwords):,}")


# ------------------------------------------
# Domain Stopwords
# ------------------------------------------

domain_stopwords = load_txt_folder(DOMAIN_DIR)

print(f"Domain Stopwords : {len(domain_stopwords):,}")


# ------------------------------------------
# Keywords
# ------------------------------------------

keywords = load_txt_folder(KEYWORD_DIR)

print(f"Keywords : {len(keywords):,}")


# ------------------------------------------
# Phrases
# ------------------------------------------

phrases = load_txt_folder(PHRASE_DIR)

print(f"Phrases : {len(phrases):,}")


print("="*60)
print("Dictionary Loaded")
print("="*60)
print("STATUS : PASS")

In [ ]:
# ==========================================
# PTMS v4.5
# Module 14-2 : Phrase Processing
# ==========================================

from copy import deepcopy

# 원본 보존
advanced_df["filtered_tokens"] = deepcopy(
    advanced_df["advanced_tokens"]
)

# phrase 적용 함수
def apply_phrases(tokens, phrase_set):

    text = " ".join(tokens)

    for phrase in sorted(phrase_set, key=len, reverse=True):

        replacement = phrase.replace(" ", "_")

        text = text.replace(
            phrase,
            replacement
        )

    return text.split()


advanced_df["filtered_tokens"] = (
    advanced_df["filtered_tokens"]
    .apply(
        lambda x: apply_phrases(
            x,
            phrases
        )
    )
)

print("="*60)
print("Phrase Processing Complete")
print("="*60)

display(
    advanced_df[
        ["filename", "filtered_tokens"]
    ].head()
)

print("STATUS : PASS")

In [ ]:
# ==========================================
# PTMS v4.5
# Module 14-3 : Stopword Filtering
# ==========================================

print("="*60)
print("Stopword Filtering")
print("="*60)

before_count = 0
after_count = 0


def remove_stopwords(tokens):

    filtered = []

    for token in tokens:

        token_lower = token.lower()

        if token_lower in stopwords:
            continue

        if token_lower in domain_stopwords:
            continue

        filtered.append(token)

    return filtered


before_count = advanced_df["filtered_tokens"].apply(len).sum()

advanced_df["filtered_tokens"] = (
    advanced_df["filtered_tokens"]
    .apply(remove_stopwords)
)

after_count = advanced_df["filtered_tokens"].apply(len).sum()

print(f"Before : {before_count:,}")
print(f"After  : {after_count:,}")
print(f"Removed: {before_count-after_count:,}")

print()
display(
    advanced_df[
        ["filename","filtered_tokens"]
    ].head()
)

print("STATUS : PASS")

In [ ]:
# ==========================================
# PTMS v4.5
# Module 14-4 : Keyword Filtering
# ==========================================

print("="*60)
print("Keyword Filtering")
print("="*60)

# ------------------------------------------
# Option
# True  : keywords만 사용
# False : 전체 단어 사용
# ------------------------------------------

USE_KEYWORDS = False

before_count = advanced_df["filtered_tokens"].apply(len).sum()

if USE_KEYWORDS:

    keyword_set = {k.lower() for k in keywords}

    def keep_keywords(tokens):

        return [
            token
            for token in tokens
            if token.lower() in keyword_set
        ]

    advanced_df["filtered_tokens"] = (
        advanced_df["filtered_tokens"]
        .apply(keep_keywords)
    )

    after_count = advanced_df["filtered_tokens"].apply(len).sum()

    print(f"Before : {before_count:,}")
    print(f"After  : {after_count:,}")
    print(f"Removed: {before_count-after_count:,}")

else:

    after_count = before_count

    print("Keyword Filter : OFF")

print()

display(
    advanced_df[
        ["filename","filtered_tokens"]
    ].head()
)

print("STATUS : PASS")

In [ ]:
# ==========================================
# PTMS v4.5
# Module 14-5 : Create TF-IDF Documents
# ==========================================

advanced_df["document"] = (
    advanced_df["filtered_tokens"]
    .apply(lambda tokens: " ".join(tokens))
)

print("=" * 60)
print("TF-IDF Documents Created")
print("=" * 60)

display(
    advanced_df[
        ["filename", "language", "document"]
    ].head()
)

print("STATUS : PASS")

In [ ]:
# ==========================================
# PTMS v4.5
# Module 14-5 : Create TF-IDF Documents
# ==========================================

print("="*60)
print("Create TF-IDF Documents")
print("="*60)

advanced_df["document"] = (
    advanced_df["filtered_tokens"]
    .apply(lambda tokens: " ".join(tokens))
)

display(
    advanced_df[
        ["filename","language","document"]
    ].head()
)

print("STATUS : PASS")

In [ ]:
# ==========================================
# PTMS v4.5
# Module 14-6 : TF-IDF Matrix
# ==========================================

from sklearn.feature_extraction.text import TfidfVectorizer

print("="*60)
print("Create TF-IDF Matrix")
print("="*60)

vectorizer = TfidfVectorizer(
    lowercase=False,
    min_df=2,
    max_df=0.90
)

tfidf_matrix = vectorizer.fit_transform(
    advanced_df["document"]
)

feature_names = vectorizer.get_feature_names_out()

print(f"Documents  : {tfidf_matrix.shape[0]}")
print(f"Vocabulary : {tfidf_matrix.shape[1]}")

print("STATUS : PASS")

In [ ]:
# ==========================================
# PTMS v4.5
# Module 14-7 : Country TF-IDF
# ==========================================

import numpy as np
import pandas as pd

print("="*60)
print("Country TF-IDF")
print("="*60)

TOP_N = 100

tfidf_results = {}

# ------------------------------------------
# Country List
# ------------------------------------------

countries = sorted(
    advanced_df["filename"]
    .str.split("_")
    .str[0]
    .unique()
)

print("Countries :", countries)
print()

# ------------------------------------------
# Calculate Country TF-IDF
# ------------------------------------------

for country in countries:

    docs = advanced_df[
        advanced_df["filename"].str.startswith(country + "_")
    ]

    print(f"{country} Documents : {len(docs)}")

    if len(docs) == 0:
        continue

    idx = docs.index.tolist()

    scores = np.asarray(
        tfidf_matrix[idx].mean(axis=0)
    ).flatten()

    result = pd.DataFrame({
        "word": feature_names,
        "tfidf": scores
    })

    # ------------------------------------------
    # Keyword Weight 적용
    # ------------------------------------------

    result["weight"] = (
        result["word"]
        .map(keyword_weight_dict)
        .fillna(1.0)
    )

    # ------------------------------------------
    # Final Score
    # (가중치가 TF-IDF를 완전히 덮지 않도록 30%만 반영)
    # ------------------------------------------

    result["score"] = (
        result["tfidf"]
        * (1 + 0.3 * (result["weight"] - 1))
    )

    # ------------------------------------------
    # Score > 0
    # ------------------------------------------

    result = result[
        result["score"] > 0
    ]

    # ------------------------------------------
    # Score 기준 정렬
    # ------------------------------------------

    result = result.sort_values(
        by="score",
        ascending=False
    )

    result = result.head(TOP_N)

    result = result.reset_index(drop=True)

    # Rank
    result.insert(
        0,
        "rank",
        range(1, len(result)+1)
    )

    tfidf_results[country] = result

    print("-"*60)
    display(result.head(20))
    print()

print("="*60)
print("Country TF-IDF Complete")
print("="*60)
print("Countries :", len(tfidf_results))
print("STATUS : PASS")

In [ ]:
# ==========================================
# PTMS v4.5
# Module 14-8 : Save TF-IDF Tables
# ==========================================

from pathlib import Path

print("="*60)
print("Save TF-IDF Tables")
print("="*60)

TABLE_DIR = Path(BASE_DIR) / "output" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

for country, table in tfidf_results.items():

    # ------------------------------------------
    # Score 기준 정렬
    # ------------------------------------------

    table = (
        table
        .sort_values(
            by="score",
            ascending=False
        )
        .reset_index(drop=True)
    )

    table["rank"] = range(1, len(table)+1)

    # ------------------------------------------
    # 전체 결과 저장
    # ------------------------------------------

    table.to_excel(
        TABLE_DIR / f"{country}_TFIDF.xlsx",
        index=False
    )

    table.to_csv(
        TABLE_DIR / f"{country}_TFIDF.csv",
        index=False,
        encoding="utf-8-sig"
    )

    # ------------------------------------------
    # 논문용 TOP20 저장
    # ------------------------------------------

    top20 = table.head(20)

    top20.to_excel(
        TABLE_DIR / f"{country}_TOP20.xlsx",
        index=False
    )

    top20.to_csv(
        TABLE_DIR / f"{country}_TOP20.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Saved : {country}")

print("="*60)
print("TF-IDF Tables Saved")
print("="*60)
print(TABLE_DIR)
print("STATUS : PASS")

In [ ]:
# ==========================================
# PTMS v4.5
# Module 14-9 : Country Keyword Comparison
# ==========================================

import pandas as pd
from pathlib import Path

print("="*60)
print("Country Keyword Comparison")
print("="*60)

TABLE_DIR = Path(BASE_DIR) / "output" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

comparison = pd.DataFrame()

# ------------------------------------------
# 국가별 TOP20 비교표 생성
# ------------------------------------------

for country, table in tfidf_results.items():

    temp = (
        table
        .sort_values(
            by="score",
            ascending=False
        )
        .head(20)
        .reset_index(drop=True)
    )

    comparison[f"{country}_Keyword"] = temp["word"]
    comparison[f"{country}_Score"] = temp["score"].round(4)
    comparison[f"{country}_TFIDF"] = temp["tfidf"].round(4)
    comparison[f"{country}_Weight"] = temp["weight"]

comparison.index = comparison.index + 1
comparison.index.name = "Rank"

display(comparison)

# ------------------------------------------
# 저장
# ------------------------------------------

comparison.to_excel(
    TABLE_DIR / "Country_Keyword_Comparison.xlsx"
)

comparison.to_csv(
    TABLE_DIR / "Country_Keyword_Comparison.csv",
    encoding="utf-8-sig"
)

print("="*60)
print("Comparison Table Saved")
print("="*60)
print(TABLE_DIR)
print("STATUS : PASS")

In [ ]:
# ==========================================
# PTMS v4.5
# Module 14-10 : Research Summary
# ==========================================

import pandas as pd
from pathlib import Path

print("="*60)
print("Research Summary")
print("="*60)

TABLE_DIR = Path(BASE_DIR) / "output" / "tables"

summary = []

for country, table in tfidf_results.items():

    top10 = (
        table
        .sort_values(
            "score",
            ascending=False
        )
        .head(10)
    )

    summary.append({

        "Country": country,

        "Top Keywords":
        ", ".join(top10["word"]),

        "Average Score":
        round(top10["score"].mean(),4),

        "Max Score":
        round(top10["score"].max(),4)

    })

summary = pd.DataFrame(summary)

display(summary)

summary.to_excel(
    TABLE_DIR / "Research_Summary.xlsx",
    index=False
)

summary.to_csv(
    TABLE_DIR / "Research_Summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("="*60)
print("Research Summary Saved")
print("="*60)
print(TABLE_DIR)
print("STATUS : PASS")

In [ ]:
# ============================================================
# PTMS v4.5
# Module 15 : Research-Grade WordCloud
# ============================================================

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from wordcloud import WordCloud


print("=" * 70)
print("PTMS Module 15 : Research-Grade WordCloud")
print("=" * 70)


# ============================================================
# 15-1. 기본 경로 설정
# ============================================================

FIGURE_DIR = Path(BASE_DIR) / "output" / "figures" / "wordcloud"
TABLE_DIR = Path(BASE_DIR) / "output" / "tables" / "wordcloud"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 15-2. WordCloud 설정값
# ============================================================

# 최종 표시 단어 수
TOP_N = 40

# score 기준 상위 30%를 우선 선택
# 0.70이면 70 percentile 이상만 사용
SCORE_QUANTILE = 0.70

# 분위수 필터 이후 단어가 너무 적으면 최소한 이 수만큼 확보
MIN_WORDS = 20

# WordCloud 이미지 설정
WC_WIDTH = 1600
WC_HEIGHT = 1000
WC_BACKGROUND = "white"
WC_RANDOM_STATE = 42

# 논문용 출력 해상도
SAVE_DPI = 300


# ============================================================
# 15-3. 폰트 자동 탐색
# ============================================================

def find_wordcloud_font():
    """
    Colab 또는 일반 Linux 환경에서 사용 가능한 폰트를 탐색한다.
    CJK 폰트를 우선하고, 없으면 기존 FONT_PATH 변수를 확인한다.
    """

    font_candidates = []

    # 이전 모듈에서 FONT_PATH가 설정되어 있으면 우선 확인
    if "FONT_PATH" in globals():
        font_candidates.append(str(FONT_PATH))

    # Colab / Linux에서 자주 사용되는 CJK 폰트
    font_candidates.extend([
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc",
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf",
    ])

    for font_path in font_candidates:
        if font_path and Path(font_path).exists():
            return font_path

    return None


WORDCLOUD_FONT_PATH = find_wordcloud_font()

if WORDCLOUD_FONT_PATH:
    print(f"[FONT] {WORDCLOUD_FONT_PATH}")
else:
    print("[WARNING] 사용 가능한 폰트를 찾지 못했습니다.")
    print("영어는 출력될 수 있지만 한국어·중국어가 깨질 수 있습니다.")


# ============================================================
# 15-4. 국가별 색상표 설정
# ============================================================

# Matplotlib 기본 colormap 이름을 사용한다.
# tfidf_results의 국가명이 다르더라도 default가 적용된다.

COUNTRY_COLORMAPS = {
    "USA": "Blues",
    "US": "Blues",
    "United States": "Blues",
    "United_States": "Blues",
    "미국": "Blues",

    "China": "Reds",
    "CHINA": "Reds",
    "중국": "Reds",

    "Korea": "Greens",
    "South Korea": "Greens",
    "South_Korea": "Greens",
    "ROK": "Greens",
    "한국": "Greens",

    "North Korea": "Purples",
    "North_Korea": "Purples",
    "DPRK": "Purples",
    "북한": "Purples",
}

DEFAULT_COLORMAP = "viridis"


# ============================================================
# 15-5. WordCloud용 데이터 정제 함수
# ============================================================

def prepare_wordcloud_data(
    table,
    top_n=TOP_N,
    score_quantile=SCORE_QUANTILE,
    min_words=MIN_WORDS
):
    """
    국가별 TF-IDF 결과표에서 WordCloud에 사용할 단어를 선별한다.

    처리 순서
    1. 필요한 열 확인
    2. score 결측치 및 비정상 값 제거
    3. 중복 단어 통합
    4. 분위수 기준 필터
    5. 최소 단어 수 보정
    6. TOP_N 제한
    """

    required_columns = {"word", "score"}

    if not required_columns.issubset(table.columns):
        missing = required_columns - set(table.columns)

        raise ValueError(
            f"WordCloud 생성에 필요한 열이 없습니다: {missing}"
        )

    data = table.copy()

    # 필요한 열만 유지
    keep_columns = ["word", "score"]

    for optional_column in ["tfidf", "weight", "rank"]:
        if optional_column in data.columns:
            keep_columns.append(optional_column)

    data = data[keep_columns].copy()

    # 문자열 및 숫자 형식 정리
    data["word"] = data["word"].astype(str).str.strip()
    data["score"] = pd.to_numeric(
        data["score"],
        errors="coerce"
    )

    # 결측치 및 무한대 제거
    data = data.replace(
        [np.inf, -np.inf],
        np.nan
    )

    data = data.dropna(
        subset=["word", "score"]
    )

    # 빈 문자열 제거
    data = data[
        data["word"].str.len() > 0
    ]

    # 0 이하 점수 제거
    data = data[
        data["score"] > 0
    ]

    if data.empty:
        return pd.DataFrame(
            columns=["word", "display_word", "score"]
        )

    # 같은 단어가 중복된 경우 최고 점수 유지
    data = (
        data
        .sort_values("score", ascending=False)
        .drop_duplicates(subset="word", keep="first")
        .reset_index(drop=True)
    )

    # score 분위수 계산
    cutoff = data["score"].quantile(score_quantile)

    filtered = data[
        data["score"] >= cutoff
    ].copy()

    # 필터 결과가 너무 적으면 score 상위 단어를 최소 개수만큼 확보
    target_minimum = min(
        min_words,
        len(data)
    )

    if len(filtered) < target_minimum:
        filtered = (
            data
            .sort_values("score", ascending=False)
            .head(target_minimum)
            .copy()
        )

    # 최종 TOP_N 제한
    filtered = (
        filtered
        .sort_values("score", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    # 시각화에서는 underscore를 공백으로 표시
    filtered["display_word"] = (
        filtered["word"]
        .str.replace("_", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    # 서로 다른 원형 단어가 같은 표시어가 된 경우 점수 합산
    aggregation = {
        "score": "sum",
        "word": lambda x: " | ".join(sorted(set(x)))
    }

    for optional_column in ["tfidf", "weight"]:
        if optional_column in filtered.columns:
            aggregation[optional_column] = "max"

    filtered = (
        filtered
        .groupby(
            "display_word",
            as_index=False
        )
        .agg(aggregation)
        .sort_values(
            "score",
            ascending=False
        )
        .reset_index(drop=True)
    )

    filtered.insert(
        0,
        "rank",
        range(1, len(filtered) + 1)
    )

    return filtered


# ============================================================
# 15-6. WordCloud 생성 함수
# ============================================================

def create_country_wordcloud(
    country,
    table
):
    """
    국가별 WordCloud 생성, 출력 및 저장
    """

    selected = prepare_wordcloud_data(table)

    if selected.empty:
        print(f"[SKIP] {country}: 사용할 단어가 없습니다.")
        return None

    # display_word와 score로 빈도 사전 생성
    frequencies = dict(
        zip(
            selected["display_word"],
            selected["score"]
        )
    )

    colormap = COUNTRY_COLORMAPS.get(
        str(country),
        DEFAULT_COLORMAP
    )

    wordcloud = WordCloud(
        width=WC_WIDTH,
        height=WC_HEIGHT,
        background_color=WC_BACKGROUND,
        font_path=WORDCLOUD_FONT_PATH,
        max_words=TOP_N,
        colormap=colormap,
        prefer_horizontal=0.90,
        relative_scaling=0.50,
        margin=5,
        random_state=WC_RANDOM_STATE,
        collocations=False,
        normalize_plurals=False,
        contour_width=0
    ).generate_from_frequencies(
        frequencies
    )

    # 파일명에 사용할 수 없는 문자 정리
    safe_country = re.sub(
        r'[\\/:*?"<>| ]+',
        "_",
        str(country)
    ).strip("_")

    figure_path = (
        FIGURE_DIR
        / f"WordCloud_{safe_country}.png"
    )

    table_path_csv = (
        TABLE_DIR
        / f"WordCloud_Terms_{safe_country}.csv"
    )

    table_path_excel = (
        TABLE_DIR
        / f"WordCloud_Terms_{safe_country}.xlsx"
    )

    # 그림 생성
    plt.figure(
        figsize=(12, 7.5)
    )

    plt.imshow(
        wordcloud,
        interpolation="bilinear"
    )

    plt.axis("off")

    plt.title(
        f"{country} — Key Discourse Terms",
        fontsize=18,
        pad=18
    )

    plt.tight_layout(
        pad=0.5
    )

    plt.savefig(
        figure_path,
        dpi=SAVE_DPI,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()
    plt.close()

    # WordCloud에 사용된 단어 저장
    selected.to_csv(
        table_path_csv,
        index=False,
        encoding="utf-8-sig"
    )

    selected.to_excel(
        table_path_excel,
        index=False
    )

    print(
        f"[PASS] {country}: "
        f"{len(selected)} words | "
        f"{figure_path.name}"
    )

    return {
        "country": country,
        "word_count": len(selected),
        "figure_path": str(figure_path),
        "table_csv": str(table_path_csv),
        "table_excel": str(table_path_excel),
        "selected_terms": selected
    }


# ============================================================
# 15-7. 전체 국가 WordCloud 실행
# ============================================================

if "tfidf_results" not in globals():
    raise NameError(
        "tfidf_results가 존재하지 않습니다. "
        "Module 14-7을 먼저 실행하세요."
    )

if not isinstance(tfidf_results, dict):
    raise TypeError(
        "tfidf_results는 국가명을 key로 갖는 dictionary여야 합니다."
    )

if len(tfidf_results) == 0:
    raise ValueError(
        "tfidf_results가 비어 있습니다."
    )


wordcloud_results = {}

for country, table in tfidf_results.items():

    print("-" * 70)
    print(f"Generating WordCloud: {country}")

    try:
        result = create_country_wordcloud(
            country=country,
            table=table
        )

        if result is not None:
            wordcloud_results[country] = result

    except Exception as error:
        print(f"[ERROR] {country}: {error}")


# ============================================================
# 15-8. 실행 요약표 생성
# ============================================================

wordcloud_summary_rows = []

for country, result in wordcloud_results.items():

    selected_terms = result["selected_terms"]

    top_terms = ", ".join(
        selected_terms
        .head(10)["display_word"]
        .astype(str)
        .tolist()
    )

    wordcloud_summary_rows.append({
        "Country": country,
        "Word Count": result["word_count"],
        "Top 10 Terms": top_terms,
        "Figure Path": result["figure_path"]
    })


wordcloud_summary = pd.DataFrame(
    wordcloud_summary_rows
)

if not wordcloud_summary.empty:

    display(wordcloud_summary)

    wordcloud_summary.to_csv(
        TABLE_DIR / "WordCloud_Summary.csv",
        index=False,
        encoding="utf-8-sig"
    )

    wordcloud_summary.to_excel(
        TABLE_DIR / "WordCloud_Summary.xlsx",
        index=False
    )


print("=" * 70)
print("Module 15 Completed")
print(f"Generated Countries : {len(wordcloud_results)}")
print(f"Figure Directory    : {FIGURE_DIR}")
print(f"Table Directory     : {TABLE_DIR}")
print("STATUS              : PASS")
print("=" * 70)

In [ ]:
# ============================================================
# PTMS v4.5
# Module 16 : Research-Grade Semantic Network
# Revised for CHN / KOR / USA
# ============================================================

import re
import math
import ast
import warnings
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import networkx as nx

from pathlib import Path
from collections import Counter

warnings.filterwarnings("ignore")


print("=" * 80)
print("PTMS Module 16 : Research-Grade Semantic Network")
print("=" * 80)


# ============================================================
# 16-1. 저장 경로
# ============================================================

FIGURE_DIR = (
    Path(BASE_DIR)
    / "output"
    / "figures"
    / "network"
)

TABLE_DIR = (
    Path(BASE_DIR)
    / "output"
    / "tables"
    / "network"
)

REPORT_DIR = (
    Path(BASE_DIR)
    / "output"
    / "reports"
    / "network"
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 16-2. 네트워크 설정
# ============================================================

TOP_NODES = 30

WINDOW_SIZE = 10

MIN_COOCCURRENCE = 2

EDGE_QUANTILE = 0.70

MIN_EDGES = 20

MAX_EDGES_PER_NODE = 6

REMOVE_ISOLATES = True

RANDOM_STATE = 42

LAYOUT_K = 1.2

SAVE_DPI = 300

MIN_NODE_SIZE = 700

MAX_NODE_SIZE = 4200

MIN_EDGE_WIDTH = 0.5

MAX_EDGE_WIDTH = 5.0

LABEL_FONT_SIZE = 10

FIGSIZE = (14, 11)


# ============================================================
# 16-3. 입력 데이터 확인
# ============================================================

if "tfidf_results" not in globals():

    raise NameError(
        "tfidf_results가 없습니다. "
        "Module 14를 먼저 실행하세요."
    )


if "advanced_df" not in globals():

    raise NameError(
        "advanced_df가 없습니다. "
        "Module 13을 먼저 실행하세요."
    )


if not isinstance(
    tfidf_results,
    dict
):

    raise TypeError(
        "tfidf_results는 dictionary여야 합니다."
    )


if len(tfidf_results) == 0:

    raise ValueError(
        "tfidf_results가 비어 있습니다."
    )


if not isinstance(
    advanced_df,
    pd.DataFrame
):

    raise TypeError(
        "advanced_df는 pandas DataFrame이어야 합니다."
    )


if advanced_df.empty:

    raise ValueError(
        "advanced_df가 비어 있습니다."
    )


advanced_df = advanced_df.copy()


# ============================================================
# 16-4. 국가 열 재생성 및 토큰 열 탐색
# ============================================================

TOKEN_COLUMN_CANDIDATES = [
    "advanced_tokens",
    "filtered_tokens",
    "normalized_tokens",
    "tokens",
    "token"
]


# ------------------------------------------------------------
# 국가 표준값
#
# tfidf_results의 key와 동일하게 통일
# ------------------------------------------------------------

STANDARD_COUNTRIES = [
    "CHN",
    "KOR",
    "USA"
]


# ------------------------------------------------------------
# 파일명 기반 국가 판별 패턴
# ------------------------------------------------------------

COUNTRY_FILENAME_PATTERNS = {

    "CHN": [
        "chn",
        "china",
        "chinese",
        "prc",
        "people_republic_of_china",
        "people's_republic_of_china",
        "peoples_republic_of_china",
        "mfa_china",
        "mfa_cn",
        "beijing",
        "중국",
        "中国",
        "中國"
    ],

    "KOR": [
        "kor",
        "korea",
        "south_korea",
        "republic_of_korea",
        "rok",
        "mofa_korea",
        "mofa_kr",
        "seoul",
        "대한민국",
        "한국"
    ],

    "USA": [
        "usa",
        "us",
        "united_states",
        "america",
        "american",
        "white_house",
        "state_department",
        "department_of_state",
        "dod",
        "pentagon",
        "congress",
        "washington",
        "미국"
    ]
}


def normalize_filename(value):
    """
    파일명을 국가 판별에 적합한 형태로 정규화한다.
    """

    value = str(
        value
    ).strip().lower()

    value = re.sub(
        r"\.[a-z0-9]{1,10}$",
        "",
        value
    )

    value = re.sub(
        r"[\s\-\./\\]+",
        "_",
        value
    )

    value = re.sub(
        r"_+",
        "_",
        value
    )

    return value.strip("_")


def pattern_in_filename(
    normalized_filename,
    pattern
):
    """
    짧은 국가 코드가 다른 단어 내부에서
    잘못 매칭되는 것을 방지한다.
    """

    normalized_pattern = (
        normalize_filename(
            pattern
        )
    )

    if not normalized_pattern:

        return False

    short_patterns = {
        "chn",
        "kor",
        "usa",
        "us",
        "rok",
        "prc",
        "dod"
    }

    if normalized_pattern in short_patterns:

        filename_tokens = set(
            normalized_filename.split("_")
        )

        return (
            normalized_pattern
            in filename_tokens
        )

    return (
        normalized_pattern
        in normalized_filename
    )


def infer_country_from_filename(
    filename
):
    """
    filename으로부터 CHN / KOR / USA를 판별한다.
    """

    normalized_filename = (
        normalize_filename(
            filename
        )
    )

    matches = []

    for country, patterns in (
        COUNTRY_FILENAME_PATTERNS.items()
    ):

        for pattern in patterns:

            if pattern_in_filename(
                normalized_filename,
                pattern
            ):

                matches.append(
                    country
                )

                break

    matches = list(
        dict.fromkeys(
            matches
        )
    )

    if len(matches) == 1:

        return matches[0]

    if len(matches) == 0:

        return "Unknown"

    # 여러 국가가 동시에 파일명에 포함되는 경우,
    # 파일명에서 가장 먼저 등장한 국가 패턴 선택
    country_positions = {}

    for country in matches:

        positions = []

        for pattern in (
            COUNTRY_FILENAME_PATTERNS[
                country
            ]
        ):

            normalized_pattern = (
                normalize_filename(
                    pattern
                )
            )

            position = (
                normalized_filename.find(
                    normalized_pattern
                )
            )

            if position >= 0:

                positions.append(
                    position
                )

        if positions:

            country_positions[
                country
            ] = min(
                positions
            )

    if country_positions:

        return min(
            country_positions,
            key=country_positions.get
        )

    return matches[0]


def find_existing_column(
    dataframe,
    candidates
):
    """
    후보 중 존재하는 열을 반환한다.
    """

    for column in candidates:

        if column in dataframe.columns:

            return column

    return None


# ------------------------------------------------------------
# country 열은 항상 filename 기준으로 다시 생성
# ------------------------------------------------------------

if "filename" not in advanced_df.columns:

    raise KeyError(
        "advanced_df에 filename 열이 없습니다. "
        "국가 분류를 수행할 수 없습니다."
    )


advanced_df["country"] = (
    advanced_df["filename"]
    .apply(
        infer_country_from_filename
    )
)

COUNTRY_COLUMN = "country"


print(
    "[CREATED] Country Column: "
    "advanced_df['country'] from filename"
)


# ------------------------------------------------------------
# 토큰 열 탐색
# ------------------------------------------------------------

TOKEN_COLUMN = (
    find_existing_column(
        advanced_df,
        TOKEN_COLUMN_CANDIDATES
    )
)


if TOKEN_COLUMN is None:

    raise KeyError(
        "토큰 열을 찾을 수 없습니다.\n"
        f"후보 열: {TOKEN_COLUMN_CANDIDATES}\n"
        f"현재 열: {list(advanced_df.columns)}"
    )


print(
    f"[DETECTED] Token Column: "
    f"{TOKEN_COLUMN}"
)


# ------------------------------------------------------------
# 국가 분류 결과
# ------------------------------------------------------------

country_detection_summary = (
    advanced_df[
        COUNTRY_COLUMN
    ]
    .fillna(
        "Unknown"
    )
    .astype(str)
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "Country"
    )
    .reset_index(
        name="Document Count"
    )
)


print(
    "\n[COUNTRY DETECTION SUMMARY]"
)

display(
    country_detection_summary
)


unknown_documents = (
    advanced_df[
        advanced_df[
            COUNTRY_COLUMN
        ]
        .fillna(
            "Unknown"
        )
        .astype(str)
        .str.strip()
        .eq(
            "Unknown"
        )
    ]
    .copy()
)


if not unknown_documents.empty:

    print(
        f"[WARNING] 국가 미분류 문서: "
        f"{len(unknown_documents)}개"
    )

    unknown_columns = [
        column
        for column in [
            "year",
            "filename",
            "language"
        ]
        if column in (
            unknown_documents.columns
        )
    ]

    display(
        unknown_documents[
            unknown_columns
        ].head(30)
    )

else:

    print(
        "[PASS] 모든 문서의 국가가 분류되었습니다."
    )


print(
    f"\nFinal Country Column : "
    f"{COUNTRY_COLUMN}"
)

print(
    f"Final Token Column   : "
    f"{TOKEN_COLUMN}"
)


# ============================================================
# 16-5. TF-IDF 국가 키와 국가 분류 결과 점검
# ============================================================

detected_countries = set(
    advanced_df[
        COUNTRY_COLUMN
    ]
    .dropna()
    .astype(str)
    .unique()
)


tfidf_countries = set(
    str(
        country
    ).strip()
    for country in (
        tfidf_results.keys()
    )
)


print(
    "\n[TF-IDF COUNTRY KEYS]"
)

print(
    sorted(
        tfidf_countries
    )
)


print(
    "\n[DETECTED DOCUMENT COUNTRIES]"
)

print(
    sorted(
        detected_countries
    )
)


missing_document_countries = (
    tfidf_countries
    - detected_countries
)


if missing_document_countries:

    print(
        "[WARNING] TF-IDF에는 있지만 "
        "문서 분류 결과에는 없는 국가:"
    )

    print(
        sorted(
            missing_document_countries
        )
    )

else:

    print(
        "[PASS] TF-IDF 국가와 문서 국가가 일치합니다."
    )


# ============================================================
# 16-6. 폰트 설정
# ============================================================

def find_network_font():
    """
    Colab 또는 Linux 환경에서 CJK 폰트를 탐색한다.
    """

    candidates = []

    for variable_name in [
        "FONT_PATH",
        "WORDCLOUD_FONT_PATH"
    ]:

        if variable_name in globals():

            value = globals()[
                variable_name
            ]

            if value:

                candidates.append(
                    str(value)
                )

    candidates.extend([
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc",
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf"
    ])

    for path in candidates:

        if (
            path
            and Path(
                path
            ).exists()
        ):

            return path

    return None


NETWORK_FONT_PATH = (
    find_network_font()
)


if NETWORK_FONT_PATH:

    NETWORK_FONT_PROPERTY = (
        fm.FontProperties(
            fname=NETWORK_FONT_PATH
        )
    )

    NETWORK_FONT_FAMILY = (
        NETWORK_FONT_PROPERTY
        .get_name()
    )

    plt.rcParams[
        "font.family"
    ] = NETWORK_FONT_FAMILY

    print(
        f"[FONT] {NETWORK_FONT_PATH}"
    )

else:

    NETWORK_FONT_PROPERTY = None

    NETWORK_FONT_FAMILY = (
        "sans-serif"
    )

    print(
        "[WARNING] CJK 폰트를 찾지 못했습니다."
    )


plt.rcParams[
    "axes.unicode_minus"
] = False


# ============================================================
# 16-7. 국가 별칭 및 기본 함수
# ============================================================

COUNTRY_VALUE_ALIASES = {

    "CHN": {
        "chn",
        "china",
        "chinese",
        "prc",
        "people's republic of china",
        "peoples republic of china",
        "people_republic_of_china",
        "중국",
        "中国",
        "中國"
    },

    "KOR": {
        "kor",
        "korea",
        "south korea",
        "south_korea",
        "republic of korea",
        "republic_of_korea",
        "rok",
        "한국",
        "대한민국"
    },

    "USA": {
        "usa",
        "us",
        "united states",
        "united_states",
        "america",
        "american",
        "미국"
    }
}


def normalize_country_name(
    value
):
    """
    국가명을 비교 가능한 형태로 정규화한다.
    """

    value = str(
        value
    ).strip().lower()

    value = re.sub(
        r"[\s\-\./]+",
        "_",
        value
    )

    value = re.sub(
        r"_+",
        "_",
        value
    )

    return value.strip("_")


def get_country_aliases(
    country
):
    """
    tfidf_results 국가 키와 문서 국가명을 연결한다.
    """

    normalized_input = (
        normalize_country_name(
            country
        )
    )

    aliases = {
        normalized_input
    }

    for standard_country, values in (
        COUNTRY_VALUE_ALIASES.items()
    ):

        normalized_standard = (
            normalize_country_name(
                standard_country
            )
        )

        normalized_values = {
            normalize_country_name(
                value
            )
            for value in values
        }

        if (
            normalized_input
            == normalized_standard
            or normalized_input
            in normalized_values
        ):

            aliases.add(
                normalized_standard
            )

            aliases.update(
                normalized_values
            )

    return aliases


def parse_tokens(
    value
):
    """
    다양한 형식의 토큰을 리스트로 변환한다.
    """

    if isinstance(
        value,
        list
    ):

        return [
            str(
                token
            ).strip()
            for token in value
            if str(
                token
            ).strip()
        ]

    if isinstance(
        value,
        tuple
    ):

        return [
            str(
                token
            ).strip()
            for token in value
            if str(
                token
            ).strip()
        ]

    if isinstance(
        value,
        np.ndarray
    ):

        return [
            str(
                token
            ).strip()
            for token in value.tolist()
            if str(
                token
            ).strip()
        ]

    if value is None:

        return []

    try:

        if pd.isna(
            value
        ):

            return []

    except Exception:

        pass

    value = str(
        value
    ).strip()

    if not value:

        return []

    if (
        value.startswith("[")
        and value.endswith("]")
    ):

        try:

            parsed = ast.literal_eval(
                value
            )

            if isinstance(
                parsed,
                (
                    list,
                    tuple
                )
            ):

                return [
                    str(
                        token
                    ).strip()
                    for token in parsed
                    if str(
                        token
                    ).strip()
                ]

        except Exception:

            pass

        value = re.sub(
            r"[\[\]'\" ]+",
            " ",
            value
        )

        tokens = re.split(
            r"[,\s]+",
            value
        )

    else:

        tokens = re.split(
            r"\s+",
            value
        )

    return [
        token.strip()
        for token in tokens
        if token.strip()
    ]


def display_label(
    word
):
    """
    underscore를 공백으로 표시한다.
    """

    return (
        str(
            word
        )
        .replace(
            "_",
            " "
        )
        .strip()
    )


def safe_filename(
    value
):
    """
    안전한 파일명 생성
    """

    return re.sub(
        r'[\\/:*?"<>| ]+',
        "_",
        str(
            value
        )
    ).strip("_")


def minmax_scale(
    values,
    minimum,
    maximum
):
    """
    값의 범위를 변환한다.
    """

    values = np.asarray(
        values,
        dtype=float
    )

    if len(
        values
    ) == 0:

        return np.array([])

    values = np.nan_to_num(
        values,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    value_min = np.min(
        values
    )

    value_max = np.max(
        values
    )

    if math.isclose(
        float(
            value_min
        ),
        float(
            value_max
        )
    ):

        return np.full(
            len(
                values
            ),
            (
                minimum
                + maximum
            )
            / 2
        )

    scaled = (
        (
            values
            - value_min
        )
        /
        (
            value_max
            - value_min
        )
    )

    return (
        minimum
        + scaled
        * (
            maximum
            - minimum
        )
    )


# ============================================================
# 16-8. 국가별 문서 추출
# ============================================================

def get_country_documents(
    dataframe,
    country
):
    """
    특정 국가의 토큰 문서를 추출한다.
    """

    country_aliases = (
        get_country_aliases(
            country
        )
    )

    normalized_country_series = (
        dataframe[
            COUNTRY_COLUMN
        ]
        .fillna(
            "Unknown"
        )
        .astype(str)
        .map(
            normalize_country_name
        )
    )

    country_mask = (
        normalized_country_series
        .isin(
            country_aliases
        )
    )

    country_df = (
        dataframe[
            country_mask
        ]
        .copy()
    )

    documents = [
        parse_tokens(
            value
        )
        for value in (
            country_df[
                TOKEN_COLUMN
            ]
        )
    ]

    documents = [
        document
        for document in documents
        if len(
            document
        ) > 0
    ]

    return (
        country_df,
        documents
    )


# ============================================================
# 16-9. 핵심 노드 선정
# ============================================================

def select_top_nodes(
    tfidf_table,
    top_nodes=TOP_NODES
):
    """
    Module 14의 score 기준 상위 핵심어 선정
    """

    required_columns = {
        "word",
        "score"
    }

    if not required_columns.issubset(
        tfidf_table.columns
    ):

        missing = (
            required_columns
            - set(
                tfidf_table.columns
            )
        )

        raise ValueError(
            f"TF-IDF 표 필수 열 누락: "
            f"{missing}"
        )

    table = (
        tfidf_table.copy()
    )

    table[
        "word"
    ] = (
        table[
            "word"
        ]
        .astype(str)
        .str.strip()
    )

    table[
        "score"
    ] = pd.to_numeric(
        table[
            "score"
        ],
        errors="coerce"
    )

    table = table.replace(
        [
            np.inf,
            -np.inf
        ],
        np.nan
    )

    table = table.dropna(
        subset=[
            "word",
            "score"
        ]
    )

    table = table[
        table[
            "word"
        ].str.len() > 0
    ]

    table = table[
        table[
            "score"
        ] > 0
    ]

    table = (
        table
        .sort_values(
            "score",
            ascending=False
        )
        .drop_duplicates(
            subset="word",
            keep="first"
        )
        .head(
            top_nodes
        )
        .reset_index(
            drop=True
        )
    )

    return table


# ============================================================
# 16-10. 슬라이딩 윈도우 공출현
# ============================================================

def calculate_cooccurrence(
    documents,
    allowed_words,
    window_size=WINDOW_SIZE
):
    """
    핵심어 간 슬라이딩 윈도우 공출현 계산
    """

    allowed_words = set(
        allowed_words
    )

    node_window_frequency = Counter()

    edge_cooccurrence = Counter()

    total_windows = 0

    for document in documents:

        filtered_document = [
            token
            for token in document
            if token in allowed_words
        ]

        if len(
            filtered_document
        ) < 2:

            continue

        if len(
            filtered_document
        ) <= window_size:

            windows = [
                filtered_document
            ]

        else:

            windows = [
                filtered_document[
                    start:
                    start
                    + window_size
                ]
                for start in range(
                    len(
                        filtered_document
                    )
                    - window_size
                    + 1
                )
            ]

        for window in windows:

            unique_words = sorted(
                set(
                    window
                )
            )

            if len(
                unique_words
            ) < 2:

                continue

            total_windows += 1

            for word in unique_words:

                node_window_frequency[
                    word
                ] += 1

            for word_a, word_b in (
                itertools.combinations(
                    unique_words,
                    2
                )
            ):

                edge_key = tuple(
                    sorted(
                        (
                            word_a,
                            word_b
                        )
                    )
                )

                edge_cooccurrence[
                    edge_key
                ] += 1

    return (
        node_window_frequency,
        edge_cooccurrence,
        total_windows
    )


# ============================================================
# 16-11. Edge 연관강도
# ============================================================

def create_edge_table(
    node_frequency,
    edge_frequency,
    total_windows
):
    """
    association과 PMI 계산
    """

    rows = []

    for (
        source,
        target
    ), cooccurrence in (
        edge_frequency.items()
    ):

        source_frequency = (
            node_frequency[
                source
            ]
        )

        target_frequency = (
            node_frequency[
                target
            ]
        )

        if (
            source_frequency <= 0
            or target_frequency <= 0
        ):

            continue

        association = (
            cooccurrence
            /
            math.sqrt(
                source_frequency
                * target_frequency
            )
        )

        pmi = np.nan

        if total_windows > 0:

            source_probability = (
                source_frequency
                / total_windows
            )

            target_probability = (
                target_frequency
                / total_windows
            )

            joint_probability = (
                cooccurrence
                / total_windows
            )

            denominator = (
                source_probability
                * target_probability
            )

            if (
                joint_probability > 0
                and denominator > 0
            ):

                pmi = math.log2(
                    joint_probability
                    / denominator
                )

        rows.append({
            "source": source,
            "target": target,
            "cooccurrence": int(
                cooccurrence
            ),
            "source_frequency": int(
                source_frequency
            ),
            "target_frequency": int(
                target_frequency
            ),
            "association": float(
                association
            ),
            "pmi": (
                float(
                    pmi
                )
                if not pd.isna(
                    pmi
                )
                else np.nan
            )
        })

    edge_table = pd.DataFrame(
        rows
    )

    if edge_table.empty:

        return pd.DataFrame(
            columns=[
                "source",
                "target",
                "cooccurrence",
                "source_frequency",
                "target_frequency",
                "association",
                "pmi"
            ]
        )

    return (
        edge_table
        .sort_values(
            [
                "association",
                "cooccurrence"
            ],
            ascending=[
                False,
                False
            ]
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# 16-12. Edge 필터링
# ============================================================

def filter_edges(
    edge_table,
    min_cooccurrence=MIN_COOCCURRENCE,
    edge_quantile=EDGE_QUANTILE,
    min_edges=MIN_EDGES,
    max_edges_per_node=MAX_EDGES_PER_NODE
):
    """
    약한 edge 제거
    """

    if edge_table.empty:

        return edge_table.copy()

    candidate = edge_table[
        edge_table[
            "cooccurrence"
        ] >= min_cooccurrence
    ].copy()

    if candidate.empty:

        candidate = (
            edge_table.copy()
        )

    cutoff = candidate[
        "association"
    ].quantile(
        edge_quantile
    )

    filtered = candidate[
        candidate[
            "association"
        ] >= cutoff
    ].copy()

    target_minimum = min(
        min_edges,
        len(
            candidate
        )
    )

    if len(
        filtered
    ) < target_minimum:

        filtered = (
            candidate
            .sort_values(
                [
                    "association",
                    "cooccurrence"
                ],
                ascending=[
                    False,
                    False
                ]
            )
            .head(
                target_minimum
            )
            .copy()
        )

    filtered = (
        filtered
        .sort_values(
            [
                "association",
                "cooccurrence"
            ],
            ascending=[
                False,
                False
            ]
        )
        .reset_index(
            drop=True
        )
    )

    node_edge_counts = Counter()

    selected_rows = []

    for _, row in (
        filtered.iterrows()
    ):

        source = row[
            "source"
        ]

        target = row[
            "target"
        ]

        if (
            node_edge_counts[
                source
            ] >= max_edges_per_node
            or node_edge_counts[
                target
            ] >= max_edges_per_node
        ):

            continue

        selected_rows.append(
            row.to_dict()
        )

        node_edge_counts[
            source
        ] += 1

        node_edge_counts[
            target
        ] += 1

    selected = pd.DataFrame(
        selected_rows
    )

    if selected.empty:

        selected = (
            filtered.copy()
        )

    elif len(
        selected
    ) < min(
        5,
        len(
            filtered
        )
    ):

        selected = (
            filtered.copy()
        )

    return (
        selected
        .sort_values(
            [
                "association",
                "cooccurrence"
            ],
            ascending=[
                False,
                False
            ]
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# 16-13. 그래프 생성
# ============================================================

def build_network_graph(
    selected_nodes,
    selected_edges,
    node_frequency
):
    """
    NetworkX 그래프 생성
    """

    graph = nx.Graph()

    score_lookup = dict(
        zip(
            selected_nodes[
                "word"
            ],
            selected_nodes[
                "score"
            ]
        )
    )

    tfidf_lookup = {}

    if "tfidf" in (
        selected_nodes.columns
    ):

        tfidf_lookup = dict(
            zip(
                selected_nodes[
                    "word"
                ],
                selected_nodes[
                    "tfidf"
                ]
            )
        )

    weight_lookup = {}

    if "weight" in (
        selected_nodes.columns
    ):

        weight_lookup = dict(
            zip(
                selected_nodes[
                    "word"
                ],
                selected_nodes[
                    "weight"
                ]
            )
        )

    connected_words = set()

    if not selected_edges.empty:

        connected_words.update(
            selected_edges[
                "source"
            ].tolist()
        )

        connected_words.update(
            selected_edges[
                "target"
            ].tolist()
        )

    for word in (
        selected_nodes[
            "word"
        ]
    ):

        if (
            REMOVE_ISOLATES
            and word not in connected_words
        ):

            continue

        graph.add_node(
            word,
            score=float(
                score_lookup.get(
                    word,
                    0
                )
            ),
            tfidf=float(
                tfidf_lookup.get(
                    word,
                    0
                )
            ),
            keyword_weight=float(
                weight_lookup.get(
                    word,
                    1
                )
            ),
            window_frequency=int(
                node_frequency.get(
                    word,
                    0
                )
            ),
            label=display_label(
                word
            )
        )

    for _, row in (
        selected_edges.iterrows()
    ):

        source = row[
            "source"
        ]

        target = row[
            "target"
        ]

        if (
            source not in graph
            or target not in graph
        ):

            continue

        graph.add_edge(
            source,
            target,
            weight=float(
                row[
                    "association"
                ]
            ),
            association=float(
                row[
                    "association"
                ]
            ),
            cooccurrence=int(
                row[
                    "cooccurrence"
                ]
            ),
            pmi=(
                float(
                    row[
                        "pmi"
                    ]
                )
                if not pd.isna(
                    row[
                        "pmi"
                    ]
                )
                else 0.0
            )
        )

    if REMOVE_ISOLATES:

        graph.remove_nodes_from(
            list(
                nx.isolates(
                    graph
                )
            )
        )

    return graph


# ============================================================
# 16-14. 커뮤니티 탐지
# ============================================================

def detect_communities(
    graph
):
    """
    Louvain 우선, 실패 시 Greedy Modularity
    """

    if graph.number_of_nodes() == 0:

        return (
            {},
            "None"
        )

    if graph.number_of_edges() == 0:

        community_mapping = {
            node: index
            for index, node in enumerate(
                graph.nodes(),
                start=1
            )
        }

        nx.set_node_attributes(
            graph,
            community_mapping,
            "community"
        )

        return (
            community_mapping,
            "Isolated Nodes"
        )

    try:

        communities = (
            nx.community
            .louvain_communities(
                graph,
                weight="weight",
                seed=RANDOM_STATE,
                resolution=1
            )
        )

        method = "Louvain"

    except Exception:

        communities = (
            nx.community
            .greedy_modularity_communities(
                graph,
                weight="weight"
            )
        )

        method = (
            "Greedy Modularity"
        )

    sorted_communities = sorted(
        communities,
        key=len,
        reverse=True
    )

    community_mapping = {}

    for community_id, community_nodes in enumerate(
        sorted_communities,
        start=1
    ):

        for node in community_nodes:

            community_mapping[
                node
            ] = community_id

    nx.set_node_attributes(
        graph,
        community_mapping,
        "community"
    )

    return (
        community_mapping,
        method
    )


# ============================================================
# 16-15. 그래프 표 변환
# ============================================================

def graph_to_tables(
    graph,
    country
):
    """
    노드와 edge 표 생성
    """

    node_rows = []

    for node, attributes in (
        graph.nodes(
            data=True
        )
    ):

        node_rows.append({
            "country": country,
            "word": node,
            "display_word": attributes.get(
                "label",
                display_label(
                    node
                )
            ),
            "score": attributes.get(
                "score",
                0
            ),
            "tfidf": attributes.get(
                "tfidf",
                0
            ),
            "keyword_weight": attributes.get(
                "keyword_weight",
                1
            ),
            "window_frequency": attributes.get(
                "window_frequency",
                0
            ),
            "community": attributes.get(
                "community",
                0
            ),
            "degree": graph.degree(
                node
            ),
            "weighted_degree": graph.degree(
                node,
                weight="weight"
            )
        })

    edge_rows = []

    for (
        source,
        target,
        attributes
    ) in graph.edges(
        data=True
    ):

        edge_rows.append({
            "country": country,
            "source": source,
            "target": target,
            "cooccurrence": attributes.get(
                "cooccurrence",
                0
            ),
            "association": attributes.get(
                "association",
                attributes.get(
                    "weight",
                    0
                )
            ),
            "pmi": attributes.get(
                "pmi",
                0
            )
        })

    node_table = pd.DataFrame(
        node_rows
    )

    edge_table = pd.DataFrame(
        edge_rows
    )

    if not node_table.empty:

        node_table = (
            node_table
            .sort_values(
                [
                    "community",
                    "score"
                ],
                ascending=[
                    True,
                    False
                ]
            )
            .reset_index(
                drop=True
            )
        )

    if not edge_table.empty:

        edge_table = (
            edge_table
            .sort_values(
                [
                    "association",
                    "cooccurrence"
                ],
                ascending=[
                    False,
                    False
                ]
            )
            .reset_index(
                drop=True
            )
        )

    return (
        node_table,
        edge_table
    )


# ============================================================
# 16-16. 네트워크 시각화
# ============================================================

def draw_semantic_network(
    graph,
    country,
    community_method
):
    """
    논문용 의미연결망 시각화
    """

    if graph.number_of_nodes() == 0:

        print(
            f"[SKIP] {country}: "
            "시각화할 노드가 없습니다."
        )

        return None

    if graph.number_of_edges() == 0:

        position = (
            nx.circular_layout(
                graph
            )
        )

    else:

        position = (
            nx.spring_layout(
                graph,
                weight="weight",
                seed=RANDOM_STATE,
                k=LAYOUT_K,
                iterations=500
            )
        )

    node_list = list(
        graph.nodes()
    )

    edge_list = list(
        graph.edges()
    )

    node_scores = np.array([
        graph.nodes[
            node
        ].get(
            "score",
            0
        )
        for node in node_list
    ])

    node_sizes = minmax_scale(
        node_scores,
        MIN_NODE_SIZE,
        MAX_NODE_SIZE
    )

    community_values = np.array([
        graph.nodes[
            node
        ].get(
            "community",
            0
        )
        for node in node_list
    ])

    if edge_list:

        edge_associations = np.array([
            graph.edges[
                source,
                target
            ].get(
                "association",
                graph.edges[
                    source,
                    target
                ].get(
                    "weight",
                    0
                )
            )
            for source, target in (
                edge_list
            )
        ])

        edge_widths = minmax_scale(
            edge_associations,
            MIN_EDGE_WIDTH,
            MAX_EDGE_WIDTH
        )

        edge_alphas = minmax_scale(
            edge_associations,
            0.20,
            0.75
        )

    else:

        edge_widths = np.array([])

        edge_alphas = np.array([])

    figure, axis = plt.subplots(
        figsize=FIGSIZE
    )

    for index, edge in enumerate(
        edge_list
    ):

        nx.draw_networkx_edges(
            graph,
            position,
            edgelist=[
                edge
            ],
            width=float(
                edge_widths[
                    index
                ]
            ),
            alpha=float(
                edge_alphas[
                    index
                ]
            ),
            edge_color="gray",
            ax=axis
        )

    nx.draw_networkx_nodes(
        graph,
        position,
        nodelist=node_list,
        node_size=node_sizes,
        node_color=community_values,
        cmap=plt.cm.tab20,
        alpha=0.90,
        linewidths=1.2,
        edgecolors="white",
        ax=axis
    )

    label_mapping = {
        node: display_label(
            node
        )
        for node in node_list
    }

    nx.draw_networkx_labels(
        graph,
        position,
        labels=label_mapping,
        font_size=LABEL_FONT_SIZE,
        font_family=NETWORK_FONT_FAMILY,
        font_weight="bold",
        ax=axis
    )

    community_count = len(
        set(
            community_values.tolist()
        )
    )

    title_text = (
        f"{country} — Semantic Network\n"
        f"Nodes={graph.number_of_nodes()}, "
        f"Edges={graph.number_of_edges()}, "
        f"Communities={community_count}"
    )

    if NETWORK_FONT_PROPERTY:

        axis.set_title(
            title_text,
            fontsize=18,
            pad=20,
            fontproperties=(
                NETWORK_FONT_PROPERTY
            )
        )

    else:

        axis.set_title(
            title_text,
            fontsize=18,
            pad=20
        )

    note_text = (
        "Node size: weighted TF-IDF score  |  "
        "Node color: community  |  "
        "Edge width: association strength  |  "
        f"Method: {community_method}"
    )

    axis.text(
        0.01,
        0.01,
        note_text,
        transform=axis.transAxes,
        fontsize=9,
        alpha=0.75
    )

    axis.axis(
        "off"
    )

    plt.tight_layout()

    safe_country = (
        safe_filename(
            country
        )
    )

    figure_path = (
        FIGURE_DIR
        / f"Semantic_Network_{safe_country}.png"
    )

    plt.savefig(
        figure_path,
        dpi=SAVE_DPI,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()

    plt.close()

    return figure_path


# ============================================================
# 16-17. 국가별 네트워크 실행
# ============================================================

def create_country_network(
    country,
    tfidf_table
):
    """
    한 국가의 의미연결망 전체 분석
    """

    print("-" * 80)

    print(
        f"Creating Semantic Network: "
        f"{country}"
    )

    (
        country_df,
        documents
    ) = get_country_documents(
        advanced_df,
        country
    )

    if len(
        documents
    ) == 0:

        print(
            f"[SKIP] {country}: "
            "해당 국가 문서를 찾지 못했습니다."
        )

        available_countries = (
            advanced_df[
                COUNTRY_COLUMN
            ]
            .fillna(
                "Unknown"
            )
            .astype(str)
            .value_counts()
            .rename_axis(
                "Country"
            )
            .reset_index(
                name="Document Count"
            )
        )

        display(
            available_countries
        )

        return None

    selected_nodes = (
        select_top_nodes(
            tfidf_table,
            top_nodes=TOP_NODES
        )
    )

    if selected_nodes.empty:

        print(
            f"[SKIP] {country}: "
            "핵심어가 없습니다."
        )

        return None

    allowed_words = (
        selected_nodes[
            "word"
        ].tolist()
    )

    (
        node_frequency,
        edge_frequency,
        total_windows
    ) = calculate_cooccurrence(
        documents=documents,
        allowed_words=allowed_words,
        window_size=WINDOW_SIZE
    )

    raw_edge_table = (
        create_edge_table(
            node_frequency=node_frequency,
            edge_frequency=edge_frequency,
            total_windows=total_windows
        )
    )

    if raw_edge_table.empty:

        print(
            f"[SKIP] {country}: "
            "핵심어 간 공출현이 없습니다."
        )

        return None

    selected_edge_table = (
        filter_edges(
            edge_table=raw_edge_table,
            min_cooccurrence=(
                MIN_COOCCURRENCE
            ),
            edge_quantile=(
                EDGE_QUANTILE
            ),
            min_edges=MIN_EDGES,
            max_edges_per_node=(
                MAX_EDGES_PER_NODE
            )
        )
    )

    graph = build_network_graph(
        selected_nodes=selected_nodes,
        selected_edges=selected_edge_table,
        node_frequency=node_frequency
    )

    if graph.number_of_nodes() == 0:

        print(
            f"[SKIP] {country}: "
            "필터링 후 네트워크가 비어 있습니다."
        )

        return None

    (
        community_mapping,
        community_method
    ) = detect_communities(
        graph
    )

    (
        node_table,
        edge_table
    ) = graph_to_tables(
        graph,
        country
    )

    figure_path = (
        draw_semantic_network(
            graph=graph,
            country=country,
            community_method=(
                community_method
            )
        )
    )

    safe_country = (
        safe_filename(
            country
        )
    )

    node_csv_path = (
        TABLE_DIR
        / f"Network_Nodes_{safe_country}.csv"
    )

    node_excel_path = (
        TABLE_DIR
        / f"Network_Nodes_{safe_country}.xlsx"
    )

    edge_csv_path = (
        TABLE_DIR
        / f"Network_Edges_{safe_country}.csv"
    )

    edge_excel_path = (
        TABLE_DIR
        / f"Network_Edges_{safe_country}.xlsx"
    )

    raw_edge_csv_path = (
        TABLE_DIR
        / f"Network_Raw_Edges_{safe_country}.csv"
    )

    raw_edge_excel_path = (
        TABLE_DIR
        / f"Network_Raw_Edges_{safe_country}.xlsx"
    )

    graphml_path = (
        TABLE_DIR
        / f"Semantic_Network_{safe_country}.graphml"
    )

    node_table.to_csv(
        node_csv_path,
        index=False,
        encoding="utf-8-sig"
    )

    node_table.to_excel(
        node_excel_path,
        index=False
    )

    edge_table.to_csv(
        edge_csv_path,
        index=False,
        encoding="utf-8-sig"
    )

    edge_table.to_excel(
        edge_excel_path,
        index=False
    )

    raw_edge_table.to_csv(
        raw_edge_csv_path,
        index=False,
        encoding="utf-8-sig"
    )

    raw_edge_table.to_excel(
        raw_edge_excel_path,
        index=False
    )

    nx.write_graphml(
        graph,
        graphml_path
    )

    density = nx.density(
        graph
    )

    connected_components = (
        nx.number_connected_components(
            graph
        )
    )

    community_count = len(
        set(
            community_mapping.values()
        )
    )

    average_degree = (
        sum(
            dict(
                graph.degree()
            ).values()
        )
        / graph.number_of_nodes()
        if graph.number_of_nodes() > 0
        else 0
    )

    print(
        f"\n[PASS] {country}\n"
        f"  Documents          : {len(documents)}\n"
        f"  Windows            : {total_windows}\n"
        f"  Selected Keywords  : {len(selected_nodes)}\n"
        f"  Raw Edges          : {len(raw_edge_table)}\n"
        f"  Final Nodes        : {graph.number_of_nodes()}\n"
        f"  Final Edges        : {graph.number_of_edges()}\n"
        f"  Average Degree     : {average_degree:.4f}\n"
        f"  Density            : {density:.4f}\n"
        f"  Components         : {connected_components}\n"
        f"  Communities        : {community_count}\n"
        f"  Community Method   : {community_method}"
    )

    return {
        "country": country,
        "graph": graph,
        "nodes": node_table,
        "edges": edge_table,
        "raw_edges": raw_edge_table,
        "selected_tfidf_nodes": (
            selected_nodes
        ),
        "documents": len(
            documents
        ),
        "total_windows": (
            total_windows
        ),
        "selected_keyword_count": (
            len(
                selected_nodes
            )
        ),
        "raw_edge_count": (
            len(
                raw_edge_table
            )
        ),
        "node_count": (
            graph.number_of_nodes()
        ),
        "edge_count": (
            graph.number_of_edges()
        ),
        "average_degree": (
            average_degree
        ),
        "density": density,
        "connected_components": (
            connected_components
        ),
        "community_count": (
            community_count
        ),
        "community_method": (
            community_method
        ),
        "figure_path": (
            str(
                figure_path
            )
            if figure_path
            else None
        ),
        "node_table_path": (
            str(
                node_excel_path
            )
        ),
        "edge_table_path": (
            str(
                edge_excel_path
            )
        ),
        "raw_edge_table_path": (
            str(
                raw_edge_excel_path
            )
        ),
        "graphml_path": (
            str(
                graphml_path
            )
        )
    }


# ============================================================
# 16-18. 전체 국가 실행
# ============================================================

network_results = {}


for country, tfidf_table in (
    tfidf_results.items()
):

    try:

        result = (
            create_country_network(
                country=country,
                tfidf_table=tfidf_table
            )
        )

        if result is not None:

            network_results[
                country
            ] = result

    except Exception as error:

        print(
            f"[ERROR] {country}: "
            f"{type(error).__name__}: "
            f"{error}"
        )


# ============================================================
# 16-19. 후속 모듈 호환 변수
# ============================================================

country_networks = {
    country: result[
        "graph"
    ]
    for country, result in (
        network_results.items()
    )
}


network_graphs = (
    country_networks
)


# ============================================================
# 16-20. 전체 요약표
# ============================================================

network_summary_rows = []


for country, result in (
    network_results.items()
):

    network_summary_rows.append({
        "Country": country,
        "Documents": result[
            "documents"
        ],
        "Windows": result[
            "total_windows"
        ],
        "Selected Keywords": result[
            "selected_keyword_count"
        ],
        "Raw Edges": result[
            "raw_edge_count"
        ],
        "Final Nodes": result[
            "node_count"
        ],
        "Final Edges": result[
            "edge_count"
        ],
        "Average Degree": round(
            result[
                "average_degree"
            ],
            4
        ),
        "Density": round(
            result[
                "density"
            ],
            4
        ),
        "Connected Components": result[
            "connected_components"
        ],
        "Communities": result[
            "community_count"
        ],
        "Community Method": result[
            "community_method"
        ],
        "Figure Path": result[
            "figure_path"
        ]
    })


network_summary = pd.DataFrame(
    network_summary_rows
)


if not network_summary.empty:

    display(
        network_summary
    )

    network_summary.to_csv(
        TABLE_DIR
        / "Network_Summary.csv",
        index=False,
        encoding="utf-8-sig"
    )

    network_summary.to_excel(
        TABLE_DIR
        / "Network_Summary.xlsx",
        index=False
    )


# ============================================================
# 16-21. 국가 분류 결과 저장
# ============================================================

country_detection_summary.to_csv(
    TABLE_DIR
    / "Country_Detection_Summary.csv",
    index=False,
    encoding="utf-8-sig"
)

country_detection_summary.to_excel(
    TABLE_DIR
    / "Country_Detection_Summary.xlsx",
    index=False
)


if not unknown_documents.empty:

    unknown_save_columns = [
        column
        for column in [
            "year",
            "filename",
            "extension",
            "language"
        ]
        if column in (
            unknown_documents.columns
        )
    ]

    unknown_documents[
        unknown_save_columns
    ].to_csv(
        TABLE_DIR
        / "Unknown_Country_Documents.csv",
        index=False,
        encoding="utf-8-sig"
    )

    unknown_documents[
        unknown_save_columns
    ].to_excel(
        TABLE_DIR
        / "Unknown_Country_Documents.xlsx",
        index=False
    )


# ============================================================
# 16-22. 최종 상태
# ============================================================

print("=" * 80)

print(
    "Module 16 Completed"
)

print(
    f"Detected Country Column : "
    f"{COUNTRY_COLUMN}"
)

print(
    f"Detected Token Column   : "
    f"{TOKEN_COLUMN}"
)

print(
    f"Generated Countries     : "
    f"{len(network_results)}"
)

print(
    f"Generated Country Keys  : "
    f"{list(network_results.keys())}"
)

print(
    f"Figure Directory        : "
    f"{FIGURE_DIR}"
)

print(
    f"Table Directory         : "
    f"{TABLE_DIR}"
)


expected_countries = set(
    str(
        country
    ).strip()
    for country in (
        tfidf_results.keys()
    )
)


generated_countries = set(
    network_results.keys()
)


missing_network_countries = (
    expected_countries
    - generated_countries
)


if (
    len(network_results) > 0
    and not missing_network_countries
):

    print(
        "STATUS                  : PASS"
    )

else:

    print(
        "STATUS                  : CHECK REQUIRED"
    )

    if missing_network_countries:

        print(
            "Missing Countries       : "
            f"{sorted(missing_network_countries)}"
        )


print("=" * 80)

In [ ]:
# ============================================================
# PTMS v4.5
# Module 17 : Research-Grade Centrality Analysis
# ============================================================

import re
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import networkx as nx

from pathlib import Path

warnings.filterwarnings("ignore")


print("=" * 75)
print("PTMS Module 17 : Research-Grade Centrality Analysis")
print("=" * 75)


# ============================================================
# 17-1. 저장 경로
# ============================================================

FIGURE_DIR = (
    Path(BASE_DIR)
    / "output"
    / "figures"
    / "centrality"
)

TABLE_DIR = (
    Path(BASE_DIR)
    / "output"
    / "tables"
    / "centrality"
)

REPORT_DIR = (
    Path(BASE_DIR)
    / "output"
    / "reports"
    / "centrality"
)

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 17-2. 분석 설정
# ============================================================

# 표와 그래프에 표시할 상위 단어 수
TOP_N = 15

# 논문 본문용 그래프 표시 단어 수
PLOT_TOP_N = 10

# 저장 해상도
SAVE_DPI = 300

# 중심성 종합점수 가중치
CENTRALITY_WEIGHTS = {
    "degree_centrality_norm": 0.20,
    "weighted_degree_norm": 0.20,
    "betweenness_centrality_norm": 0.25,
    "eigenvector_centrality_norm": 0.20,
    "pagerank_norm": 0.15
}

# 네트워크에서 edge의 association이 클수록 가까운 관계이므로
# 최단거리 계산에는 역수를 사용
DISTANCE_EPSILON = 1e-9


# ============================================================
# 17-3. Module 16 결과 확인
# ============================================================

if "network_results" in globals():

    graph_dictionary = {
        country: result["graph"]
        for country, result in network_results.items()
        if (
            isinstance(result, dict)
            and "graph" in result
            and isinstance(result["graph"], nx.Graph)
        )
    }

elif "country_networks" in globals():

    graph_dictionary = {
        country: graph
        for country, graph in country_networks.items()
        if isinstance(graph, nx.Graph)
    }

elif "network_graphs" in globals():

    graph_dictionary = {
        country: graph
        for country, graph in network_graphs.items()
        if isinstance(graph, nx.Graph)
    }

else:

    raise NameError(
        "Module 16의 네트워크 결과가 없습니다. "
        "Module 16을 먼저 실행하세요."
    )


if len(graph_dictionary) == 0:

    raise ValueError(
        "분석 가능한 네트워크 그래프가 없습니다."
    )


# ============================================================
# 17-4. 폰트 설정
# ============================================================

def find_centrality_font():

    candidates = []

    if "NETWORK_FONT_PATH" in globals():
        candidates.append(str(NETWORK_FONT_PATH))

    if "FONT_PATH" in globals():
        candidates.append(str(FONT_PATH))

    if "WORDCLOUD_FONT_PATH" in globals():
        candidates.append(str(WORDCLOUD_FONT_PATH))

    candidates.extend([
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc",
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf"
    ])

    for path in candidates:
        if path and Path(path).exists():
            return path

    return None


CENTRALITY_FONT_PATH = find_centrality_font()

if CENTRALITY_FONT_PATH:

    CENTRALITY_FONT_PROPERTY = fm.FontProperties(
        fname=CENTRALITY_FONT_PATH
    )

    CENTRALITY_FONT_FAMILY = (
        CENTRALITY_FONT_PROPERTY.get_name()
    )

    plt.rcParams["font.family"] = (
        CENTRALITY_FONT_FAMILY
    )

    print(f"[FONT] {CENTRALITY_FONT_PATH}")

else:

    CENTRALITY_FONT_PROPERTY = None
    CENTRALITY_FONT_FAMILY = "sans-serif"

    print(
        "[WARNING] CJK 폰트를 찾지 못했습니다. "
        "한국어·중국어 라벨이 깨질 수 있습니다."
    )

plt.rcParams["axes.unicode_minus"] = False


# ============================================================
# 17-5. 유틸리티 함수
# ============================================================

def safe_filename(value):

    return re.sub(
        r'[\\/:*?"<>| ]+',
        "_",
        str(value)
    ).strip("_")


def display_label(word):

    return (
        str(word)
        .replace("_", " ")
        .strip()
    )


def minmax_normalize(series):
    """
    중심성 지표를 0~1 범위로 정규화한다.
    """

    values = pd.to_numeric(
        series,
        errors="coerce"
    ).fillna(0.0)

    minimum = values.min()
    maximum = values.max()

    if math.isclose(
        float(minimum),
        float(maximum)
    ):

        if maximum > 0:
            return pd.Series(
                np.ones(len(values)),
                index=series.index
            )

        return pd.Series(
            np.zeros(len(values)),
            index=series.index
        )

    return (
        (values - minimum)
        / (maximum - minimum)
    )


def calculate_rank(
    series
):
    """
    동점에는 동일 순위를 부여한다.
    """

    return (
        series
        .rank(
            method="min",
            ascending=False
        )
        .astype(int)
    )


# ============================================================
# 17-6. 거리 속성 생성
# ============================================================

def add_distance_attribute(graph):
    """
    Module 16에서 association은 연결 강도다.

    최단거리 기반 중심성을 계산할 때는
    강한 연결일수록 거리가 짧아야 하므로:

    distance = 1 / association
    """

    graph_copy = graph.copy()

    for source, target, attributes in graph_copy.edges(
        data=True
    ):

        association = float(
            attributes.get(
                "association",
                attributes.get("weight", 0)
            )
        )

        distance = (
            1.0
            / max(
                association,
                DISTANCE_EPSILON
            )
        )

        graph_copy[source][target][
            "distance"
        ] = distance

    return graph_copy


# ============================================================
# 17-7. Eigenvector 안전 계산
# ============================================================

def safe_eigenvector_centrality(graph):
    """
    수렴 오류에 대비해 여러 방법을 순차적으로 적용한다.
    """

    if graph.number_of_nodes() == 0:
        return {}

    if graph.number_of_edges() == 0:

        return {
            node: 0.0
            for node in graph.nodes()
        }

    try:

        return nx.eigenvector_centrality(
            graph,
            max_iter=3000,
            tolerance=1e-8,
            weight="weight"
        )

    except Exception:

        try:

            return nx.eigenvector_centrality_numpy(
                graph,
                weight="weight"
            )

        except Exception:

            return {
                node: 0.0
                for node in graph.nodes()
            }


# ============================================================
# 17-8. HITS 안전 계산
# ============================================================

def safe_hits(graph):
    """
    무방향 네트워크에서는 hub와 authority가 유사할 수 있지만,
    보조적 결과로 저장한다.
    """

    if graph.number_of_nodes() == 0:
        return {}, {}

    if graph.number_of_edges() == 0:

        empty_scores = {
            node: 0.0
            for node in graph.nodes()
        }

        return (
            empty_scores.copy(),
            empty_scores.copy()
        )

    try:

        hubs, authorities = nx.hits(
            graph,
            max_iter=3000,
            normalized=True
        )

        return hubs, authorities

    except Exception:

        empty_scores = {
            node: 0.0
            for node in graph.nodes()
        }

        return (
            empty_scores.copy(),
            empty_scores.copy()
        )


# ============================================================
# 17-9. 국가별 중심성 계산
# ============================================================

def calculate_centrality(
    country,
    graph
):
    """
    한 국가 네트워크의 중심성 지표를 계산한다.
    """

    if graph.number_of_nodes() == 0:

        print(
            f"[SKIP] {country}: "
            "노드가 없습니다."
        )

        return None

    working_graph = add_distance_attribute(
        graph
    )

    node_count = working_graph.number_of_nodes()

    # ----------------------------------------
    # Degree
    # ----------------------------------------

    degree_raw = dict(
        working_graph.degree()
    )

    degree_centrality = (
        nx.degree_centrality(
            working_graph
        )
    )

    # ----------------------------------------
    # Weighted Degree / Strength
    # ----------------------------------------

    weighted_degree = dict(
        working_graph.degree(
            weight="weight"
        )
    )

    # ----------------------------------------
    # Betweenness
    # association의 역수를 distance로 사용
    # ----------------------------------------

    if node_count > 2:

        betweenness_centrality = (
            nx.betweenness_centrality(
                working_graph,
                weight="distance",
                normalized=True
            )
        )

    else:

        betweenness_centrality = {
            node: 0.0
            for node in working_graph.nodes()
        }

    # ----------------------------------------
    # Closeness
    # disconnected graph도 처리 가능
    # ----------------------------------------

    if working_graph.number_of_edges() > 0:

        closeness_centrality = (
            nx.closeness_centrality(
                working_graph,
                distance="distance",
                wf_improved=True
            )
        )

    else:

        closeness_centrality = {
            node: 0.0
            for node in working_graph.nodes()
        }

    # ----------------------------------------
    # Eigenvector
    # ----------------------------------------

    eigenvector_centrality = (
        safe_eigenvector_centrality(
            working_graph
        )
    )

    # ----------------------------------------
    # PageRank
    # ----------------------------------------

    if working_graph.number_of_edges() > 0:

        pagerank = nx.pagerank(
            working_graph,
            alpha=0.85,
            weight="weight",
            max_iter=3000,
            tol=1e-8
        )

    else:

        pagerank = {
            node: 1 / node_count
            for node in working_graph.nodes()
        }

    # ----------------------------------------
    # HITS
    # ----------------------------------------

    hubs, authorities = safe_hits(
        working_graph
    )

    # ----------------------------------------
    # Clustering coefficient
    # ----------------------------------------

    if working_graph.number_of_edges() > 0:

        clustering_coefficient = (
            nx.clustering(
                working_graph,
                weight="weight"
            )
        )

    else:

        clustering_coefficient = {
            node: 0.0
            for node in working_graph.nodes()
        }

    # ----------------------------------------
    # Core number
    # ----------------------------------------

    try:

        core_number = nx.core_number(
            working_graph
        )

    except Exception:

        core_number = {
            node: 0
            for node in working_graph.nodes()
        }

    # ----------------------------------------
    # 노드 속성
    # ----------------------------------------

    rows = []

    for node in working_graph.nodes():

        node_attributes = (
            working_graph.nodes[node]
        )

        rows.append({
            "country": country,
            "word": node,
            "display_word": node_attributes.get(
                "label",
                display_label(node)
            ),
            "community": node_attributes.get(
                "community",
                0
            ),
            "tfidf_score": float(
                node_attributes.get(
                    "score",
                    0
                )
            ),
            "tfidf": float(
                node_attributes.get(
                    "tfidf",
                    0
                )
            ),
            "keyword_weight": float(
                node_attributes.get(
                    "keyword_weight",
                    1
                )
            ),
            "window_frequency": int(
                node_attributes.get(
                    "window_frequency",
                    0
                )
            ),
            "degree": int(
                degree_raw.get(
                    node,
                    0
                )
            ),
            "degree_centrality": float(
                degree_centrality.get(
                    node,
                    0
                )
            ),
            "weighted_degree": float(
                weighted_degree.get(
                    node,
                    0
                )
            ),
            "betweenness_centrality": float(
                betweenness_centrality.get(
                    node,
                    0
                )
            ),
            "closeness_centrality": float(
                closeness_centrality.get(
                    node,
                    0
                )
            ),
            "eigenvector_centrality": float(
                eigenvector_centrality.get(
                    node,
                    0
                )
            ),
            "pagerank": float(
                pagerank.get(
                    node,
                    0
                )
            ),
            "hub_score": float(
                hubs.get(
                    node,
                    0
                )
            ),
            "authority_score": float(
                authorities.get(
                    node,
                    0
                )
            ),
            "clustering_coefficient": float(
                clustering_coefficient.get(
                    node,
                    0
                )
            ),
            "core_number": int(
                core_number.get(
                    node,
                    0
                )
            )
        })

    centrality_table = pd.DataFrame(
        rows
    )

    # ----------------------------------------
    # 중심성 정규화
    # ----------------------------------------

    normalization_columns = {
        "degree_centrality":
            "degree_centrality_norm",

        "weighted_degree":
            "weighted_degree_norm",

        "betweenness_centrality":
            "betweenness_centrality_norm",

        "closeness_centrality":
            "closeness_centrality_norm",

        "eigenvector_centrality":
            "eigenvector_centrality_norm",

        "pagerank":
            "pagerank_norm",

        "tfidf_score":
            "tfidf_score_norm"
    }

    for source_column, target_column in (
        normalization_columns.items()
    ):

        centrality_table[target_column] = (
            minmax_normalize(
                centrality_table[source_column]
            )
        )

    # ----------------------------------------
    # Composite Centrality
    # ----------------------------------------

    centrality_table[
        "composite_centrality"
    ] = 0.0

    for column, weight in (
        CENTRALITY_WEIGHTS.items()
    ):

        centrality_table[
            "composite_centrality"
        ] += (
            centrality_table[column]
            * weight
        )

    # ----------------------------------------
    # 담론 중요도
    #
    # 텍스트 중요도와 네트워크 중심성을
    # 동시에 반영한 보조 지표
    # ----------------------------------------

    centrality_table[
        "discourse_importance"
    ] = (
        0.60
        * centrality_table[
            "composite_centrality"
        ]
        +
        0.40
        * centrality_table[
            "tfidf_score_norm"
        ]
    )

    # ----------------------------------------
    # 순위
    # ----------------------------------------

    centrality_table[
        "degree_rank"
    ] = calculate_rank(
        centrality_table[
            "degree_centrality"
        ]
    )

    centrality_table[
        "betweenness_rank"
    ] = calculate_rank(
        centrality_table[
            "betweenness_centrality"
        ]
    )

    centrality_table[
        "eigenvector_rank"
    ] = calculate_rank(
        centrality_table[
            "eigenvector_centrality"
        ]
    )

    centrality_table[
        "pagerank_rank"
    ] = calculate_rank(
        centrality_table[
            "pagerank"
        ]
    )

    centrality_table[
        "composite_rank"
    ] = calculate_rank(
        centrality_table[
            "composite_centrality"
        ]
    )

    centrality_table[
        "discourse_importance_rank"
    ] = calculate_rank(
        centrality_table[
            "discourse_importance"
        ]
    )

    centrality_table = (
        centrality_table
        .sort_values(
            [
                "composite_centrality",
                "betweenness_centrality",
                "weighted_degree"
            ],
            ascending=[
                False,
                False,
                False
            ]
        )
        .reset_index(drop=True)
    )

    return centrality_table


# ============================================================
# 17-10. 중심성 역할 자동 분류
# ============================================================

def classify_node_roles(
    centrality_table
):
    """
    지표의 상대적 수준을 기준으로 노드 역할을 분류한다.

    Core:
        연결성과 영향력이 모두 높은 중심 개념

    Broker:
        betweenness가 높아 담론 군집을 연결하는 개념

    Authority:
        eigenvector와 PageRank가 높은 영향력 개념

    Local:
        특정 군집 내부에서 제한적으로 연결된 개념

    Peripheral:
        중심성이 전반적으로 낮은 주변 개념
    """

    table = centrality_table.copy()

    degree_threshold = (
        table["degree_centrality"]
        .quantile(0.70)
    )

    betweenness_threshold = (
        table["betweenness_centrality"]
        .quantile(0.70)
    )

    eigenvector_threshold = (
        table["eigenvector_centrality"]
        .quantile(0.70)
    )

    pagerank_threshold = (
        table["pagerank"]
        .quantile(0.70)
    )

    composite_threshold = (
        table["composite_centrality"]
        .quantile(0.70)
    )

    roles = []

    for _, row in table.iterrows():

        if (
            row["composite_centrality"]
            >= composite_threshold
            and row["degree_centrality"]
            >= degree_threshold
        ):

            role = "Core"

        elif (
            row["betweenness_centrality"]
            >= betweenness_threshold
            and row["betweenness_centrality"] > 0
        ):

            role = "Broker"

        elif (
            row["eigenvector_centrality"]
            >= eigenvector_threshold
            and row["pagerank"]
            >= pagerank_threshold
        ):

            role = "Authority"

        elif row["degree"] > 1:

            role = "Local"

        else:

            role = "Peripheral"

        roles.append(role)

    table["network_role"] = roles

    return table


# ============================================================
# 17-11. 중심성 그래프 생성
# ============================================================

def create_centrality_barplot(
    country,
    centrality_table,
    metric,
    metric_label,
    filename_suffix,
    top_n=PLOT_TOP_N
):
    """
    상위 중심어 수평 막대그래프
    """

    plot_data = (
        centrality_table
        .sort_values(
            metric,
            ascending=False
        )
        .head(top_n)
        .sort_values(
            metric,
            ascending=True
        )
        .copy()
    )

    if plot_data.empty:
        return None

    labels = (
        plot_data["display_word"]
        .astype(str)
        .tolist()
    )

    values = (
        plot_data[metric]
        .astype(float)
        .tolist()
    )

    figure, axis = plt.subplots(
        figsize=(10, 6.5)
    )

    bars = axis.barh(
        labels,
        values
    )

    maximum_value = max(values) if values else 0

    for bar, value in zip(
        bars,
        values
    ):

        axis.text(
            value
            + maximum_value * 0.015,
            bar.get_y()
            + bar.get_height() / 2,
            f"{value:.3f}",
            va="center",
            fontsize=9
        )

    axis.set_xlabel(
        metric_label,
        fontsize=11
    )

    axis.set_ylabel(
        "Keyword",
        fontsize=11
    )

    axis.set_title(
        f"{country} — Top {top_n} {metric_label}",
        fontsize=15,
        pad=15
    )

    axis.grid(
        axis="x",
        linestyle="--",
        alpha=0.30
    )

    axis.set_xlim(
        0,
        maximum_value * 1.18
        if maximum_value > 0
        else 1
    )

    plt.tight_layout()

    safe_country = safe_filename(
        country
    )

    figure_path = (
        FIGURE_DIR
        / (
            f"Centrality_{filename_suffix}_"
            f"{safe_country}.png"
        )
    )

    plt.savefig(
        figure_path,
        dpi=SAVE_DPI,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()
    plt.close()

    return figure_path


# ============================================================
# 17-12. 국가별 중심성 실행
# ============================================================

def analyze_country_centrality(
    country,
    graph
):
    """
    국가별 중심성 분석, 저장, 시각화
    """

    print("-" * 75)
    print(f"Analyzing Centrality: {country}")

    centrality_table = calculate_centrality(
        country=country,
        graph=graph
    )

    if centrality_table is None:
        return None

    centrality_table = classify_node_roles(
        centrality_table
    )

    safe_country = safe_filename(
        country
    )

    # ----------------------------------------
    # 전체 표 저장
    # ----------------------------------------

    csv_path = (
        TABLE_DIR
        / f"Centrality_Full_{safe_country}.csv"
    )

    excel_path = (
        TABLE_DIR
        / f"Centrality_Full_{safe_country}.xlsx"
    )

    centrality_table.to_csv(
        csv_path,
        index=False,
        encoding="utf-8-sig"
    )

    centrality_table.to_excel(
        excel_path,
        index=False
    )

    # ----------------------------------------
    # 논문용 TOP N 표
    # ----------------------------------------

    top_table_columns = [
        "country",
        "word",
        "display_word",
        "community",
        "network_role",
        "tfidf_score",
        "degree_centrality",
        "weighted_degree",
        "betweenness_centrality",
        "closeness_centrality",
        "eigenvector_centrality",
        "pagerank",
        "composite_centrality",
        "discourse_importance",
        "composite_rank"
    ]

    top_table = (
        centrality_table
        .sort_values(
            "composite_centrality",
            ascending=False
        )
        .head(TOP_N)[
            top_table_columns
        ]
        .copy()
    )

    top_csv_path = (
        TABLE_DIR
        / f"Centrality_TOP{TOP_N}_{safe_country}.csv"
    )

    top_excel_path = (
        TABLE_DIR
        / f"Centrality_TOP{TOP_N}_{safe_country}.xlsx"
    )

    top_table.to_csv(
        top_csv_path,
        index=False,
        encoding="utf-8-sig"
    )

    top_table.to_excel(
        top_excel_path,
        index=False
    )

    # ----------------------------------------
    # 역할별 요약표
    # ----------------------------------------

    role_summary = (
        centrality_table
        .groupby(
            "network_role",
            as_index=False
        )
        .agg(
            Node_Count=(
                "word",
                "count"
            ),
            Mean_Composite=(
                "composite_centrality",
                "mean"
            ),
            Mean_Betweenness=(
                "betweenness_centrality",
                "mean"
            ),
            Mean_TFIDF=(
                "tfidf_score",
                "mean"
            )
        )
        .sort_values(
            "Mean_Composite",
            ascending=False
        )
    )

    role_summary.to_excel(
        TABLE_DIR
        / f"Centrality_Roles_{safe_country}.xlsx",
        index=False
    )

    # ----------------------------------------
    # 그래프 생성
    # ----------------------------------------

    composite_figure = (
        create_centrality_barplot(
            country=country,
            centrality_table=centrality_table,
            metric="composite_centrality",
            metric_label="Composite Centrality",
            filename_suffix="Composite"
        )
    )

    betweenness_figure = (
        create_centrality_barplot(
            country=country,
            centrality_table=centrality_table,
            metric="betweenness_centrality",
            metric_label="Betweenness Centrality",
            filename_suffix="Betweenness"
        )
    )

    discourse_figure = (
        create_centrality_barplot(
            country=country,
            centrality_table=centrality_table,
            metric="discourse_importance",
            metric_label="Discourse Importance",
            filename_suffix="Discourse_Importance"
        )
    )

    # ----------------------------------------
    # 핵심 결과 출력
    # ----------------------------------------

    print(
        f"[PASS] {country}\n"
        f"  Nodes                 : "
        f"{len(centrality_table)}\n"
        f"  Top Composite         : "
        f"{centrality_table.iloc[0]['word']}\n"
        f"  Top Broker            : "
        f"{centrality_table.sort_values('betweenness_centrality', ascending=False).iloc[0]['word']}\n"
        f"  Top Discourse Term    : "
        f"{centrality_table.sort_values('discourse_importance', ascending=False).iloc[0]['word']}"
    )

    display(
        top_table.head(10)
    )

    return {
        "country": country,
        "centrality_table": centrality_table,
        "top_table": top_table,
        "role_summary": role_summary,
        "csv_path": str(csv_path),
        "excel_path": str(excel_path),
        "top_excel_path": str(top_excel_path),
        "composite_figure": (
            str(composite_figure)
            if composite_figure
            else None
        ),
        "betweenness_figure": (
            str(betweenness_figure)
            if betweenness_figure
            else None
        ),
        "discourse_figure": (
            str(discourse_figure)
            if discourse_figure
            else None
        )
    }


# ============================================================
# 17-13. 전체 국가 실행
# ============================================================

centrality_results = {}

for country, graph in graph_dictionary.items():

    try:

        result = analyze_country_centrality(
            country=country,
            graph=graph
        )

        if result is not None:
            centrality_results[country] = result

    except Exception as error:

        print(
            f"[ERROR] {country}: "
            f"{type(error).__name__}: {error}"
        )


# 기존 하위 모듈과 호환성을 위한 별칭
country_centrality = {
    country: result["centrality_table"]
    for country, result in centrality_results.items()
}

centrality_tables = country_centrality


# ============================================================
# 17-14. 국가별 TOP 키워드 비교표
# ============================================================

country_comparison = pd.DataFrame()

for country, result in centrality_results.items():

    table = (
        result["centrality_table"]
        .sort_values(
            "composite_centrality",
            ascending=False
        )
        .head(TOP_N)
        .reset_index(drop=True)
    )

    country_comparison[
        f"{country}_Keyword"
    ] = table["word"]

    country_comparison[
        f"{country}_Composite"
    ] = table[
        "composite_centrality"
    ].round(4)

    country_comparison[
        f"{country}_Betweenness"
    ] = table[
        "betweenness_centrality"
    ].round(4)

    country_comparison[
        f"{country}_Role"
    ] = table[
        "network_role"
    ]


if not country_comparison.empty:

    country_comparison.index = (
        country_comparison.index + 1
    )

    country_comparison.index.name = "Rank"

    display(
        country_comparison
    )

    country_comparison.to_csv(
        TABLE_DIR
        / "Country_Centrality_Comparison.csv",
        encoding="utf-8-sig"
    )

    country_comparison.to_excel(
        TABLE_DIR
        / "Country_Centrality_Comparison.xlsx"
    )


# ============================================================
# 17-15. 국가별 핵심 지표 요약
# ============================================================

summary_rows = []

for country, result in centrality_results.items():

    table = result["centrality_table"]

    top_composite = (
        table
        .sort_values(
            "composite_centrality",
            ascending=False
        )
        .iloc[0]
    )

    top_betweenness = (
        table
        .sort_values(
            "betweenness_centrality",
            ascending=False
        )
        .iloc[0]
    )

    top_eigenvector = (
        table
        .sort_values(
            "eigenvector_centrality",
            ascending=False
        )
        .iloc[0]
    )

    top_discourse = (
        table
        .sort_values(
            "discourse_importance",
            ascending=False
        )
        .iloc[0]
    )

    summary_rows.append({
        "Country": country,

        "Top Composite Keyword":
            top_composite["word"],

        "Top Composite Score":
            round(
                top_composite[
                    "composite_centrality"
                ],
                4
            ),

        "Top Broker Keyword":
            top_betweenness["word"],

        "Top Betweenness":
            round(
                top_betweenness[
                    "betweenness_centrality"
                ],
                4
            ),

        "Top Eigenvector Keyword":
            top_eigenvector["word"],

        "Top Eigenvector":
            round(
                top_eigenvector[
                    "eigenvector_centrality"
                ],
                4
            ),

        "Top Discourse Keyword":
            top_discourse["word"],

        "Top Discourse Importance":
            round(
                top_discourse[
                    "discourse_importance"
                ],
                4
            )
    })


centrality_summary = pd.DataFrame(
    summary_rows
)

if not centrality_summary.empty:

    display(
        centrality_summary
    )

    centrality_summary.to_csv(
        TABLE_DIR
        / "Centrality_Summary.csv",
        index=False,
        encoding="utf-8-sig"
    )

    centrality_summary.to_excel(
        TABLE_DIR
        / "Centrality_Summary.xlsx",
        index=False
    )


print("=" * 75)
print("Module 17 Completed")
print(f"Generated Countries : {len(centrality_results)}")
print(f"Figure Directory    : {FIGURE_DIR}")
print(f"Table Directory     : {TABLE_DIR}")
print("STATUS              : PASS")
print("=" * 75)

In [ ]:
# ============================================================
# PTMS v4.5
# Module 18 : Cross-Country Comparative Discourse Analysis
# Revised for CHN / KOR / USA
# ============================================================

import re
import math
import warnings
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

from pathlib import Path
from collections import Counter

warnings.filterwarnings("ignore")


print("=" * 80)
print("PTMS Module 18 : Cross-Country Comparative Discourse Analysis")
print("=" * 80)


# ============================================================
# 18-1. 저장 경로
# ============================================================

FIGURE_DIR = (
    Path(BASE_DIR)
    / "output"
    / "figures"
    / "comparison"
)

TABLE_DIR = (
    Path(BASE_DIR)
    / "output"
    / "tables"
    / "comparison"
)

REPORT_DIR = (
    Path(BASE_DIR)
    / "output"
    / "reports"
    / "comparison"
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 18-2. 분석 설정
# ============================================================

# TF-IDF 비교에 사용할 국가별 상위 키워드
TOP_KEYWORDS = 30

# 중심성 비교에 사용할 국가별 상위 키워드
TOP_CENTRALITY_WORDS = 30

# 국가별 고유 키워드 출력 수
TOP_UNIQUE_WORDS = 15

# 국가 쌍별 공유 키워드 출력 수
TOP_SHARED_WORDS = 20

# 국가별 프로필에 표시할 핵심어 수
PROFILE_TOP_N = 10

# 통합 담론 중요도 계산 가중치
TFIDF_COMPONENT_WEIGHT = 0.50
CENTRALITY_COMPONENT_WEIGHT = 0.50

# 이미지 해상도
SAVE_DPI = 300

# 히트맵 크기
HEATMAP_FIGSIZE = (8, 7)

# 막대그래프 크기
BAR_FIGSIZE = (11, 7)

# 소수점 표시
ROUND_DIGITS = 4


# ============================================================
# 18-3. 국가 표준화 설정
# ============================================================

STANDARD_COUNTRY_ORDER = [
    "CHN",
    "KOR",
    "USA"
]


COUNTRY_DISPLAY_NAMES = {
    "CHN": "China",
    "KOR": "South Korea",
    "USA": "United States"
}


COUNTRY_ALIASES = {

    "CHN": {
        "chn",
        "china",
        "chinese",
        "prc",
        "중국",
        "中国",
        "中國"
    },

    "KOR": {
        "kor",
        "korea",
        "south korea",
        "south_korea",
        "rok",
        "한국",
        "대한민국"
    },

    "USA": {
        "usa",
        "us",
        "u.s.",
        "united states",
        "united_states",
        "america",
        "american",
        "미국"
    }
}


def normalize_country_code(value):
    """
    다양한 국가명을 CHN / KOR / USA로 통일한다.
    """

    normalized = str(
        value
    ).strip().lower()

    normalized = re.sub(
        r"[\s\-\./]+",
        "_",
        normalized
    )

    normalized = re.sub(
        r"_+",
        "_",
        normalized
    ).strip("_")

    for standard_code, aliases in (
        COUNTRY_ALIASES.items()
    ):

        normalized_aliases = {
            re.sub(
                r"[\s\-\./]+",
                "_",
                str(alias).strip().lower()
            ).strip("_")
            for alias in aliases
        }

        normalized_aliases.add(
            standard_code.lower()
        )

        if normalized in normalized_aliases:

            return standard_code

    return str(
        value
    ).strip()


# ============================================================
# 18-4. 입력 자료 확인
# ============================================================

if "tfidf_results" not in globals():

    raise NameError(
        "tfidf_results가 없습니다. "
        "Module 14를 먼저 실행하세요."
    )


if not isinstance(
    tfidf_results,
    dict
):

    raise TypeError(
        "tfidf_results는 dictionary여야 합니다."
    )


if len(tfidf_results) < 2:

    raise ValueError(
        "국가 간 비교를 위해 최소 2개 국가의 "
        "TF-IDF 결과가 필요합니다."
    )


HAS_NETWORK_RESULTS = (
    "network_results" in globals()
    and isinstance(
        network_results,
        dict
    )
    and len(network_results) > 0
)


HAS_CENTRALITY_RESULTS = (
    "centrality_results" in globals()
    and isinstance(
        centrality_results,
        dict
    )
    and len(centrality_results) > 0
)


print(
    f"[INPUT] TF-IDF Results    : "
    f"{list(tfidf_results.keys())}"
)

print(
    f"[INPUT] Network Results   : "
    f"{list(network_results.keys()) if HAS_NETWORK_RESULTS else 'Not Available'}"
)

print(
    f"[INPUT] Centrality Results: "
    f"{list(centrality_results.keys()) if HAS_CENTRALITY_RESULTS else 'Not Available'}"
)


# ============================================================
# 18-5. 폰트 설정
# ============================================================

def find_comparison_font():
    """
    이전 모듈에서 사용한 폰트 또는
    Colab 환경의 CJK 폰트를 탐색한다.
    """

    candidates = []

    for variable_name in [
        "CENTRALITY_FONT_PATH",
        "NETWORK_FONT_PATH",
        "WORDCLOUD_FONT_PATH",
        "FONT_PATH"
    ]:

        if variable_name in globals():

            value = globals()[
                variable_name
            ]

            if value:

                candidates.append(
                    str(value)
                )

    candidates.extend([
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc",
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf"
    ])

    for path in candidates:

        if (
            path
            and Path(path).exists()
        ):

            return path

    return None


COMPARISON_FONT_PATH = (
    find_comparison_font()
)


if COMPARISON_FONT_PATH:

    COMPARISON_FONT_PROPERTY = (
        fm.FontProperties(
            fname=COMPARISON_FONT_PATH
        )
    )

    COMPARISON_FONT_FAMILY = (
        COMPARISON_FONT_PROPERTY
        .get_name()
    )

    plt.rcParams[
        "font.family"
    ] = COMPARISON_FONT_FAMILY

    print(
        f"[FONT] {COMPARISON_FONT_PATH}"
    )

else:

    COMPARISON_FONT_PROPERTY = None

    COMPARISON_FONT_FAMILY = (
        "sans-serif"
    )

    print(
        "[WARNING] CJK 폰트를 찾지 못했습니다."
    )


plt.rcParams[
    "axes.unicode_minus"
] = False


# ============================================================
# 18-6. 기본 유틸리티 함수
# ============================================================

def safe_filename(value):
    """
    안전한 파일명으로 변환한다.
    """

    return re.sub(
        r'[\\/:*?"<>| ]+',
        "_",
        str(value)
    ).strip("_")


def display_word(value):
    """
    phrase의 underscore를 공백으로 표시한다.
    """

    return (
        str(value)
        .replace("_", " ")
        .strip()
    )


def safe_numeric(series):
    """
    Series를 수치형으로 안전하게 변환한다.
    """

    return (
        pd.to_numeric(
            series,
            errors="coerce"
        )
        .replace(
            [
                np.inf,
                -np.inf
            ],
            np.nan
        )
        .fillna(0.0)
    )


def minmax_normalize(series):
    """
    Series를 0~1 범위로 정규화한다.
    """

    values = safe_numeric(
        series
    )

    if len(values) == 0:

        return pd.Series(
            dtype=float
        )

    minimum = values.min()
    maximum = values.max()

    if math.isclose(
        float(minimum),
        float(maximum)
    ):

        if maximum > 0:

            return pd.Series(
                np.ones(
                    len(values)
                ),
                index=values.index
            )

        return pd.Series(
            np.zeros(
                len(values)
            ),
            index=values.index
        )

    return (
        (values - minimum)
        / (maximum - minimum)
    )


def cosine_similarity(
    vector_a,
    vector_b
):
    """
    코사인 유사도를 계산한다.
    """

    vector_a = np.asarray(
        vector_a,
        dtype=float
    )

    vector_b = np.asarray(
        vector_b,
        dtype=float
    )

    denominator = (
        np.linalg.norm(
            vector_a
        )
        * np.linalg.norm(
            vector_b
        )
    )

    if math.isclose(
        float(denominator),
        0.0
    ):

        return 0.0

    similarity = (
        np.dot(
            vector_a,
            vector_b
        )
        / denominator
    )

    return float(
        similarity
    )


def jaccard_similarity(
    values_a,
    values_b
):
    """
    두 키워드 집합의 Jaccard 유사도
    """

    set_a = set(
        values_a
    )

    set_b = set(
        values_b
    )

    union = (
        set_a | set_b
    )

    if len(union) == 0:

        return 0.0

    return float(
        len(
            set_a & set_b
        )
        / len(union)
    )


def rank_correlation(
    ranks_a,
    ranks_b
):
    """
    Spearman 순위상관계수를 계산한다.
    """

    series_a = pd.Series(
        ranks_a,
        dtype=float
    )

    series_b = pd.Series(
        ranks_b,
        dtype=float
    )

    if (
        len(series_a) < 2
        or len(series_b) < 2
    ):

        return np.nan

    correlation = (
        series_a.corr(
            series_b,
            method="spearman"
        )
    )

    if pd.isna(
        correlation
    ):

        return np.nan

    return float(
        correlation
    )


def join_words(
    words,
    limit=10
):
    """
    키워드 목록을 보고서용 문자열로 변환한다.
    """

    cleaned_words = [
        display_word(
            word
        )
        for word in words
        if str(
            word
        ).strip()
    ]

    return ", ".join(
        cleaned_words[
            :limit
        ]
    )


def ordered_country_list(
    available_countries
):
    """
    CHN, KOR, USA 순으로 국가를 정렬한다.
    """

    available_countries = list(
        dict.fromkeys(
            available_countries
        )
    )

    ordered = [
        country
        for country in STANDARD_COUNTRY_ORDER
        if country in available_countries
    ]

    ordered.extend([
        country
        for country in available_countries
        if country not in ordered
    ])

    return ordered


# ============================================================
# 18-7. TF-IDF 결과 표준화
# ============================================================

def standardize_tfidf_table(
    country,
    table
):
    """
    Module 14의 TF-IDF 결과를 표준 형식으로 변환한다.
    """

    if not isinstance(
        table,
        pd.DataFrame
    ):

        raise TypeError(
            f"{country} TF-IDF 결과가 "
            "DataFrame이 아닙니다."
        )

    if table.empty:

        raise ValueError(
            f"{country} TF-IDF 결과가 비어 있습니다."
        )

    if "word" not in table.columns:

        raise KeyError(
            f"{country} TF-IDF 표에 word 열이 없습니다."
        )

    standardized = (
        table.copy()
    )

    standardized[
        "word"
    ] = (
        standardized[
            "word"
        ]
        .astype(str)
        .str.strip()
    )

    if "score" in standardized.columns:

        standardized[
            "score"
        ] = safe_numeric(
            standardized[
                "score"
            ]
        )

    elif "weighted_tfidf" in standardized.columns:

        standardized[
            "score"
        ] = safe_numeric(
            standardized[
                "weighted_tfidf"
            ]
        )

    elif "tfidf" in standardized.columns:

        standardized[
            "score"
        ] = safe_numeric(
            standardized[
                "tfidf"
            ]
        )

    else:

        raise KeyError(
            f"{country} TF-IDF 표에 score, "
            "weighted_tfidf 또는 tfidf 열이 없습니다."
        )

    if "tfidf" in standardized.columns:

        standardized[
            "tfidf"
        ] = safe_numeric(
            standardized[
                "tfidf"
            ]
        )

    else:

        standardized[
            "tfidf"
        ] = standardized[
            "score"
        ]

    if "weight" in standardized.columns:

        standardized[
            "keyword_weight"
        ] = safe_numeric(
            standardized[
                "weight"
            ]
        )

    elif "keyword_weight" in standardized.columns:

        standardized[
            "keyword_weight"
        ] = safe_numeric(
            standardized[
                "keyword_weight"
            ]
        )

    else:

        standardized[
            "keyword_weight"
        ] = 1.0

    standardized = (
        standardized[
            standardized[
                "word"
            ].str.len() > 0
        ]
        .sort_values(
            "score",
            ascending=False
        )
        .drop_duplicates(
            subset="word",
            keep="first"
        )
        .reset_index(
            drop=True
        )
    )

    standardized[
        "rank"
    ] = np.arange(
        1,
        len(
            standardized
        ) + 1
    )

    standardized[
        "country"
    ] = normalize_country_code(
        country
    )

    standardized[
        "display_word"
    ] = (
        standardized[
            "word"
        ].map(
            display_word
        )
    )

    return standardized


standardized_tfidf = {}


for original_country, table in (
    tfidf_results.items()
):

    standard_country = (
        normalize_country_code(
            original_country
        )
    )

    try:

        standardized_tfidf[
            standard_country
        ] = standardize_tfidf_table(
            country=standard_country,
            table=table
        )

    except Exception as error:

        print(
            f"[ERROR] TF-IDF {original_country}: "
            f"{type(error).__name__}: {error}"
        )


if len(
    standardized_tfidf
) < 2:

    raise ValueError(
        "표준화에 성공한 TF-IDF 국가가 2개 미만입니다."
    )


COUNTRIES = ordered_country_list(
    standardized_tfidf.keys()
)


print(
    f"[PASS] TF-IDF Countries: "
    f"{COUNTRIES}"
)


# ============================================================
# 18-8. Module 16 네트워크 결과 표준화
# ============================================================

standardized_network_results = {}


if HAS_NETWORK_RESULTS:

    for original_country, result in (
        network_results.items()
    ):

        standard_country = (
            normalize_country_code(
                original_country
            )
        )

        if not isinstance(
            result,
            dict
        ):

            continue

        graph = result.get(
            "graph"
        )

        node_table = result.get(
            "nodes"
        )

        edge_table = result.get(
            "edges"
        )

        if (
            graph is None
            and node_table is None
        ):

            continue

        standardized_network_results[
            standard_country
        ] = result


print(
    f"[PASS] Standardized Network Countries: "
    f"{list(standardized_network_results.keys())}"
)


# ============================================================
# 18-9. Module 17 중심성 결과 자동 탐색
# ============================================================

CENTRALITY_TABLE_KEYS = [
    "centrality_table",
    "centrality",
    "nodes",
    "node_table",
    "result",
    "data"
]


CENTRALITY_SCORE_CANDIDATES = [
    "discourse_importance",
    "composite_centrality",
    "composite_score",
    "centrality_score",
    "pagerank",
    "PageRank",
    "eigenvector_centrality",
    "eigenvector",
    "weighted_degree_centrality",
    "weighted_degree",
    "degree_centrality",
    "degree"
]


WORD_COLUMN_CANDIDATES = [
    "word",
    "node",
    "keyword",
    "term",
    "token"
]


def extract_centrality_dataframe(
    result
):
    """
    Module 17 결과에서 중심성 DataFrame을 추출한다.
    """

    if isinstance(
        result,
        pd.DataFrame
    ):

        return result.copy()

    if isinstance(
        result,
        dict
    ):

        for key in CENTRALITY_TABLE_KEYS:

            if (
                key in result
                and isinstance(
                    result[key],
                    pd.DataFrame
                )
            ):

                return result[
                    key
                ].copy()

        for value in result.values():

            if isinstance(
                value,
                pd.DataFrame
            ):

                value_columns = set(
                    value.columns
                )

                has_word_column = any(
                    column in value_columns
                    for column in WORD_COLUMN_CANDIDATES
                )

                if has_word_column:

                    return value.copy()

    return None


def standardize_centrality_table(
    country,
    result
):
    """
    Module 17 중심성 결과를 표준 형식으로 변환한다.
    """

    table = extract_centrality_dataframe(
        result
    )

    if table is None:

        return None

    if table.empty:

        return None

    word_column = None

    for candidate in WORD_COLUMN_CANDIDATES:

        if candidate in table.columns:

            word_column = candidate
            break

    if word_column is None:

        return None

    score_column = None

    for candidate in CENTRALITY_SCORE_CANDIDATES:

        if candidate in table.columns:

            score_column = candidate
            break

    if score_column is None:

        numeric_candidates = []

        for column in table.columns:

            if column == word_column:

                continue

            converted = pd.to_numeric(
                table[column],
                errors="coerce"
            )

            if converted.notna().sum() > 0:

                numeric_candidates.append(
                    column
                )

        if numeric_candidates:

            score_column = (
                numeric_candidates[0]
            )

        else:

            return None

    standardized = (
        table.copy()
    )

    standardized[
        "word"
    ] = (
        standardized[
            word_column
        ]
        .astype(str)
        .str.strip()
    )

    standardized[
        "centrality_score"
    ] = safe_numeric(
        standardized[
            score_column
        ]
    )

    standardized = (
        standardized[
            standardized[
                "word"
            ].str.len() > 0
        ]
        .sort_values(
            "centrality_score",
            ascending=False
        )
        .drop_duplicates(
            subset="word",
            keep="first"
        )
        .reset_index(
            drop=True
        )
    )

    standardized[
        "centrality_rank"
    ] = np.arange(
        1,
        len(
            standardized
        ) + 1
    )

    standardized[
        "country"
    ] = normalize_country_code(
        country
    )

    standardized[
        "centrality_metric"
    ] = score_column

    standardized[
        "display_word"
    ] = (
        standardized[
            "word"
        ].map(
            display_word
        )
    )

    return standardized


standardized_centrality = {}


if HAS_CENTRALITY_RESULTS:

    for original_country, result in (
        centrality_results.items()
    ):

        standard_country = (
            normalize_country_code(
                original_country
            )
        )

        try:

            table = (
                standardize_centrality_table(
                    country=standard_country,
                    result=result
                )
            )

            if table is not None:

                standardized_centrality[
                    standard_country
                ] = table

        except Exception as error:

            print(
                f"[WARNING] Centrality {original_country}: "
                f"{type(error).__name__}: {error}"
            )


print(
    f"[PASS] Standardized Centrality Countries: "
    f"{list(standardized_centrality.keys())}"
)


# ============================================================
# 18-10. TF-IDF 통합 행렬 생성
# ============================================================

all_tfidf_words = sorted(
    set().union(
        *[
            set(
                table[
                    "word"
                ].tolist()
            )
            for table in (
                standardized_tfidf.values()
            )
        ]
    )
)


tfidf_score_matrix = pd.DataFrame(
    0.0,
    index=all_tfidf_words,
    columns=COUNTRIES
)


tfidf_rank_matrix = pd.DataFrame(
    np.nan,
    index=all_tfidf_words,
    columns=COUNTRIES
)


for country in COUNTRIES:

    table = standardized_tfidf[
        country
    ]

    score_lookup = dict(
        zip(
            table[
                "word"
            ],
            table[
                "score"
            ]
        )
    )

    rank_lookup = dict(
        zip(
            table[
                "word"
            ],
            table[
                "rank"
            ]
        )
    )

    for word, score in (
        score_lookup.items()
    ):

        tfidf_score_matrix.loc[
            word,
            country
        ] = float(
            score
        )

    for word, rank in (
        rank_lookup.items()
    ):

        tfidf_rank_matrix.loc[
            word,
            country
        ] = float(
            rank
        )


tfidf_score_matrix.index.name = (
    "word"
)

tfidf_rank_matrix.index.name = (
    "word"
)


# ============================================================
# 18-11. TF-IDF 코사인 유사도
# ============================================================

tfidf_cosine_similarity = pd.DataFrame(
    0.0,
    index=COUNTRIES,
    columns=COUNTRIES
)


for country_a in COUNTRIES:

    for country_b in COUNTRIES:

        tfidf_cosine_similarity.loc[
            country_a,
            country_b
        ] = cosine_similarity(
            tfidf_score_matrix[
                country_a
            ].values,
            tfidf_score_matrix[
                country_b
            ].values
        )


tfidf_cosine_similarity.index.name = (
    "Country"
)


# ============================================================
# 18-12. 상위 키워드 Jaccard 유사도
# ============================================================

top_keyword_sets = {}


for country in COUNTRIES:

    top_keyword_sets[
        country
    ] = set(
        standardized_tfidf[
            country
        ]
        .head(
            TOP_KEYWORDS
        )[
            "word"
        ]
        .tolist()
    )


keyword_jaccard_similarity = pd.DataFrame(
    0.0,
    index=COUNTRIES,
    columns=COUNTRIES
)


for country_a in COUNTRIES:

    for country_b in COUNTRIES:

        keyword_jaccard_similarity.loc[
            country_a,
            country_b
        ] = jaccard_similarity(
            top_keyword_sets[
                country_a
            ],
            top_keyword_sets[
                country_b
            ]
        )


keyword_jaccard_similarity.index.name = (
    "Country"
)


# ============================================================
# 18-13. 공통 단어 순위상관
# ============================================================

rank_similarity = pd.DataFrame(
    np.nan,
    index=COUNTRIES,
    columns=COUNTRIES
)


for country_a in COUNTRIES:

    for country_b in COUNTRIES:

        if country_a == country_b:

            rank_similarity.loc[
                country_a,
                country_b
            ] = 1.0

            continue

        shared_words = (
            top_keyword_sets[
                country_a
            ]
            & top_keyword_sets[
                country_b
            ]
        )

        if len(
            shared_words
        ) < 2:

            rank_similarity.loc[
                country_a,
                country_b
            ] = np.nan

            continue

        ranks_a = [
            tfidf_rank_matrix.loc[
                word,
                country_a
            ]
            for word in shared_words
        ]

        ranks_b = [
            tfidf_rank_matrix.loc[
                word,
                country_b
            ]
            for word in shared_words
        ]

        rank_similarity.loc[
            country_a,
            country_b
        ] = rank_correlation(
            ranks_a,
            ranks_b
        )


rank_similarity.index.name = (
    "Country"
)


# ============================================================
# 18-14. 중심성 행렬 및 유사도
# ============================================================

centrality_score_matrix = pd.DataFrame()

centrality_cosine_similarity = pd.DataFrame()


centrality_countries = [
    country
    for country in COUNTRIES
    if country in (
        standardized_centrality
    )
]


if len(
    centrality_countries
) >= 2:

    all_centrality_words = sorted(
        set().union(
            *[
                set(
                    standardized_centrality[
                        country
                    ][
                        "word"
                    ].tolist()
                )
                for country in (
                    centrality_countries
                )
            ]
        )
    )

    centrality_score_matrix = pd.DataFrame(
        0.0,
        index=all_centrality_words,
        columns=centrality_countries
    )

    for country in centrality_countries:

        table = (
            standardized_centrality[
                country
            ]
        )

        score_lookup = dict(
            zip(
                table[
                    "word"
                ],
                table[
                    "centrality_score"
                ]
            )
        )

        for word, score in (
            score_lookup.items()
        ):

            centrality_score_matrix.loc[
                word,
                country
            ] = float(
                score
            )

    centrality_score_matrix.index.name = (
        "word"
    )

    centrality_cosine_similarity = pd.DataFrame(
        0.0,
        index=centrality_countries,
        columns=centrality_countries
    )

    for country_a in centrality_countries:

        for country_b in centrality_countries:

            centrality_cosine_similarity.loc[
                country_a,
                country_b
            ] = cosine_similarity(
                centrality_score_matrix[
                    country_a
                ].values,
                centrality_score_matrix[
                    country_b
                ].values
            )

    centrality_cosine_similarity.index.name = (
        "Country"
    )


# ============================================================
# 18-15. 국가 쌍별 공유·고유 키워드 분석
# ============================================================

pairwise_comparison_rows = []

pairwise_keyword_tables = {}


for country_a, country_b in (
    itertools.combinations(
        COUNTRIES,
        2
    )
):

    table_a = (
        standardized_tfidf[
            country_a
        ]
        .head(
            TOP_KEYWORDS
        )
        .copy()
    )

    table_b = (
        standardized_tfidf[
            country_b
        ]
        .head(
            TOP_KEYWORDS
        )
        .copy()
    )

    words_a = set(
        table_a[
            "word"
        ]
    )

    words_b = set(
        table_b[
            "word"
        ]
    )

    shared_words = (
        words_a & words_b
    )

    unique_a = (
        words_a - words_b
    )

    unique_b = (
        words_b - words_a
    )

    score_a = dict(
        zip(
            table_a[
                "word"
            ],
            table_a[
                "score"
            ]
        )
    )

    score_b = dict(
        zip(
            table_b[
                "word"
            ],
            table_b[
                "score"
            ]
        )
    )

    rank_a = dict(
        zip(
            table_a[
                "word"
            ],
            table_a[
                "rank"
            ]
        )
    )

    rank_b = dict(
        zip(
            table_b[
                "word"
            ],
            table_b[
                "rank"
            ]
        )
    )

    shared_rows = []

    for word in shared_words:

        shared_rows.append({
            "word": word,
            "display_word": display_word(
                word
            ),
            f"{country_a}_score": score_a.get(
                word,
                0
            ),
            f"{country_b}_score": score_b.get(
                word,
                0
            ),
            f"{country_a}_rank": rank_a.get(
                word,
                np.nan
            ),
            f"{country_b}_rank": rank_b.get(
                word,
                np.nan
            ),
            "mean_score": np.mean([
                score_a.get(
                    word,
                    0
                ),
                score_b.get(
                    word,
                    0
                )
            ]),
            "absolute_score_difference": abs(
                score_a.get(
                    word,
                    0
                )
                - score_b.get(
                    word,
                    0
                )
            )
        })

    shared_table = pd.DataFrame(
        shared_rows
    )

    if not shared_table.empty:

        shared_table = (
            shared_table
            .sort_values(
                "mean_score",
                ascending=False
            )
            .reset_index(
                drop=True
            )
        )

    pair_key = (
        f"{country_a}_{country_b}"
    )

    pairwise_keyword_tables[
        pair_key
    ] = shared_table

    sorted_shared_words = (
        shared_table[
            "word"
        ].tolist()
        if not shared_table.empty
        else []
    )

    sorted_unique_a = sorted(
        unique_a,
        key=lambda word: score_a.get(
            word,
            0
        ),
        reverse=True
    )

    sorted_unique_b = sorted(
        unique_b,
        key=lambda word: score_b.get(
            word,
            0
        ),
        reverse=True
    )

    pairwise_comparison_rows.append({
        "Country A": country_a,
        "Country B": country_b,
        "TF-IDF Cosine Similarity": round(
            tfidf_cosine_similarity.loc[
                country_a,
                country_b
            ],
            ROUND_DIGITS
        ),
        "Top Keyword Jaccard": round(
            keyword_jaccard_similarity.loc[
                country_a,
                country_b
            ],
            ROUND_DIGITS
        ),
        "Shared Rank Correlation": (
            round(
                rank_similarity.loc[
                    country_a,
                    country_b
                ],
                ROUND_DIGITS
            )
            if not pd.isna(
                rank_similarity.loc[
                    country_a,
                    country_b
                ]
            )
            else np.nan
        ),
        "Shared Keyword Count": len(
            shared_words
        ),
        f"{country_a} Unique Count": len(
            unique_a
        ),
        f"{country_b} Unique Count": len(
            unique_b
        ),
        "Top Shared Keywords": join_words(
            sorted_shared_words,
            TOP_SHARED_WORDS
        ),
        f"{country_a} Distinctive Keywords": join_words(
            sorted_unique_a,
            TOP_UNIQUE_WORDS
        ),
        f"{country_b} Distinctive Keywords": join_words(
            sorted_unique_b,
            TOP_UNIQUE_WORDS
        )
    })


pairwise_comparison = pd.DataFrame(
    pairwise_comparison_rows
)


# ============================================================
# 18-16. 국가별 고유 키워드
# ============================================================

country_unique_keyword_tables = {}


for country in COUNTRIES:

    current_table = (
        standardized_tfidf[
            country
        ]
        .head(
            TOP_KEYWORDS
        )
        .copy()
    )

    other_words = set()

    for other_country in COUNTRIES:

        if other_country == country:

            continue

        other_words.update(
            standardized_tfidf[
                other_country
            ]
            .head(
                TOP_KEYWORDS
            )[
                "word"
            ]
            .tolist()
        )

    unique_table = current_table[
        ~current_table[
            "word"
        ].isin(
            other_words
        )
    ].copy()

    unique_table[
        "distinctiveness_type"
    ] = "Unique Top Keyword"

    country_unique_keyword_tables[
        country
    ] = (
        unique_table
        .head(
            TOP_UNIQUE_WORDS
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# 18-17. 전체 국가 공통 키워드
# ============================================================

common_keyword_set = set.intersection(
    *[
        top_keyword_sets[
            country
        ]
        for country in COUNTRIES
    ]
)


common_keyword_rows = []


for word in common_keyword_set:

    row = {
        "word": word,
        "display_word": display_word(
            word
        )
    }

    scores = []
    ranks = []

    for country in COUNTRIES:

        score_value = (
            tfidf_score_matrix.loc[
                word,
                country
            ]
        )

        rank_value = (
            tfidf_rank_matrix.loc[
                word,
                country
            ]
        )

        row[
            f"{country}_score"
        ] = score_value

        row[
            f"{country}_rank"
        ] = rank_value

        scores.append(
            score_value
        )

        ranks.append(
            rank_value
        )

    row[
        "mean_score"
    ] = float(
        np.mean(
            scores
        )
    )

    row[
        "score_std"
    ] = float(
        np.std(
            scores
        )
    )

    row[
        "mean_rank"
    ] = float(
        np.mean(
            ranks
        )
    )

    common_keyword_rows.append(
        row
    )


common_keyword_table = pd.DataFrame(
    common_keyword_rows
)


if not common_keyword_table.empty:

    common_keyword_table = (
        common_keyword_table
        .sort_values(
            [
                "mean_score",
                "score_std"
            ],
            ascending=[
                False,
                True
            ]
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# 18-18. TF-IDF와 중심성 통합 중요도
# ============================================================

integrated_keyword_results = {}


for country in COUNTRIES:

    tfidf_table = (
        standardized_tfidf[
            country
        ][
            [
                "word",
                "display_word",
                "score",
                "rank"
            ]
        ]
        .copy()
    )

    tfidf_table = tfidf_table.rename(
        columns={
            "score": "tfidf_score",
            "rank": "tfidf_rank"
        }
    )

    if country in standardized_centrality:

        centrality_table = (
            standardized_centrality[
                country
            ][
                [
                    "word",
                    "centrality_score",
                    "centrality_rank",
                    "centrality_metric"
                ]
            ]
            .copy()
        )

        integrated = pd.merge(
            tfidf_table,
            centrality_table,
            on="word",
            how="outer"
        )

    else:

        integrated = (
            tfidf_table.copy()
        )

        integrated[
            "centrality_score"
        ] = 0.0

        integrated[
            "centrality_rank"
        ] = np.nan

        integrated[
            "centrality_metric"
        ] = "Not Available"

    integrated[
        "display_word"
    ] = (
        integrated[
            "display_word"
        ]
        .fillna(
            integrated[
                "word"
            ].map(
                display_word
            )
        )
    )

    integrated[
        "tfidf_score"
    ] = safe_numeric(
        integrated[
            "tfidf_score"
        ]
    )

    integrated[
        "centrality_score"
    ] = safe_numeric(
        integrated[
            "centrality_score"
        ]
    )

    integrated[
        "normalized_tfidf"
    ] = minmax_normalize(
        integrated[
            "tfidf_score"
        ]
    )

    integrated[
        "normalized_centrality"
    ] = minmax_normalize(
        integrated[
            "centrality_score"
        ]
    )

    if country in standardized_centrality:

        integrated[
            "integrated_importance"
        ] = (
            TFIDF_COMPONENT_WEIGHT
            * integrated[
                "normalized_tfidf"
            ]
            + CENTRALITY_COMPONENT_WEIGHT
            * integrated[
                "normalized_centrality"
            ]
        )

    else:

        integrated[
            "integrated_importance"
        ] = integrated[
            "normalized_tfidf"
        ]

    integrated = (
        integrated
        .sort_values(
            "integrated_importance",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )

    integrated[
        "integrated_rank"
    ] = np.arange(
        1,
        len(
            integrated
        ) + 1
    )

    integrated[
        "country"
    ] = country

    integrated_keyword_results[
        country
    ] = integrated


# ============================================================
# 18-19. 네트워크 구조 비교표
# ============================================================

network_structure_rows = []


for country in COUNTRIES:

    if country not in (
        standardized_network_results
    ):

        network_structure_rows.append({
            "Country": country,
            "Documents": np.nan,
            "Windows": np.nan,
            "Nodes": np.nan,
            "Edges": np.nan,
            "Average Degree": np.nan,
            "Density": np.nan,
            "Connected Components": np.nan,
            "Communities": np.nan,
            "Community Method": "Not Available"
        })

        continue

    result = (
        standardized_network_results[
            country
        ]
    )

    network_structure_rows.append({
        "Country": country,
        "Documents": result.get(
            "documents",
            np.nan
        ),
        "Windows": result.get(
            "total_windows",
            np.nan
        ),
        "Nodes": result.get(
            "node_count",
            np.nan
        ),
        "Edges": result.get(
            "edge_count",
            np.nan
        ),
        "Average Degree": result.get(
            "average_degree",
            np.nan
        ),
        "Density": result.get(
            "density",
            np.nan
        ),
        "Connected Components": result.get(
            "connected_components",
            np.nan
        ),
        "Communities": result.get(
            "community_count",
            np.nan
        ),
        "Community Method": result.get(
            "community_method",
            "Unknown"
        )
    })


network_structure_comparison = pd.DataFrame(
    network_structure_rows
)


# ============================================================
# 18-20. 국가별 담론 프로필
# ============================================================

country_profile_rows = []


for country in COUNTRIES:

    tfidf_table = (
        standardized_tfidf[
            country
        ]
    )

    integrated_table = (
        integrated_keyword_results[
            country
        ]
    )

    unique_table = (
        country_unique_keyword_tables[
            country
        ]
    )

    top_tfidf_words = (
        tfidf_table
        .head(
            PROFILE_TOP_N
        )[
            "word"
        ]
        .tolist()
    )

    top_integrated_words = (
        integrated_table
        .head(
            PROFILE_TOP_N
        )[
            "word"
        ]
        .tolist()
    )

    unique_words = (
        unique_table[
            "word"
        ].tolist()
        if not unique_table.empty
        else []
    )

    centrality_words = []

    centrality_metric = (
        "Not Available"
    )

    if country in standardized_centrality:

        centrality_table = (
            standardized_centrality[
                country
            ]
        )

        centrality_words = (
            centrality_table
            .head(
                PROFILE_TOP_N
            )[
                "word"
            ]
            .tolist()
        )

        if not centrality_table.empty:

            centrality_metric = (
                centrality_table[
                    "centrality_metric"
                ].iloc[0]
            )

    network_row = (
        network_structure_comparison[
            network_structure_comparison[
                "Country"
            ] == country
        ]
    )

    if not network_row.empty:

        density = (
            network_row[
                "Density"
            ].iloc[0]
        )

        communities = (
            network_row[
                "Communities"
            ].iloc[0]
        )

        components = (
            network_row[
                "Connected Components"
            ].iloc[0]
        )

    else:

        density = np.nan
        communities = np.nan
        components = np.nan

    country_profile_rows.append({
        "Country": country,
        "Country Name": COUNTRY_DISPLAY_NAMES.get(
            country,
            country
        ),
        "Top TF-IDF Keywords": join_words(
            top_tfidf_words,
            PROFILE_TOP_N
        ),
        "Top Central Keywords": (
            join_words(
                centrality_words,
                PROFILE_TOP_N
            )
            if centrality_words
            else "Not Available"
        ),
        "Top Integrated Keywords": join_words(
            top_integrated_words,
            PROFILE_TOP_N
        ),
        "Distinctive Keywords": (
            join_words(
                unique_words,
                TOP_UNIQUE_WORDS
            )
            if unique_words
            else "None within selected top keywords"
        ),
        "Centrality Metric": centrality_metric,
        "Network Density": density,
        "Communities": communities,
        "Connected Components": components
    })


country_discourse_profiles = pd.DataFrame(
    country_profile_rows
)


# ============================================================
# 18-21. 히트맵 함수
# ============================================================

def draw_similarity_heatmap(
    matrix,
    title,
    filename,
    value_format=".3f"
):
    """
    외부 시각화 패키지 없이 유사도 히트맵을 생성한다.
    """

    if (
        matrix is None
        or matrix.empty
    ):

        print(
            f"[SKIP] {title}: "
            "행렬이 비어 있습니다."
        )

        return None

    figure, axis = plt.subplots(
        figsize=HEATMAP_FIGSIZE
    )

    image = axis.imshow(
        matrix.values,
        aspect="auto",
        cmap="viridis",
        vmin=(
            -1
            if matrix.min().min() < 0
            else 0
        ),
        vmax=1
    )

    axis.set_xticks(
        np.arange(
            len(
                matrix.columns
            )
        )
    )

    axis.set_yticks(
        np.arange(
            len(
                matrix.index
            )
        )
    )

    axis.set_xticklabels(
        matrix.columns
    )

    axis.set_yticklabels(
        matrix.index
    )

    for row_index in range(
        len(
            matrix.index
        )
    ):

        for column_index in range(
            len(
                matrix.columns
            )
        ):

            value = matrix.iloc[
                row_index,
                column_index
            ]

            label = (
                "NA"
                if pd.isna(
                    value
                )
                else format(
                    value,
                    value_format
                )
            )

            axis.text(
                column_index,
                row_index,
                label,
                ha="center",
                va="center",
                fontsize=11
            )

    axis.set_title(
        title,
        fontsize=16,
        pad=16
    )

    colorbar = figure.colorbar(
        image,
        ax=axis
    )

    colorbar.set_label(
        "Similarity",
        rotation=270,
        labelpad=18
    )

    plt.tight_layout()

    output_path = (
        FIGURE_DIR
        / filename
    )

    plt.savefig(
        output_path,
        dpi=SAVE_DPI,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()
    plt.close()

    return output_path


# ============================================================
# 18-22. 유사도 히트맵 생성
# ============================================================

tfidf_heatmap_path = (
    draw_similarity_heatmap(
        matrix=tfidf_cosine_similarity,
        title="Cross-Country Weighted TF-IDF Cosine Similarity",
        filename="TFIDF_Cosine_Similarity_Heatmap.png"
    )
)


jaccard_heatmap_path = (
    draw_similarity_heatmap(
        matrix=keyword_jaccard_similarity,
        title=f"Top-{TOP_KEYWORDS} Keyword Jaccard Similarity",
        filename="Keyword_Jaccard_Similarity_Heatmap.png"
    )
)


rank_heatmap_path = (
    draw_similarity_heatmap(
        matrix=rank_similarity,
        title="Shared Keyword Rank Correlation",
        filename="Keyword_Rank_Correlation_Heatmap.png"
    )
)


centrality_heatmap_path = None


if not centrality_cosine_similarity.empty:

    centrality_heatmap_path = (
        draw_similarity_heatmap(
            matrix=centrality_cosine_similarity,
            title="Cross-Country Centrality Cosine Similarity",
            filename="Centrality_Cosine_Similarity_Heatmap.png"
        )
    )


# ============================================================
# 18-23. 네트워크 구조 비교 그래프
# ============================================================

def draw_network_metric_bar(
    comparison_table,
    metric,
    filename
):
    """
    국가별 네트워크 구조 지표 막대그래프
    """

    if metric not in (
        comparison_table.columns
    ):

        return None

    plot_table = (
        comparison_table[
            [
                "Country",
                metric
            ]
        ]
        .dropna()
        .copy()
    )

    if plot_table.empty:

        return None

    figure, axis = plt.subplots(
        figsize=BAR_FIGSIZE
    )

    bars = axis.bar(
        plot_table[
            "Country"
        ],
        plot_table[
            metric
        ]
    )

    axis.set_title(
        f"Cross-Country Comparison: {metric}",
        fontsize=16,
        pad=15
    )

    axis.set_xlabel(
        "Country"
    )

    axis.set_ylabel(
        metric
    )

    for bar, value in zip(
        bars,
        plot_table[
            metric
        ]
    ):

        axis.text(
            bar.get_x()
            + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.4f}"
            if isinstance(
                value,
                (float, np.floating)
            )
            else str(
                value
            ),
            ha="center",
            va="bottom",
            fontsize=10
        )

    plt.tight_layout()

    output_path = (
        FIGURE_DIR
        / filename
    )

    plt.savefig(
        output_path,
        dpi=SAVE_DPI,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()
    plt.close()

    return output_path


network_density_path = (
    draw_network_metric_bar(
        comparison_table=network_structure_comparison,
        metric="Density",
        filename="Network_Density_Comparison.png"
    )
)


community_count_path = (
    draw_network_metric_bar(
        comparison_table=network_structure_comparison,
        metric="Communities",
        filename="Community_Count_Comparison.png"
    )
)


average_degree_path = (
    draw_network_metric_bar(
        comparison_table=network_structure_comparison,
        metric="Average Degree",
        filename="Average_Degree_Comparison.png"
    )
)


# ============================================================
# 18-24. 국가별 통합 중요도 그래프
# ============================================================

def draw_integrated_keyword_bar(
    country,
    table,
    top_n=15
):
    """
    TF-IDF와 중심성을 결합한 통합 중요도 시각화
    """

    plot_table = (
        table
        .head(
            top_n
        )
        .sort_values(
            "integrated_importance",
            ascending=True
        )
        .copy()
    )

    if plot_table.empty:

        return None

    figure, axis = plt.subplots(
        figsize=(11, 8)
    )

    bars = axis.barh(
        plot_table[
            "display_word"
        ],
        plot_table[
            "integrated_importance"
        ]
    )

    axis.set_title(
        f"{country} — Integrated Discourse Importance",
        fontsize=16,
        pad=15
    )

    axis.set_xlabel(
        "Integrated Importance"
    )

    axis.set_ylabel(
        "Keyword"
    )

    for bar, value in zip(
        bars,
        plot_table[
            "integrated_importance"
        ]
    ):

        axis.text(
            value,
            bar.get_y()
            + bar.get_height() / 2,
            f" {value:.3f}",
            va="center",
            fontsize=9
        )

    plt.tight_layout()

    output_path = (
        FIGURE_DIR
        / f"Integrated_Importance_{safe_filename(country)}.png"
    )

    plt.savefig(
        output_path,
        dpi=SAVE_DPI,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()
    plt.close()

    return output_path


integrated_figure_paths = {}


for country, table in (
    integrated_keyword_results.items()
):

    integrated_figure_paths[
        country
    ] = draw_integrated_keyword_bar(
        country=country,
        table=table,
        top_n=15
    )


# ============================================================
# 18-25. 표 저장
# ============================================================

tfidf_score_matrix.to_csv(
    TABLE_DIR
    / "TFIDF_Score_Matrix.csv",
    encoding="utf-8-sig"
)

tfidf_score_matrix.to_excel(
    TABLE_DIR
    / "TFIDF_Score_Matrix.xlsx"
)


tfidf_rank_matrix.to_csv(
    TABLE_DIR
    / "TFIDF_Rank_Matrix.csv",
    encoding="utf-8-sig"
)

tfidf_rank_matrix.to_excel(
    TABLE_DIR
    / "TFIDF_Rank_Matrix.xlsx"
)


tfidf_cosine_similarity.to_csv(
    TABLE_DIR
    / "TFIDF_Cosine_Similarity.csv",
    encoding="utf-8-sig"
)

tfidf_cosine_similarity.to_excel(
    TABLE_DIR
    / "TFIDF_Cosine_Similarity.xlsx"
)


keyword_jaccard_similarity.to_csv(
    TABLE_DIR
    / "Keyword_Jaccard_Similarity.csv",
    encoding="utf-8-sig"
)

keyword_jaccard_similarity.to_excel(
    TABLE_DIR
    / "Keyword_Jaccard_Similarity.xlsx"
)


rank_similarity.to_csv(
    TABLE_DIR
    / "Keyword_Rank_Correlation.csv",
    encoding="utf-8-sig"
)

rank_similarity.to_excel(
    TABLE_DIR
    / "Keyword_Rank_Correlation.xlsx"
)


pairwise_comparison.to_csv(
    TABLE_DIR
    / "Pairwise_Discourse_Comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

pairwise_comparison.to_excel(
    TABLE_DIR
    / "Pairwise_Discourse_Comparison.xlsx",
    index=False
)


common_keyword_table.to_csv(
    TABLE_DIR
    / "Common_Keywords_All_Countries.csv",
    index=False,
    encoding="utf-8-sig"
)

common_keyword_table.to_excel(
    TABLE_DIR
    / "Common_Keywords_All_Countries.xlsx",
    index=False
)


network_structure_comparison.to_csv(
    TABLE_DIR
    / "Network_Structure_Comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

network_structure_comparison.to_excel(
    TABLE_DIR
    / "Network_Structure_Comparison.xlsx",
    index=False
)


country_discourse_profiles.to_csv(
    TABLE_DIR
    / "Country_Discourse_Profiles.csv",
    index=False,
    encoding="utf-8-sig"
)

country_discourse_profiles.to_excel(
    TABLE_DIR
    / "Country_Discourse_Profiles.xlsx",
    index=False
)


if not centrality_score_matrix.empty:

    centrality_score_matrix.to_csv(
        TABLE_DIR
        / "Centrality_Score_Matrix.csv",
        encoding="utf-8-sig"
    )

    centrality_score_matrix.to_excel(
        TABLE_DIR
        / "Centrality_Score_Matrix.xlsx"
    )


if not centrality_cosine_similarity.empty:

    centrality_cosine_similarity.to_csv(
        TABLE_DIR
        / "Centrality_Cosine_Similarity.csv",
        encoding="utf-8-sig"
    )

    centrality_cosine_similarity.to_excel(
        TABLE_DIR
        / "Centrality_Cosine_Similarity.xlsx"
    )


for pair_key, table in (
    pairwise_keyword_tables.items()
):

    table.to_csv(
        TABLE_DIR
        / f"Shared_Keywords_{pair_key}.csv",
        index=False,
        encoding="utf-8-sig"
    )

    table.to_excel(
        TABLE_DIR
        / f"Shared_Keywords_{pair_key}.xlsx",
        index=False
    )


for country, table in (
    country_unique_keyword_tables.items()
):

    table.to_csv(
        TABLE_DIR
        / f"Distinctive_Keywords_{safe_filename(country)}.csv",
        index=False,
        encoding="utf-8-sig"
    )

    table.to_excel(
        TABLE_DIR
        / f"Distinctive_Keywords_{safe_filename(country)}.xlsx",
        index=False
    )


for country, table in (
    integrated_keyword_results.items()
):

    table.to_csv(
        TABLE_DIR
        / f"Integrated_Keyword_Importance_{safe_filename(country)}.csv",
        index=False,
        encoding="utf-8-sig"
    )

    table.to_excel(
        TABLE_DIR
        / f"Integrated_Keyword_Importance_{safe_filename(country)}.xlsx",
        index=False
    )


# ============================================================
# 18-26. 통합 Excel 파일
# ============================================================

integrated_excel_path = (
    TABLE_DIR
    / "Module18_Comparative_Analysis.xlsx"
)


with pd.ExcelWriter(
    integrated_excel_path,
    engine="openpyxl"
) as writer:

    tfidf_cosine_similarity.to_excel(
        writer,
        sheet_name="TFIDF Cosine"
    )

    keyword_jaccard_similarity.to_excel(
        writer,
        sheet_name="Keyword Jaccard"
    )

    rank_similarity.to_excel(
        writer,
        sheet_name="Rank Correlation"
    )

    pairwise_comparison.to_excel(
        writer,
        sheet_name="Pairwise Comparison",
        index=False
    )

    network_structure_comparison.to_excel(
        writer,
        sheet_name="Network Structure",
        index=False
    )

    country_discourse_profiles.to_excel(
        writer,
        sheet_name="Country Profiles",
        index=False
    )

    common_keyword_table.to_excel(
        writer,
        sheet_name="Common Keywords",
        index=False
    )

    if not centrality_cosine_similarity.empty:

        centrality_cosine_similarity.to_excel(
            writer,
            sheet_name="Centrality Cosine"
        )

    for country in COUNTRIES:

        integrated_keyword_results[
            country
        ].head(
            100
        ).to_excel(
            writer,
            sheet_name=f"{country} Integrated"[:31],
            index=False
        )

        country_unique_keyword_tables[
            country
        ].to_excel(
            writer,
            sheet_name=f"{country} Distinctive"[:31],
            index=False
        )


# ============================================================
# 18-27. 자동 비교 보고서 생성
# ============================================================

def interpret_similarity(value):
    """
    코사인·Jaccard 유사도의 설명 범주
    """

    if pd.isna(
        value
    ):

        return "판단 불가"

    if value >= 0.80:

        return "매우 높은 유사성"

    if value >= 0.60:

        return "높은 유사성"

    if value >= 0.40:

        return "중간 수준의 유사성"

    if value >= 0.20:

        return "낮은 유사성"

    return "매우 낮은 유사성"


report_lines = []

report_lines.append(
    "PTMS v4.5 Module 18"
)

report_lines.append(
    "Cross-Country Comparative Discourse Analysis"
)

report_lines.append(
    "=" * 70
)

report_lines.append(
    ""
)

report_lines.append(
    "1. 분석 대상 국가"
)

report_lines.append(
    ", ".join(
        COUNTRIES
    )
)

report_lines.append(
    ""
)

report_lines.append(
    "2. 분석 방법"
)

report_lines.append(
    f"- 국가별 상위 {TOP_KEYWORDS}개 Weighted TF-IDF 키워드 비교"
)

report_lines.append(
    "- 전체 TF-IDF 벡터의 코사인 유사도 계산"
)

report_lines.append(
    "- 상위 키워드 집합의 Jaccard 유사도 계산"
)

report_lines.append(
    "- 공유 키워드 순위의 Spearman 상관계수 계산"
)

report_lines.append(
    "- Module 16 네트워크 구조 비교"
)

report_lines.append(
    "- Module 17 중심성 결과와 TF-IDF의 통합 중요도 계산"
)

report_lines.append(
    ""
)

report_lines.append(
    "3. 국가별 담론 프로필"
)


for _, row in (
    country_discourse_profiles.iterrows()
):

    report_lines.append(
        ""
    )

    report_lines.append(
        f"[{row['Country']}]"
    )

    report_lines.append(
        f"- Top TF-IDF Keywords: "
        f"{row['Top TF-IDF Keywords']}"
    )

    report_lines.append(
        f"- Top Central Keywords: "
        f"{row['Top Central Keywords']}"
    )

    report_lines.append(
        f"- Integrated Keywords: "
        f"{row['Top Integrated Keywords']}"
    )

    report_lines.append(
        f"- Distinctive Keywords: "
        f"{row['Distinctive Keywords']}"
    )

    if not pd.isna(
        row[
            "Network Density"
        ]
    ):

        report_lines.append(
            f"- Network Density: "
            f"{row['Network Density']:.4f}"
        )

    if not pd.isna(
        row[
            "Communities"
        ]
    ):

        report_lines.append(
            f"- Communities: "
            f"{int(row['Communities'])}"
        )


report_lines.append(
    ""
)

report_lines.append(
    "4. 국가 쌍별 비교"
)


for _, row in (
    pairwise_comparison.iterrows()
):

    country_a = row[
        "Country A"
    ]

    country_b = row[
        "Country B"
    ]

    cosine_value = row[
        "TF-IDF Cosine Similarity"
    ]

    jaccard_value = row[
        "Top Keyword Jaccard"
    ]

    report_lines.append(
        ""
    )

    report_lines.append(
        f"[{country_a} - {country_b}]"
    )

    report_lines.append(
        f"- TF-IDF Cosine Similarity: "
        f"{cosine_value:.4f} "
        f"({interpret_similarity(cosine_value)})"
    )

    report_lines.append(
        f"- Keyword Jaccard Similarity: "
        f"{jaccard_value:.4f} "
        f"({interpret_similarity(jaccard_value)})"
    )

    report_lines.append(
        f"- Shared Keyword Count: "
        f"{row['Shared Keyword Count']}"
    )

    report_lines.append(
        f"- Shared Keywords: "
        f"{row['Top Shared Keywords']}"
    )

    report_lines.append(
        f"- {country_a} Distinctive: "
        f"{row[f'{country_a} Distinctive Keywords']}"
    )

    report_lines.append(
        f"- {country_b} Distinctive: "
        f"{row[f'{country_b} Distinctive Keywords']}"
    )


report_lines.append(
    ""
)

report_lines.append(
    "5. 전체 국가 공통 키워드"
)

if not common_keyword_table.empty:

    report_lines.append(
        join_words(
            common_keyword_table[
                "word"
            ].tolist(),
            30
        )
    )

else:

    report_lines.append(
        f"상위 {TOP_KEYWORDS}개 범위에서 "
        "모든 국가가 공유하는 키워드가 없습니다."
    )


report_lines.append(
    ""
)

report_lines.append(
    "6. 해석 시 주의사항"
)

report_lines.append(
    "- 코사인 유사도는 키워드 중요도 분포의 유사성을 의미한다."
)

report_lines.append(
    "- Jaccard 유사도는 상위 키워드 구성의 중복 정도를 의미한다."
)

report_lines.append(
    "- 유사도가 높다고 정책 입장이나 문맥이 동일한 것은 아니다."
)

report_lines.append(
    "- 중심성은 의미연결망 내부의 구조적 위치를 나타내며, "
    "단순 빈도와 구분하여 해석해야 한다."
)

report_lines.append(
    "- 국가별 문서 수와 문서 길이 차이를 함께 고려해야 한다."
)


comparison_report_text = "\n".join(
    report_lines
)


report_path = (
    REPORT_DIR
    / "Module18_Comparative_Discourse_Report.txt"
)


with open(
    report_path,
    "w",
    encoding="utf-8"
) as report_file:

    report_file.write(
        comparison_report_text
    )


# ============================================================
# 18-28. 후속 모듈 호환 변수
# ============================================================

comparative_results = {

    "countries": COUNTRIES,

    "standardized_tfidf": (
        standardized_tfidf
    ),

    "standardized_centrality": (
        standardized_centrality
    ),

    "tfidf_score_matrix": (
        tfidf_score_matrix
    ),

    "tfidf_rank_matrix": (
        tfidf_rank_matrix
    ),

    "tfidf_cosine_similarity": (
        tfidf_cosine_similarity
    ),

    "keyword_jaccard_similarity": (
        keyword_jaccard_similarity
    ),

    "rank_similarity": (
        rank_similarity
    ),

    "centrality_score_matrix": (
        centrality_score_matrix
    ),

    "centrality_cosine_similarity": (
        centrality_cosine_similarity
    ),

    "pairwise_comparison": (
        pairwise_comparison
    ),

    "pairwise_keyword_tables": (
        pairwise_keyword_tables
    ),

    "common_keyword_table": (
        common_keyword_table
    ),

    "country_unique_keyword_tables": (
        country_unique_keyword_tables
    ),

    "integrated_keyword_results": (
        integrated_keyword_results
    ),

    "network_structure_comparison": (
        network_structure_comparison
    ),

    "country_discourse_profiles": (
        country_discourse_profiles
    ),

    "report_path": str(
        report_path
    ),

    "integrated_excel_path": str(
        integrated_excel_path
    )
}


comparison_results = (
    comparative_results
)


# ============================================================
# 18-29. 주요 결과 출력
# ============================================================

print(
    "\n[TF-IDF COSINE SIMILARITY]"
)

display(
    tfidf_cosine_similarity.round(
        4
    )
)


print(
    "\n[TOP KEYWORD JACCARD SIMILARITY]"
)

display(
    keyword_jaccard_similarity.round(
        4
    )
)


print(
    "\n[SHARED KEYWORD RANK CORRELATION]"
)

display(
    rank_similarity.round(
        4
    )
)


if not centrality_cosine_similarity.empty:

    print(
        "\n[CENTRALITY COSINE SIMILARITY]"
    )

    display(
        centrality_cosine_similarity.round(
            4
        )
    )


print(
    "\n[PAIRWISE DISCOURSE COMPARISON]"
)

display(
    pairwise_comparison
)


print(
    "\n[NETWORK STRUCTURE COMPARISON]"
)

display(
    network_structure_comparison
)


print(
    "\n[COUNTRY DISCOURSE PROFILES]"
)

display(
    country_discourse_profiles
)


# ============================================================
# 18-30. 최종 검증
# ============================================================

expected_countries = set(
    COUNTRIES
)


network_countries = set(
    standardized_network_results.keys()
)


centrality_available_countries = set(
    standardized_centrality.keys()
)


missing_network_countries = (
    expected_countries
    - network_countries
)


missing_centrality_countries = (
    expected_countries
    - centrality_available_countries
)


print("=" * 80)
print("Module 18 Completed")
print(
    f"Comparison Countries       : "
    f"{COUNTRIES}"
)
print(
    f"TF-IDF Countries           : "
    f"{list(standardized_tfidf.keys())}"
)
print(
    f"Network Countries          : "
    f"{list(standardized_network_results.keys())}"
)
print(
    f"Centrality Countries       : "
    f"{list(standardized_centrality.keys())}"
)
print(
    f"Pairwise Comparisons       : "
    f"{len(pairwise_comparison)}"
)
print(
    f"Common Keywords            : "
    f"{len(common_keyword_table)}"
)
print(
    f"Integrated Excel           : "
    f"{integrated_excel_path}"
)
print(
    f"Automatic Report           : "
    f"{report_path}"
)


if missing_network_countries:

    print(
        f"[WARNING] Missing Network Countries: "
        f"{sorted(missing_network_countries)}"
    )


if missing_centrality_countries:

    print(
        f"[WARNING] Missing Centrality Countries: "
        f"{sorted(missing_centrality_countries)}"
    )


if (
    len(
        standardized_tfidf
    ) >= 2
    and not missing_network_countries
    and not missing_centrality_countries
):

    print(
        "STATUS                      : PASS"
    )

elif len(
    standardized_tfidf
) >= 2:

    print(
        "STATUS                      : PARTIAL PASS"
    )

    print(
        "TF-IDF 비교는 완료되었으나 일부 네트워크 또는 "
        "중심성 결과가 누락되었습니다."
    )

else:

    print(
        "STATUS                      : CHECK REQUIRED"
    )


print(
    f"Figure Directory            : "
    f"{FIGURE_DIR}"
)

print(
    f"Table Directory             : "
    f"{TABLE_DIR}"
)

print(
    f"Report Directory            : "
    f"{REPORT_DIR}"
)

print("=" * 80)

In [ ]:
# ============================================================
# PTMS v4.5
# Module 19 : Final Output Organization and Validation
# Revised for CHN / KOR / USA
# ============================================================

import os
import re
import json
import math
import shutil
import hashlib
import warnings
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")


print("=" * 80)
print("PTMS Module 19 : Final Output Organization and Validation")
print("=" * 80)


# ============================================================
# 19-1. 기본 경로 설정
# ============================================================

if "BASE_DIR" not in globals():

    raise NameError(
        "BASE_DIR가 없습니다. "
        "기존 PTMS 모듈에서 BASE_DIR를 먼저 설정하세요."
    )


BASE_PATH = Path(
    BASE_DIR
)


OUTPUT_ROOT = (
    BASE_PATH
    / "output"
)


SOURCE_FIGURE_ROOT = (
    OUTPUT_ROOT
    / "figures"
)


SOURCE_TABLE_ROOT = (
    OUTPUT_ROOT
    / "tables"
)


SOURCE_REPORT_ROOT = (
    OUTPUT_ROOT
    / "reports"
)


FINAL_ROOT = (
    OUTPUT_ROOT
    / "final"
)


FINAL_FIGURE_DIR = (
    FINAL_ROOT
    / "figures"
)


FINAL_FIGURE_BY_COUNTRY_DIR = (
    FINAL_FIGURE_DIR
    / "by_country"
)


FINAL_FIGURE_BY_ANALYSIS_DIR = (
    FINAL_FIGURE_DIR
    / "by_analysis"
)


FINAL_OVERVIEW_DIR = (
    FINAL_FIGURE_DIR
    / "overview"
)


FINAL_TABLE_DIR = (
    FINAL_ROOT
    / "tables"
)


FINAL_REPORT_DIR = (
    FINAL_ROOT
    / "reports"
)


FINAL_MANIFEST_DIR = (
    FINAL_ROOT
    / "manifests"
)


FINAL_ARCHIVE_DIR = (
    OUTPUT_ROOT
    / "archives"
)


for directory in [
    FINAL_ROOT,
    FINAL_FIGURE_DIR,
    FINAL_FIGURE_BY_COUNTRY_DIR,
    FINAL_FIGURE_BY_ANALYSIS_DIR,
    FINAL_OVERVIEW_DIR,
    FINAL_TABLE_DIR,
    FINAL_REPORT_DIR,
    FINAL_MANIFEST_DIR,
    FINAL_ARCHIVE_DIR
]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# 19-2. 실행 설정
# ============================================================

STANDARD_COUNTRIES = [
    "CHN",
    "KOR",
    "USA"
]


COUNTRY_DISPLAY_NAMES = {
    "CHN": "China",
    "KOR": "South Korea",
    "USA": "United States"
}


ANALYSIS_ORDER = [
    "wordcloud",
    "network",
    "centrality",
    "comparison",
    "other"
]


ANALYSIS_DISPLAY_NAMES = {
    "wordcloud": "Word Cloud",
    "network": "Semantic Network",
    "centrality": "Centrality Analysis",
    "comparison": "Cross-Country Comparison",
    "other": "Other Figures"
}


FIGURE_EXTENSIONS = {
    ".png",
    ".jpg",
    ".jpeg",
    ".webp",
    ".svg",
    ".pdf"
}


RASTER_EXTENSIONS = {
    ".png",
    ".jpg",
    ".jpeg",
    ".webp"
}


TABLE_EXTENSIONS = {
    ".csv",
    ".xlsx",
    ".xls",
    ".parquet",
    ".json"
}


REPORT_EXTENSIONS = {
    ".txt",
    ".md",
    ".html",
    ".pdf",
    ".docx"
}


COPY_ALL_TABLES = True

COPY_ALL_REPORTS = True

CREATE_COUNTRY_CONTACT_SHEETS = True

CREATE_ANALYSIS_CONTACT_SHEETS = True

CREATE_FINAL_OVERVIEW = True

CREATE_ZIP_ARCHIVE = True

CONTACT_SHEET_COLUMNS = 2

CONTACT_SHEET_IMAGE_WIDTH = 1600

CONTACT_SHEET_IMAGE_HEIGHT = 1100

CONTACT_SHEET_DPI = 200

OVERVIEW_COLUMNS = 3

OVERVIEW_IMAGE_WIDTH = 1800

OVERVIEW_IMAGE_HEIGHT = 1200

SAVE_DPI = 300


# ============================================================
# 19-3. 국가명 표준화
# ============================================================

COUNTRY_ALIASES = {

    "CHN": {
        "chn",
        "china",
        "chinese",
        "prc",
        "people_republic_of_china",
        "people's_republic_of_china",
        "peoples_republic_of_china",
        "beijing",
        "중국",
        "中国",
        "中國"
    },

    "KOR": {
        "kor",
        "korea",
        "south_korea",
        "south korea",
        "republic_of_korea",
        "republic of korea",
        "rok",
        "seoul",
        "한국",
        "대한민국"
    },

    "USA": {
        "usa",
        "us",
        "u.s.",
        "united_states",
        "united states",
        "america",
        "american",
        "washington",
        "미국"
    }
}


def normalize_text_value(value):
    """
    파일명·경로·국가명을 비교 가능한 형태로 정규화한다.
    """

    value = str(
        value
    ).strip().lower()

    value = re.sub(
        r"[\s\-\./\\]+",
        "_",
        value
    )

    value = re.sub(
        r"_+",
        "_",
        value
    )

    return value.strip("_")


def normalize_country_code(value):
    """
    다양한 국가 표기를 CHN / KOR / USA로 변환한다.
    """

    normalized = normalize_text_value(
        value
    )

    for country, aliases in (
        COUNTRY_ALIASES.items()
    ):

        normalized_aliases = {
            normalize_text_value(
                alias
            )
            for alias in aliases
        }

        normalized_aliases.add(
            country.lower()
        )

        if normalized in normalized_aliases:

            return country

    return str(
        value
    ).strip()


def infer_country_from_path(path):
    """
    파일명과 경로를 이용해 국가 코드를 판별한다.
    """

    path_text = normalize_text_value(
        str(path)
    )

    path_tokens = set(
        path_text.split("_")
    )

    country_matches = []

    for country, aliases in (
        COUNTRY_ALIASES.items()
    ):

        for alias in aliases:

            normalized_alias = (
                normalize_text_value(
                    alias
                )
            )

            if not normalized_alias:

                continue

            short_aliases = {
                "chn",
                "kor",
                "usa",
                "us",
                "rok",
                "prc"
            }

            if normalized_alias in short_aliases:

                if normalized_alias in path_tokens:

                    country_matches.append(
                        country
                    )

                    break

            elif normalized_alias in path_text:

                country_matches.append(
                    country
                )

                break

    country_matches = list(
        dict.fromkeys(
            country_matches
        )
    )

    if len(
        country_matches
    ) == 1:

        return country_matches[0]

    if len(
        country_matches
    ) > 1:

        # 비교 그래프에는 여러 국가 코드가 포함될 수 있음
        return "MULTI"

    return "GLOBAL"


# ============================================================
# 19-4. 분석 유형 판별
# ============================================================

ANALYSIS_PATTERNS = {

    "wordcloud": [
        "wordcloud",
        "word_cloud",
        "word cloud",
        "cloud"
    ],

    "network": [
        "semantic_network",
        "semantic network",
        "network_graph",
        "network",
        "cooccurrence",
        "co_occurrence"
    ],

    "centrality": [
        "centrality",
        "pagerank",
        "page_rank",
        "betweenness",
        "closeness",
        "eigenvector",
        "degree_centrality",
        "discourse_importance"
    ],

    "comparison": [
        "comparison",
        "comparative",
        "similarity",
        "cosine",
        "jaccard",
        "correlation",
        "integrated_importance",
        "integrated importance",
        "density_comparison",
        "community_count",
        "average_degree"
    ]
}


def infer_analysis_type(path):
    """
    파일명과 경로를 이용해 분석 유형을 판별한다.
    """

    normalized_path = normalize_text_value(
        str(path)
    )

    # integrated importance는 Module 18 산출물
    if (
        "integrated_importance"
        in normalized_path
    ):

        return "comparison"

    for analysis_type in [
        "wordcloud",
        "centrality",
        "comparison",
        "network"
    ]:

        patterns = (
            ANALYSIS_PATTERNS[
                analysis_type
            ]
        )

        for pattern in patterns:

            normalized_pattern = (
                normalize_text_value(
                    pattern
                )
            )

            if normalized_pattern in normalized_path:

                return analysis_type

    return "other"


# ============================================================
# 19-5. 기본 파일 유틸리티
# ============================================================

def safe_filename(value):
    """
    운영체제에서 안전한 파일명으로 변환한다.
    """

    value = str(
        value
    ).strip()

    value = re.sub(
        r'[\\/:*?"<>|]+',
        "_",
        value
    )

    value = re.sub(
        r"\s+",
        "_",
        value
    )

    value = re.sub(
        r"_+",
        "_",
        value
    )

    return value.strip("_")


def path_is_inside(
    child,
    parent
):
    """
    child가 parent 내부에 있는지 확인한다.
    """

    try:

        Path(
            child
        ).resolve().relative_to(
            Path(
                parent
            ).resolve()
        )

        return True

    except Exception:

        return False


def file_md5(path):
    """
    파일의 MD5 해시를 계산한다.
    """

    hash_object = hashlib.md5()

    with open(
        path,
        "rb"
    ) as file_object:

        for chunk in iter(
            lambda: file_object.read(
                1024 * 1024
            ),
            b""
        ):

            hash_object.update(
                chunk
            )

    return hash_object.hexdigest()


def human_readable_size(size_bytes):
    """
    파일 크기를 읽기 쉬운 문자열로 변환한다.
    """

    if size_bytes is None:

        return ""

    size = float(
        size_bytes
    )

    units = [
        "B",
        "KB",
        "MB",
        "GB"
    ]

    for unit in units:

        if size < 1024:

            return (
                f"{size:.2f} {unit}"
            )

        size /= 1024

    return (
        f"{size:.2f} TB"
    )


def unique_destination_path(
    destination_directory,
    filename
):
    """
    같은 이름의 파일이 있을 경우 중복되지 않는 경로를 만든다.
    """

    destination_directory = Path(
        destination_directory
    )

    original_path = (
        destination_directory
        / filename
    )

    if not original_path.exists():

        return original_path

    stem = original_path.stem
    suffix = original_path.suffix

    counter = 2

    while True:

        candidate = (
            destination_directory
            / f"{stem}_{counter}{suffix}"
        )

        if not candidate.exists():

            return candidate

        counter += 1


def copy_file_safely(
    source_path,
    destination_directory,
    new_filename=None
):
    """
    파일을 안전하게 복사한다.
    """

    source_path = Path(
        source_path
    )

    destination_directory = Path(
        destination_directory
    )

    destination_directory.mkdir(
        parents=True,
        exist_ok=True
    )

    if new_filename is None:

        new_filename = (
            source_path.name
        )

    destination_path = (
        unique_destination_path(
            destination_directory,
            new_filename
        )
    )

    shutil.copy2(
        source_path,
        destination_path
    )

    return destination_path


# ============================================================
# 19-6. 그래프 파일 탐색
# ============================================================

def collect_files(
    root_directory,
    extensions,
    excluded_directories=None
):
    """
    지정된 확장자의 파일을 재귀적으로 수집한다.
    """

    root_directory = Path(
        root_directory
    )

    excluded_directories = (
        excluded_directories
        or []
    )

    collected = []

    if not root_directory.exists():

        return collected

    for path in root_directory.rglob(
        "*"
    ):

        if not path.is_file():

            continue

        if (
            path.suffix.lower()
            not in extensions
        ):

            continue

        excluded = False

        for excluded_directory in (
            excluded_directories
        ):

            if path_is_inside(
                path,
                excluded_directory
            ):

                excluded = True
                break

        if excluded:

            continue

        collected.append(
            path
        )

    return sorted(
        collected,
        key=lambda item: str(
            item
        ).lower()
    )


source_figure_files = collect_files(
    root_directory=SOURCE_FIGURE_ROOT,
    extensions=FIGURE_EXTENSIONS,
    excluded_directories=[
        FINAL_ROOT
    ]
)


source_table_files = collect_files(
    root_directory=SOURCE_TABLE_ROOT,
    extensions=TABLE_EXTENSIONS,
    excluded_directories=[
        FINAL_ROOT
    ]
)


source_report_files = collect_files(
    root_directory=SOURCE_REPORT_ROOT,
    extensions=REPORT_EXTENSIONS,
    excluded_directories=[
        FINAL_ROOT
    ]
)


print(
    f"[FOUND] Figure Files : "
    f"{len(source_figure_files)}"
)

print(
    f"[FOUND] Table Files  : "
    f"{len(source_table_files)}"
)

print(
    f"[FOUND] Report Files : "
    f"{len(source_report_files)}"
)


# ============================================================
# 19-7. 이미지 크기 확인
# ============================================================

def get_image_dimensions(path):
    """
    래스터 이미지의 가로·세로 크기를 확인한다.
    """

    path = Path(
        path
    )

    if (
        path.suffix.lower()
        not in RASTER_EXTENSIONS
    ):

        return (
            np.nan,
            np.nan
        )

    try:

        image = plt.imread(
            path
        )

        if len(
            image.shape
        ) >= 2:

            height = int(
                image.shape[0]
            )

            width = int(
                image.shape[1]
            )

            return (
                width,
                height
            )

    except Exception:

        pass

    return (
        np.nan,
        np.nan
    )


# ============================================================
# 19-8. 그래프 메타데이터 생성
# ============================================================

figure_index_rows = []


for figure_number, figure_path in enumerate(
    source_figure_files,
    start=1
):

    country = infer_country_from_path(
        figure_path
    )

    analysis_type = (
        infer_analysis_type(
            figure_path
        )
    )

    width, height = (
        get_image_dimensions(
            figure_path
        )
    )

    try:

        relative_path = (
            figure_path.relative_to(
                OUTPUT_ROOT
            )
        )

    except Exception:

        relative_path = (
            figure_path
        )

    file_size = (
        figure_path.stat().st_size
    )

    figure_index_rows.append({
        "Figure ID": (
            f"FIG-{figure_number:03d}"
        ),
        "Country": country,
        "Country Name": (
            COUNTRY_DISPLAY_NAMES.get(
                country,
                country
            )
        ),
        "Analysis Type": analysis_type,
        "Analysis Name": (
            ANALYSIS_DISPLAY_NAMES.get(
                analysis_type,
                analysis_type
            )
        ),
        "Filename": figure_path.name,
        "Extension": (
            figure_path.suffix.lower()
        ),
        "Relative Path": str(
            relative_path
        ),
        "Absolute Path": str(
            figure_path
        ),
        "Width": width,
        "Height": height,
        "File Size Bytes": file_size,
        "File Size": (
            human_readable_size(
                file_size
            )
        ),
        "MD5": file_md5(
            figure_path
        )
    })


figure_index = pd.DataFrame(
    figure_index_rows
)


if not figure_index.empty:

    figure_index[
        "Country Order"
    ] = (
        figure_index[
            "Country"
        ].map({
            "CHN": 1,
            "KOR": 2,
            "USA": 3,
            "MULTI": 4,
            "GLOBAL": 5
        }).fillna(9)
    )

    figure_index[
        "Analysis Order"
    ] = (
        figure_index[
            "Analysis Type"
        ].map({
            analysis: index
            for index, analysis in enumerate(
                ANALYSIS_ORDER,
                start=1
            )
        }).fillna(99)
    )

    figure_index = (
        figure_index
        .sort_values(
            [
                "Country Order",
                "Analysis Order",
                "Filename"
            ]
        )
        .drop(
            columns=[
                "Country Order",
                "Analysis Order"
            ]
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# 19-9. 표·보고서 메타데이터 생성
# ============================================================

def create_file_index(
    files,
    file_prefix,
    root_directory
):
    """
    표 또는 보고서 파일의 인덱스를 생성한다.
    """

    rows = []

    for index, path in enumerate(
        files,
        start=1
    ):

        try:

            relative_path = (
                path.relative_to(
                    root_directory
                )
            )

        except Exception:

            relative_path = path

        file_size = (
            path.stat().st_size
        )

        rows.append({
            "File ID": (
                f"{file_prefix}-{index:03d}"
            ),
            "Country": (
                infer_country_from_path(
                    path
                )
            ),
            "Analysis Type": (
                infer_analysis_type(
                    path
                )
            ),
            "Filename": path.name,
            "Extension": (
                path.suffix.lower()
            ),
            "Relative Path": str(
                relative_path
            ),
            "Absolute Path": str(
                path
            ),
            "File Size Bytes": file_size,
            "File Size": (
                human_readable_size(
                    file_size
                )
            ),
            "MD5": file_md5(
                path
            )
        })

    return pd.DataFrame(
        rows
    )


table_index = create_file_index(
    files=source_table_files,
    file_prefix="TAB",
    root_directory=OUTPUT_ROOT
)


report_index = create_file_index(
    files=source_report_files,
    file_prefix="REP",
    root_directory=OUTPUT_ROOT
)


# ============================================================
# 19-10. 그래프 최종 폴더 복사
# ============================================================

copied_figure_rows = []


for _, row in (
    figure_index.iterrows()
):

    source_path = Path(
        row[
            "Absolute Path"
        ]
    )

    country = row[
        "Country"
    ]

    analysis_type = row[
        "Analysis Type"
    ]

    country_directory = (
        FINAL_FIGURE_BY_COUNTRY_DIR
        / safe_filename(
            country
        )
    )

    analysis_directory = (
        FINAL_FIGURE_BY_ANALYSIS_DIR
        / safe_filename(
            analysis_type
        )
    )

    standardized_filename = (
        f"{safe_filename(country)}"
        f"_{safe_filename(analysis_type)}"
        f"_{safe_filename(source_path.stem)}"
        f"{source_path.suffix.lower()}"
    )

    country_copy = copy_file_safely(
        source_path=source_path,
        destination_directory=(
            country_directory
        ),
        new_filename=(
            standardized_filename
        )
    )

    analysis_copy = copy_file_safely(
        source_path=source_path,
        destination_directory=(
            analysis_directory
        ),
        new_filename=(
            standardized_filename
        )
    )

    copied_figure_rows.append({
        "Figure ID": row[
            "Figure ID"
        ],
        "Country": country,
        "Analysis Type": analysis_type,
        "Original Path": str(
            source_path
        ),
        "Country Copy Path": str(
            country_copy
        ),
        "Analysis Copy Path": str(
            analysis_copy
        )
    })


copied_figure_index = pd.DataFrame(
    copied_figure_rows
)


# ============================================================
# 19-11. 표와 보고서 최종 폴더 복사
# ============================================================

copied_table_rows = []


if COPY_ALL_TABLES:

    for _, row in (
        table_index.iterrows()
    ):

        source_path = Path(
            row[
                "Absolute Path"
            ]
        )

        copied_path = copy_file_safely(
            source_path=source_path,
            destination_directory=(
                FINAL_TABLE_DIR
            ),
            new_filename=(
                source_path.name
            )
        )

        copied_table_rows.append({
            "File ID": row[
                "File ID"
            ],
            "Original Path": str(
                source_path
            ),
            "Final Path": str(
                copied_path
            )
        })


copied_table_index = pd.DataFrame(
    copied_table_rows
)


copied_report_rows = []


if COPY_ALL_REPORTS:

    for _, row in (
        report_index.iterrows()
    ):

        source_path = Path(
            row[
                "Absolute Path"
            ]
        )

        copied_path = copy_file_safely(
            source_path=source_path,
            destination_directory=(
                FINAL_REPORT_DIR
            ),
            new_filename=(
                source_path.name
            )
        )

        copied_report_rows.append({
            "File ID": row[
                "File ID"
            ],
            "Original Path": str(
                source_path
            ),
            "Final Path": str(
                copied_path
            )
        })


copied_report_index = pd.DataFrame(
    copied_report_rows
)


# ============================================================
# 19-12. 메모리 변수 상태 점검
# ============================================================

def summarize_result_dictionary(
    variable_name
):
    """
    주요 결과 dictionary의 국가 키를 점검한다.
    """

    if variable_name not in globals():

        return {
            "Variable": variable_name,
            "Exists": False,
            "Type": "Not Available",
            "Entry Count": 0,
            "Country Keys": "",
            "Missing Countries": ", ".join(
                STANDARD_COUNTRIES
            ),
            "Status": "MISSING"
        }

    variable = globals()[
        variable_name
    ]

    if not isinstance(
        variable,
        dict
    ):

        return {
            "Variable": variable_name,
            "Exists": True,
            "Type": type(
                variable
            ).__name__,
            "Entry Count": np.nan,
            "Country Keys": "",
            "Missing Countries": "",
            "Status": "INVALID TYPE"
        }

    original_keys = list(
        variable.keys()
    )

    normalized_keys = {
        normalize_country_code(
            key
        )
        for key in original_keys
    }

    detected_country_keys = (
        normalized_keys
        & set(
            STANDARD_COUNTRIES
        )
    )

    missing_countries = (
        set(
            STANDARD_COUNTRIES
        )
        - detected_country_keys
    )

    if not missing_countries:

        status = "PASS"

    elif len(
        detected_country_keys
    ) > 0:

        status = "PARTIAL"

    else:

        status = "CHECK"

    return {
        "Variable": variable_name,
        "Exists": True,
        "Type": type(
            variable
        ).__name__,
        "Entry Count": len(
            variable
        ),
        "Country Keys": ", ".join(
            sorted(
                str(
                    key
                )
                for key in original_keys
            )
        ),
        "Missing Countries": ", ".join(
            sorted(
                missing_countries
            )
        ),
        "Status": status
    }


variable_summary_rows = []


for variable_name in [
    "tfidf_results",
    "network_results",
    "centrality_results"
]:

    variable_summary_rows.append(
        summarize_result_dictionary(
            variable_name
        )
    )


# Module 18 결과는 countries 키를 따로 점검
def summarize_comparative_results():
    """
    Module 18의 comparative_results 상태를 점검한다.
    """

    result_variable = None
    result_name = None

    if (
        "comparative_results" in globals()
        and isinstance(
            comparative_results,
            dict
        )
    ):

        result_variable = (
            comparative_results
        )

        result_name = (
            "comparative_results"
        )

    elif (
        "comparison_results" in globals()
        and isinstance(
            comparison_results,
            dict
        )
    ):

        result_variable = (
            comparison_results
        )

        result_name = (
            "comparison_results"
        )

    if result_variable is None:

        return {
            "Variable": "comparative_results",
            "Exists": False,
            "Type": "Not Available",
            "Entry Count": 0,
            "Country Keys": "",
            "Missing Countries": ", ".join(
                STANDARD_COUNTRIES
            ),
            "Status": "MISSING"
        }

    countries = result_variable.get(
        "countries",
        []
    )

    normalized_countries = {
        normalize_country_code(
            country
        )
        for country in countries
    }

    missing_countries = (
        set(
            STANDARD_COUNTRIES
        )
        - normalized_countries
    )

    if not missing_countries:

        status = "PASS"

    elif len(
        normalized_countries
        & set(
            STANDARD_COUNTRIES
        )
    ) > 0:

        status = "PARTIAL"

    else:

        status = "CHECK"

    return {
        "Variable": result_name,
        "Exists": True,
        "Type": type(
            result_variable
        ).__name__,
        "Entry Count": len(
            result_variable
        ),
        "Country Keys": ", ".join(
            sorted(
                str(
                    country
                )
                for country in countries
            )
        ),
        "Missing Countries": ", ".join(
            sorted(
                missing_countries
            )
        ),
        "Status": status
    }


variable_summary_rows.append(
    summarize_comparative_results()
)


variable_summary = pd.DataFrame(
    variable_summary_rows
)


# ============================================================
# 19-13. 국가별 그래프 완성도 점검
# ============================================================

EXPECTED_COUNTRY_ANALYSES = [
    "wordcloud",
    "network",
    "centrality"
]


expected_output_rows = []


for country in STANDARD_COUNTRIES:

    for analysis_type in (
        EXPECTED_COUNTRY_ANALYSES
    ):

        if figure_index.empty:

            matched_count = 0
            matched_files = []

        else:

            matched = figure_index[
                (
                    figure_index[
                        "Country"
                    ] == country
                )
                &
                (
                    figure_index[
                        "Analysis Type"
                    ] == analysis_type
                )
            ]

            matched_count = len(
                matched
            )

            matched_files = matched[
                "Filename"
            ].tolist()

        expected_output_rows.append({
            "Country": country,
            "Country Name": (
                COUNTRY_DISPLAY_NAMES[
                    country
                ]
            ),
            "Analysis Type": analysis_type,
            "Analysis Name": (
                ANALYSIS_DISPLAY_NAMES[
                    analysis_type
                ]
            ),
            "Expected Minimum": 1,
            "Found Count": matched_count,
            "Found Files": ", ".join(
                matched_files
            ),
            "Status": (
                "PASS"
                if matched_count >= 1
                else "MISSING"
            )
        })


expected_output_status = pd.DataFrame(
    expected_output_rows
)


# 국가 비교 그래프는 GLOBAL 또는 MULTI로 분류될 수 있음
if figure_index.empty:

    comparison_figure_count = 0
    comparison_figure_names = []

else:

    comparison_figures = figure_index[
        figure_index[
            "Analysis Type"
        ] == "comparison"
    ]

    comparison_figure_count = len(
        comparison_figures
    )

    comparison_figure_names = (
        comparison_figures[
            "Filename"
        ].tolist()
    )


comparison_output_status = pd.DataFrame([
    {
        "Analysis Type": "comparison",
        "Analysis Name": (
            ANALYSIS_DISPLAY_NAMES[
                "comparison"
            ]
        ),
        "Expected Minimum": 1,
        "Found Count": (
            comparison_figure_count
        ),
        "Found Files": ", ".join(
            comparison_figure_names
        ),
        "Status": (
            "PASS"
            if comparison_figure_count >= 1
            else "MISSING"
        )
    }
])


# ============================================================
# 19-14. 래스터 이미지 읽기
# ============================================================

def read_raster_image(path):
    """
    matplotlib로 읽을 수 있는 래스터 이미지를 반환한다.
    """

    path = Path(
        path
    )

    if (
        path.suffix.lower()
        not in RASTER_EXTENSIONS
    ):

        return None

    try:

        image = plt.imread(
            path
        )

        return image

    except Exception:

        return None


def shorten_title(
    text,
    width=45
):
    """
    긴 파일명을 시각화용으로 줄인다.
    """

    return textwrap.shorten(
        str(
            text
        ),
        width=width,
        placeholder="..."
    )


# ============================================================
# 19-15. 이미지 모음판 생성 함수
# ============================================================

def create_contact_sheet(
    image_paths,
    title,
    output_path,
    columns=2
):
    """
    여러 그래프를 하나의 이미지 모음판으로 만든다.
    """

    valid_items = []

    for image_path in image_paths:

        image = read_raster_image(
            image_path
        )

        if image is not None:

            valid_items.append(
                (
                    Path(
                        image_path
                    ),
                    image
                )
            )

    if len(
        valid_items
    ) == 0:

        print(
            f"[SKIP] Contact Sheet: {title}"
        )

        return None

    rows = math.ceil(
        len(
            valid_items
        )
        / columns
    )

    figure_width = (
        columns * 7
    )

    figure_height = (
        rows * 5.5 + 1
    )

    figure, axes = plt.subplots(
        rows,
        columns,
        figsize=(
            figure_width,
            figure_height
        )
    )

    axes = np.array(
        axes
    ).reshape(
        -1
    )

    for axis in axes:

        axis.axis(
            "off"
        )

    for axis, (
        image_path,
        image
    ) in zip(
        axes,
        valid_items
    ):

        axis.imshow(
            image
        )

        axis.set_title(
            shorten_title(
                image_path.stem,
                width=55
            ),
            fontsize=10,
            pad=8
        )

        axis.axis(
            "off"
        )

    figure.suptitle(
        title,
        fontsize=20,
        y=0.995
    )

    plt.tight_layout(
        rect=[
            0,
            0,
            1,
            0.97
        ]
    )

    output_path = Path(
        output_path
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    plt.savefig(
        output_path,
        dpi=CONTACT_SHEET_DPI,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()

    plt.close()

    return output_path


# ============================================================
# 19-16. 국가별 이미지 모음판
# ============================================================

country_contact_sheet_paths = {}


if CREATE_COUNTRY_CONTACT_SHEETS:

    for country in STANDARD_COUNTRIES:

        if figure_index.empty:

            country_paths = []

        else:

            country_paths = (
                figure_index[
                    figure_index[
                        "Country"
                    ] == country
                ][
                    "Absolute Path"
                ]
                .tolist()
            )

        output_path = (
            FINAL_OVERVIEW_DIR
            / f"Country_Figures_{country}.png"
        )

        contact_sheet_path = (
            create_contact_sheet(
                image_paths=country_paths,
                title=(
                    f"{COUNTRY_DISPLAY_NAMES[country]} "
                    f"({country}) — Analysis Figures"
                ),
                output_path=output_path,
                columns=CONTACT_SHEET_COLUMNS
            )
        )

        country_contact_sheet_paths[
            country
        ] = (
            str(
                contact_sheet_path
            )
            if contact_sheet_path
            else None
        )


# ============================================================
# 19-17. 분석 유형별 이미지 모음판
# ============================================================

analysis_contact_sheet_paths = {}


if CREATE_ANALYSIS_CONTACT_SHEETS:

    for analysis_type in ANALYSIS_ORDER:

        if figure_index.empty:

            analysis_paths = []

        else:

            analysis_paths = (
                figure_index[
                    figure_index[
                        "Analysis Type"
                    ] == analysis_type
                ][
                    "Absolute Path"
                ]
                .tolist()
            )

        output_path = (
            FINAL_OVERVIEW_DIR
            / (
                f"Analysis_Figures_"
                f"{safe_filename(analysis_type)}.png"
            )
        )

        contact_sheet_path = (
            create_contact_sheet(
                image_paths=analysis_paths,
                title=(
                    ANALYSIS_DISPLAY_NAMES.get(
                        analysis_type,
                        analysis_type
                    )
                ),
                output_path=output_path,
                columns=CONTACT_SHEET_COLUMNS
            )
        )

        analysis_contact_sheet_paths[
            analysis_type
        ] = (
            str(
                contact_sheet_path
            )
            if contact_sheet_path
            else None
        )


# ============================================================
# 19-18. 대표 그래프 선정
# ============================================================

def select_representative_figure(
    dataframe,
    country,
    analysis_type
):
    """
    특정 국가·분석유형의 대표 그래프를 선정한다.
    """

    if dataframe.empty:

        return None

    matched = dataframe[
        (
            dataframe[
                "Country"
            ] == country
        )
        &
        (
            dataframe[
                "Analysis Type"
            ] == analysis_type
        )
    ].copy()

    if matched.empty:

        return None

    # PNG 우선, 해상도 큰 파일 우선
    matched[
        "Raster Priority"
    ] = (
        matched[
            "Extension"
        ].map({
            ".png": 1,
            ".jpg": 2,
            ".jpeg": 3,
            ".webp": 4,
            ".svg": 5,
            ".pdf": 6
        }).fillna(9)
    )

    matched[
        "Pixel Area"
    ] = (
        pd.to_numeric(
            matched[
                "Width"
            ],
            errors="coerce"
        ).fillna(0)
        *
        pd.to_numeric(
            matched[
                "Height"
            ],
            errors="coerce"
        ).fillna(0)
    )

    matched = matched.sort_values(
        [
            "Raster Priority",
            "Pixel Area",
            "File Size Bytes"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )

    return Path(
        matched.iloc[0][
            "Absolute Path"
        ]
    )


representative_figure_rows = []


for country in STANDARD_COUNTRIES:

    for analysis_type in (
        EXPECTED_COUNTRY_ANALYSES
    ):

        selected_path = (
            select_representative_figure(
                dataframe=figure_index,
                country=country,
                analysis_type=analysis_type
            )
        )

        representative_figure_rows.append({
            "Country": country,
            "Country Name": (
                COUNTRY_DISPLAY_NAMES[
                    country
                ]
            ),
            "Analysis Type": analysis_type,
            "Analysis Name": (
                ANALYSIS_DISPLAY_NAMES[
                    analysis_type
                ]
            ),
            "Representative Figure": (
                str(
                    selected_path
                )
                if selected_path
                else ""
            ),
            "Status": (
                "PASS"
                if selected_path
                else "MISSING"
            )
        })


representative_figure_index = pd.DataFrame(
    representative_figure_rows
)


# ============================================================
# 19-19. 최종 대표 그래프 개요판
# ============================================================

final_overview_path = None


if CREATE_FINAL_OVERVIEW:

    representative_paths = []

    representative_labels = []

    for _, row in (
        representative_figure_index.iterrows()
    ):

        path_text = row[
            "Representative Figure"
        ]

        if not path_text:

            continue

        path = Path(
            path_text
        )

        image = read_raster_image(
            path
        )

        if image is None:

            continue

        representative_paths.append(
            path
        )

        representative_labels.append(
            (
                f"{row['Country']} — "
                f"{row['Analysis Name']}"
            )
        )

    if representative_paths:

        columns = OVERVIEW_COLUMNS

        rows = math.ceil(
            len(
                representative_paths
            )
            / columns
        )

        figure, axes = plt.subplots(
            rows,
            columns,
            figsize=(
                columns * 7,
                rows * 5.5 + 1
            )
        )

        axes = np.array(
            axes
        ).reshape(
            -1
        )

        for axis in axes:

            axis.axis(
                "off"
            )

        for axis, path, label in zip(
            axes,
            representative_paths,
            representative_labels
        ):

            image = read_raster_image(
                path
            )

            axis.imshow(
                image
            )

            axis.set_title(
                label,
                fontsize=11,
                pad=8
            )

            axis.axis(
                "off"
            )

        figure.suptitle(
            "PTMS v4.5 — Final Analysis Overview",
            fontsize=22,
            y=0.995
        )

        plt.tight_layout(
            rect=[
                0,
                0,
                1,
                0.97
            ]
        )

        final_overview_path = (
            FINAL_OVERVIEW_DIR
            / "PTMS_Final_Analysis_Overview.png"
        )

        plt.savefig(
            final_overview_path,
            dpi=SAVE_DPI,
            bbox_inches="tight",
            facecolor="white"
        )

        plt.show()

        plt.close()

    else:

        print(
            "[SKIP] 대표 그래프가 없어 최종 개요판을 생성하지 못했습니다."
        )


# ============================================================
# 19-20. 핵심 결과 변수 요약
# ============================================================

result_object_rows = []


def add_object_summary(
    object_name,
    object_value
):
    """
    주요 분석 객체의 형식과 크기를 요약한다.
    """

    if isinstance(
        object_value,
        pd.DataFrame
    ):

        object_type = (
            "DataFrame"
        )

        object_size = (
            f"{object_value.shape[0]} rows × "
            f"{object_value.shape[1]} columns"
        )

    elif isinstance(
        object_value,
        dict
    ):

        object_type = (
            "Dictionary"
        )

        object_size = (
            f"{len(object_value)} entries"
        )

    elif isinstance(
        object_value,
        list
    ):

        object_type = (
            "List"
        )

        object_size = (
            f"{len(object_value)} entries"
        )

    else:

        object_type = type(
            object_value
        ).__name__

        object_size = ""

    result_object_rows.append({
        "Object Name": object_name,
        "Object Type": object_type,
        "Object Size": object_size
    })


for object_name in [
    "advanced_df",
    "tfidf_results",
    "network_results",
    "centrality_results",
    "network_summary",
    "comparative_results",
    "comparison_results",
    "pairwise_comparison",
    "country_discourse_profiles",
    "network_structure_comparison"
]:

    if object_name in globals():

        add_object_summary(
            object_name,
            globals()[
                object_name
            ]
        )


result_object_summary = pd.DataFrame(
    result_object_rows
)


# ============================================================
# 19-21. 최종 상태 계산
# ============================================================

missing_country_outputs = (
    expected_output_status[
        expected_output_status[
            "Status"
        ] != "PASS"
    ].copy()
)


missing_variable_outputs = (
    variable_summary[
        variable_summary[
            "Status"
        ] != "PASS"
    ].copy()
)


critical_missing_count = (
    len(
        missing_country_outputs
    )
)


comparison_missing = (
    comparison_output_status.iloc[0][
        "Status"
    ] != "PASS"
)


variable_pass_count = int(
    (
        variable_summary[
            "Status"
        ] == "PASS"
    ).sum()
)


if (
    critical_missing_count == 0
    and not comparison_missing
    and variable_pass_count
    == len(
        variable_summary
    )
):

    FINAL_STATUS = "PASS"

elif (
    len(
        figure_index
    ) > 0
    and len(
        table_index
    ) > 0
):

    FINAL_STATUS = "PARTIAL PASS"

else:

    FINAL_STATUS = "CHECK REQUIRED"


# ============================================================
# 19-22. 인덱스 파일 저장
# ============================================================

figure_index_path = (
    FINAL_MANIFEST_DIR
    / "Final_Figure_Index.csv"
)


figure_index_excel_path = (
    FINAL_MANIFEST_DIR
    / "Final_Figure_Index.xlsx"
)


table_index_path = (
    FINAL_MANIFEST_DIR
    / "Final_Table_Index.csv"
)


report_index_path = (
    FINAL_MANIFEST_DIR
    / "Final_Report_Index.csv"
)


variable_summary_path = (
    FINAL_MANIFEST_DIR
    / "Final_Variable_Status.csv"
)


expected_output_path = (
    FINAL_MANIFEST_DIR
    / "Expected_Output_Status.csv"
)


representative_index_path = (
    FINAL_MANIFEST_DIR
    / "Representative_Figure_Index.csv"
)


figure_index.to_csv(
    figure_index_path,
    index=False,
    encoding="utf-8-sig"
)


figure_index.to_excel(
    figure_index_excel_path,
    index=False
)


table_index.to_csv(
    table_index_path,
    index=False,
    encoding="utf-8-sig"
)


report_index.to_csv(
    report_index_path,
    index=False,
    encoding="utf-8-sig"
)


variable_summary.to_csv(
    variable_summary_path,
    index=False,
    encoding="utf-8-sig"
)


expected_output_status.to_csv(
    expected_output_path,
    index=False,
    encoding="utf-8-sig"
)


representative_figure_index.to_csv(
    representative_index_path,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 19-23. 통합 Excel 인덱스 생성
# ============================================================

final_excel_path = (
    FINAL_MANIFEST_DIR
    / "PTMS_Final_Output_Index.xlsx"
)


with pd.ExcelWriter(
    final_excel_path,
    engine="openpyxl"
) as writer:

    figure_index.to_excel(
        writer,
        sheet_name="Figure Index",
        index=False
    )

    table_index.to_excel(
        writer,
        sheet_name="Table Index",
        index=False
    )

    report_index.to_excel(
        writer,
        sheet_name="Report Index",
        index=False
    )

    variable_summary.to_excel(
        writer,
        sheet_name="Variable Status",
        index=False
    )

    expected_output_status.to_excel(
        writer,
        sheet_name="Country Output Status",
        index=False
    )

    comparison_output_status.to_excel(
        writer,
        sheet_name="Comparison Status",
        index=False
    )

    representative_figure_index.to_excel(
        writer,
        sheet_name="Representative Figures",
        index=False
    )

    copied_figure_index.to_excel(
        writer,
        sheet_name="Copied Figures",
        index=False
    )

    copied_table_index.to_excel(
        writer,
        sheet_name="Copied Tables",
        index=False
    )

    copied_report_index.to_excel(
        writer,
        sheet_name="Copied Reports",
        index=False
    )

    result_object_summary.to_excel(
        writer,
        sheet_name="Object Summary",
        index=False
    )


# ============================================================
# 19-24. JSON Manifest 생성
# ============================================================

execution_timestamp = (
    datetime.now()
    .strftime(
        "%Y-%m-%d %H:%M:%S"
    )
)


manifest_data = {

    "pipeline": "PTMS v4.5",

    "module": (
        "Module 19 - Final Output "
        "Organization and Validation"
    ),

    "execution_timestamp": (
        execution_timestamp
    ),

    "base_directory": str(
        BASE_PATH
    ),

    "output_directory": str(
        OUTPUT_ROOT
    ),

    "final_directory": str(
        FINAL_ROOT
    ),

    "countries": (
        STANDARD_COUNTRIES
    ),

    "country_display_names": (
        COUNTRY_DISPLAY_NAMES
    ),

    "figure_count": int(
        len(
            figure_index
        )
    ),

    "table_count": int(
        len(
            table_index
        )
    ),

    "report_count": int(
        len(
            report_index
        )
    ),

    "comparison_figure_count": int(
        comparison_figure_count
    ),

    "missing_country_output_count": int(
        critical_missing_count
    ),

    "final_status": (
        FINAL_STATUS
    ),

    "paths": {
        "final_figures": str(
            FINAL_FIGURE_DIR
        ),
        "final_tables": str(
            FINAL_TABLE_DIR
        ),
        "final_reports": str(
            FINAL_REPORT_DIR
        ),
        "final_manifests": str(
            FINAL_MANIFEST_DIR
        ),
        "final_excel_index": str(
            final_excel_path
        ),
        "final_overview": (
            str(
                final_overview_path
            )
            if final_overview_path
            else None
        )
    }
}


manifest_path = (
    FINAL_MANIFEST_DIR
    / "PTMS_Final_Manifest.json"
)


with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as manifest_file:

    json.dump(
        manifest_data,
        manifest_file,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# 19-25. 최종 보고서 생성
# ============================================================

report_lines = []


report_lines.append(
    "PTMS v4.5"
)


report_lines.append(
    "Module 19: Final Output Organization and Validation"
)


report_lines.append(
    "=" * 72
)


report_lines.append(
    ""
)


report_lines.append(
    f"Execution Time: {execution_timestamp}"
)


report_lines.append(
    f"Final Status: {FINAL_STATUS}"
)


report_lines.append(
    ""
)


report_lines.append(
    "1. Analysis Countries"
)


for country in STANDARD_COUNTRIES:

    report_lines.append(
        f"- {country}: "
        f"{COUNTRY_DISPLAY_NAMES[country]}"
    )


report_lines.append(
    ""
)


report_lines.append(
    "2. Collected Outputs"
)


report_lines.append(
    f"- Figures: {len(figure_index)}"
)


report_lines.append(
    f"- Tables: {len(table_index)}"
)


report_lines.append(
    f"- Reports: {len(report_index)}"
)


report_lines.append(
    f"- Comparison Figures: "
    f"{comparison_figure_count}"
)


report_lines.append(
    ""
)


report_lines.append(
    "3. Country-Level Figure Validation"
)


for _, row in (
    expected_output_status.iterrows()
):

    report_lines.append(
        f"- {row['Country']} / "
        f"{row['Analysis Name']}: "
        f"{row['Status']} "
        f"(Found={row['Found Count']})"
    )


report_lines.append(
    ""
)


report_lines.append(
    "4. Cross-Country Comparison Validation"
)


comparison_row = (
    comparison_output_status.iloc[0]
)


report_lines.append(
    f"- {comparison_row['Analysis Name']}: "
    f"{comparison_row['Status']} "
    f"(Found={comparison_row['Found Count']})"
)


report_lines.append(
    ""
)


report_lines.append(
    "5. In-Memory Result Variables"
)


for _, row in (
    variable_summary.iterrows()
):

    report_lines.append(
        f"- {row['Variable']}: "
        f"{row['Status']} "
        f"| Keys={row['Country Keys']} "
        f"| Missing={row['Missing Countries']}"
    )


report_lines.append(
    ""
)


report_lines.append(
    "6. Missing Outputs"
)


if missing_country_outputs.empty:

    report_lines.append(
        "- No country-level figure outputs are missing."
    )

else:

    for _, row in (
        missing_country_outputs.iterrows()
    ):

        report_lines.append(
            f"- Missing: "
            f"{row['Country']} / "
            f"{row['Analysis Name']}"
        )


if comparison_missing:

    report_lines.append(
        "- Missing: Cross-country comparison figure."
    )


report_lines.append(
    ""
)


report_lines.append(
    "7. Final Output Directories"
)


report_lines.append(
    f"- Figures: {FINAL_FIGURE_DIR}"
)


report_lines.append(
    f"- Tables: {FINAL_TABLE_DIR}"
)


report_lines.append(
    f"- Reports: {FINAL_REPORT_DIR}"
)


report_lines.append(
    f"- Manifests: {FINAL_MANIFEST_DIR}"
)


report_lines.append(
    ""
)


report_lines.append(
    "8. Final Interpretation Note"
)


report_lines.append(
    "- Module 19 does not recalculate TF-IDF, "
    "semantic networks, centrality, or similarities."
)


report_lines.append(
    "- It collects, classifies, validates, copies, "
    "and indexes the outputs generated by Modules 15–18."
)


report_lines.append(
    "- Missing files should be corrected by rerunning "
    "the relevant preceding module."
)


final_report_text = "\n".join(
    report_lines
)


final_report_path = (
    FINAL_REPORT_DIR
    / "PTMS_Final_Validation_Report.txt"
)


with open(
    final_report_path,
    "w",
    encoding="utf-8"
) as report_file:

    report_file.write(
        final_report_text
    )


# Markdown 보고서
final_markdown_path = (
    FINAL_REPORT_DIR
    / "PTMS_Final_Validation_Report.md"
)


markdown_lines = [
    "# PTMS v4.5 Final Validation Report",
    "",
    f"- Execution time: `{execution_timestamp}`",
    f"- Final status: **{FINAL_STATUS}**",
    "",
    "## Output counts",
    "",
    f"- Figures: {len(figure_index)}",
    f"- Tables: {len(table_index)}",
    f"- Reports: {len(report_index)}",
    f"- Comparison figures: {comparison_figure_count}",
    "",
    "## Country-level figure status",
    ""
]


for _, row in (
    expected_output_status.iterrows()
):

    markdown_lines.append(
        f"- **{row['Country']} — "
        f"{row['Analysis Name']}**: "
        f"{row['Status']} "
        f"(found {row['Found Count']})"
    )


markdown_lines.extend([
    "",
    "## Variable status",
    ""
])


for _, row in (
    variable_summary.iterrows()
):

    markdown_lines.append(
        f"- **{row['Variable']}**: "
        f"{row['Status']}"
    )


with open(
    final_markdown_path,
    "w",
    encoding="utf-8"
) as markdown_file:

    markdown_file.write(
        "\n".join(
            markdown_lines
        )
    )


# ============================================================
# 19-26. ZIP 아카이브 생성
# ============================================================

final_zip_path = None


if CREATE_ZIP_ARCHIVE:

    archive_base_name = (
        FINAL_ARCHIVE_DIR
        / "PTMS_v4_5_Final_Output"
    )

    archive_result = shutil.make_archive(
        base_name=str(
            archive_base_name
        ),
        format="zip",
        root_dir=str(
            FINAL_ROOT
        )
    )

    final_zip_path = Path(
        archive_result
    )


# ============================================================
# 19-27. 후속 사용 변수
# ============================================================

finalization_results = {

    "status": FINAL_STATUS,

    "execution_timestamp": (
        execution_timestamp
    ),

    "countries": (
        STANDARD_COUNTRIES
    ),

    "country_display_names": (
        COUNTRY_DISPLAY_NAMES
    ),

    "figure_index": (
        figure_index
    ),

    "table_index": (
        table_index
    ),

    "report_index": (
        report_index
    ),

    "variable_summary": (
        variable_summary
    ),

    "expected_output_status": (
        expected_output_status
    ),

    "comparison_output_status": (
        comparison_output_status
    ),

    "representative_figure_index": (
        representative_figure_index
    ),

    "copied_figure_index": (
        copied_figure_index
    ),

    "copied_table_index": (
        copied_table_index
    ),

    "copied_report_index": (
        copied_report_index
    ),

    "result_object_summary": (
        result_object_summary
    ),

    "country_contact_sheet_paths": (
        country_contact_sheet_paths
    ),

    "analysis_contact_sheet_paths": (
        analysis_contact_sheet_paths
    ),

    "final_overview_path": (
        str(
            final_overview_path
        )
        if final_overview_path
        else None
    ),

    "final_excel_path": str(
        final_excel_path
    ),

    "manifest_path": str(
        manifest_path
    ),

    "final_report_path": str(
        final_report_path
    ),

    "final_markdown_path": str(
        final_markdown_path
    ),

    "final_zip_path": (
        str(
            final_zip_path
        )
        if final_zip_path
        else None
    ),

    "final_root": str(
        FINAL_ROOT
    )
}


final_results = (
    finalization_results
)


# ============================================================
# 19-28. 주요 결과 출력
# ============================================================

print(
    "\n[VARIABLE STATUS]"
)

display(
    variable_summary
)


print(
    "\n[COUNTRY-LEVEL FIGURE STATUS]"
)

display(
    expected_output_status
)


print(
    "\n[COMPARISON FIGURE STATUS]"
)

display(
    comparison_output_status
)


print(
    "\n[REPRESENTATIVE FIGURES]"
)

display(
    representative_figure_index
)


print(
    "\n[FIGURE INDEX]"
)

if not figure_index.empty:

    display(
        figure_index[
            [
                "Figure ID",
                "Country",
                "Analysis Type",
                "Filename",
                "Width",
                "Height",
                "File Size"
            ]
        ]
    )

else:

    print(
        "수집된 그래프가 없습니다."
    )


# ============================================================
# 19-29. 최종 상태 출력
# ============================================================

print("=" * 80)

print(
    "Module 19 Completed"
)

print(
    f"Countries                : "
    f"{STANDARD_COUNTRIES}"
)

print(
    f"Collected Figures         : "
    f"{len(figure_index)}"
)

print(
    f"Collected Tables          : "
    f"{len(table_index)}"
)

print(
    f"Collected Reports         : "
    f"{len(report_index)}"
)

print(
    f"Comparison Figures        : "
    f"{comparison_figure_count}"
)

print(
    f"Missing Country Outputs   : "
    f"{critical_missing_count}"
)

print(
    f"Final Overview            : "
    f"{final_overview_path}"
)

print(
    f"Final Excel Index         : "
    f"{final_excel_path}"
)

print(
    f"Final Manifest            : "
    f"{manifest_path}"
)

print(
    f"Final Validation Report   : "
    f"{final_report_path}"
)

print(
    f"Final ZIP Archive         : "
    f"{final_zip_path}"
)

print(
    f"Final Output Directory    : "
    f"{FINAL_ROOT}"
)

print(
    f"STATUS                    : "
    f"{FINAL_STATUS}"
)


if not missing_country_outputs.empty:

    print(
        "\n[WARNING] Missing country-level figures:"
    )

    for _, row in (
        missing_country_outputs.iterrows()
    ):

        print(
            f"  - {row['Country']} / "
            f"{row['Analysis Name']}"
        )


if comparison_missing:

    print(
        "\n[WARNING] 국가 간 비교 그래프가 "
        "발견되지 않았습니다."
    )


if not missing_variable_outputs.empty:

    print(
        "\n[WARNING] 일부 분석 결과 변수를 "
        "확인해야 합니다:"
    )

    for _, row in (
        missing_variable_outputs.iterrows()
    ):

        print(
            f"  - {row['Variable']}: "
            f"{row['Status']}"
        )


print("=" * 80)